Beberapa Referensi


https://stackoverflow.com/questions/59024220/error-in-computing-the-coherence-score-attributeerror-dict-object-has-no-at?rq=1

https://towardsdatascience.com/topic-modelling-in-python-with-nltk-and-gensim-4ef03213cd21


https://www.machinelearningplus.com/nlp/topic-modeling-gensim-python/

https://stackoverflow.com/questions/66759852/no-module-named-pyldavis

# Setting

In [ ]:
# Run cell ini untuk melihat output tabel/dataframe secara keseluruhan

import pandas as pd
pd.set_option('display.max_rows', None)

# Library

In [ ]:
!pip install pyLDAvis

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install wikipedia

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
pip install owlready2

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
# !wget https://raw.githubusercontent.com/hardymanm/nusantarafood/main/aziekitchen-udang.jl

In [ ]:
import pandas as pd
import numpy as np
import json
import gensim.corpora as corpora
import pyLDAvis.gensim_models #Ini library terbaru

import requests
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.7/dist-packages/past/types/oldstr.py:5: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  from collections import Iterable


In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from pydrive import auth
from google.colab import auth 
from oauth2client.client import GoogleCredentials

In [ ]:
## Ontology Module
from owlready2 import *
import types

In [ ]:
from nltk.corpus import stopwords
import string
import nltk
import re
import matplotlib.pyplot as plt
import gensim
from sklearn.feature_extraction.text import CountVectorizer

import wikipedia
from nltk.corpus import wordnet as wn

# For widget
import asyncio
%gui asyncio
import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output 

import functools

nltk.download('stopwords')
nltk.download("punkt")
nltk.download('wordnet')
nltk.download('omw')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data]   Package omw is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Subprogram untuk LDA, Cleaning, dan Probability

In [ ]:
!gdown --id "1etWM37-sxCXvfy7JFc8Mwlv8rtXupeA5"

def add_stopwords_to_txtFile(array_of_words, filename = "sorted-updated-stopwords-ms.txt"):
    with open(filename, "a") as f:
        for i in range(len(array_of_words)):
            if i == len(array_of_words) - 1:
              f.write(array_of_words[i])
            else:
              f.write(array_of_words[i])
              f.write("\n")

# test = ["sapu", "cawan"]
# add_stopwords_to_txtFile(test)
# stopwordMY = set(line.strip() for line in open('sorted-updated-stopwords-ms.txt'))

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1etWM37-sxCXvfy7JFc8Mwlv8rtXupeA5
To: /content/sorted-updated-stopwords-ms.txt
100% 8.33k/8.33k [00:00<00:00, 11.1MB/s]


In [ ]:
def delist(df, colname):
  result = []
  for i in range(len(df[colname])):
    full = []
    for j in range(len(df[colname][i])):
      teks = ""
      for k in range(len(df[colname][i][j])):
        teks += df[colname][i][j][k]
      full.append(teks)
    result.append(full)
  return result
  
def pre_cleaning_text(df, threshold = 20):
  # Pre-cleaning (ngilangin cerita), hapus jika kalimat lebih dari 20
  for teks in range(len(df['page_description'])):
    for kalimat in range(len(df['page_description'][teks])):
      temp_list = df['page_description'][teks][kalimat].split()
      if len(temp_list) > threshold:
          df['page_description'][teks][kalimat] = "-"
  return df

In [ ]:
def treat_ingredients(ing_list):
    output = []
    for ingredient in ing_list:
        ingredient_list = ingredient.split(' ')
        output.append("_".join(ingredient_list))
    return output
    
def make_LDAmodel(cleaned_description, num_topics, passes):
  # Load the list of documents
  ingredients_all = cleaned_description.apply(lambda x: ", ".join(x))

  # Use CountVectorizor to find three letter tokens, remove stop_words, 
  # remove tokens that don't appear in at least 20 documents,
  # remove tokens that appear in more than 20% of the documents
  vect = CountVectorizer(token_pattern='(?u)\\b\\w\\w\\w+\\b')

  # Fit and transform
  X = vect.fit_transform(ingredients_all)

  # Convert sparse matrix to gensim corpus.
  corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)

  # Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
  id_map = dict((v, k) for k, v in vect.vocabulary_.items())

  #fit LDA model
  #ldamodel = gensim.models.ldamodel.LdaModel(corpus,num_topics = 5,passes = 20, random_state = 0, id2word = id_map)
  ldamodel = gensim.models.ldamodel.LdaModel(corpus,num_topics = num_topics, passes = passes, random_state = 0, 
                                             id2word = id_map,chunksize=100,alpha='auto',per_word_topics=True, minimum_probability= 1E-9)
  
  # Chang idword to d
  word2id =dict((k, v) for k, v in vect.vocabulary_.items())
  d = corpora.Dictionary()
  d.id2token = id_map
  d.token2id = word2id

  # visualization
  pyLDAvis.enable_notebook()
  vis = pyLDAvis.gensim_models.prepare(ldamodel, corpus, d)

  return ldamodel, vis, corpus, id_map, d

def modelToText(modelLDA, numWords):
  # Output topic modeling dalam bentuk tulisan
  all_wordList = []
  for idx, topic in modelLDA.show_topics(formatted=False, num_words= numWords):
    word_list = [w[0] for w in topic]
    all_wordList.append(word_list)
    print('Topic:', idx+1)
    print(*word_list, sep=", ")
    print()
  return all_wordList

In [ ]:
# Stopword Malaysian
!gdown --id "1etWM37-sxCXvfy7JFc8Mwlv8rtXupeA5"
stopwordMY = set(line.strip() for line in open('sorted-updated-stopwords-ms.txt'))

# stopwords bahasa indonesia
stopwordID =  set(stopwords.words('indonesian'))
# stopwords english
stopwordEN = set(stopwords.words('english'))

# Dictionary malay (without food related)
# !gdown --id "1j0YZsdWWMs7Kfo1ROLykG6AKcCzG1SWg"
# malayDict = set(line.strip() for line in open('ms-dict.txt', encoding = "ISO-8859-1"))

# gabungkan stopwords malaysia, indonesia dan english
listStopword = stopwordID | stopwordMY
listStopword = listStopword | stopwordEN

# listStopword = listStopword | malayDict

# Tambah kata
listTambahan = {'potong', 'bahan', 'resepi', 'sudu', 'kacau', 'minit', 'buang', 'cawan', 'dicelur', 'dicelup', 'dihidangkan', 'kitchen', 
                'masak', 'best', 'ever', 'yang', 'sedap', 'delicious', 'aduhaaiii', 'nyer', 'resipi', 'sdt', 'sdm'}
listStopword = listStopword | listTambahan

def cleaning_text(ing_list, listStopword):
  removed = []
  for i in range(ing_list.shape[0]):
    tokens = " ".join(ing_list[i])
    # menghilangkan angka
    tokens = re.sub(r"\d+", "", tokens)
    
    # mengubah semua karakter menjadi lower case dan menghilangkan tanda baca dalam text
    tokens = tokens.lower()
    tokens = tokens.translate(str.maketrans("","",string.punctuation)).split()
    temp = []
    
    # penghapusan stopwords
    for t in tokens:
      if t not in listStopword:
        temp.append(t)
    kata = " ".join(temp)
    hasil = [kata]
    removed.append(hasil)

  return removed

def pre_cleaning_text(df, threshold = 20):
  # Pre-cleaning (ngilangin cerita), hapus jika kalimat lebih dari threshold
  for teks in range(len(df['page_description'])):
    for kalimat in range(len(df['page_description'][teks])):
      temp_list = df['page_description'][teks][kalimat].split()
      if len(temp_list) > threshold:
          df['page_description'][teks][kalimat] = "-"
  return df

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1etWM37-sxCXvfy7JFc8Mwlv8rtXupeA5
To: /content/sorted-updated-stopwords-ms.txt
100% 8.33k/8.33k [00:00<00:00, 8.85MB/s]


In [ ]:
def cleaning_title(ing_list, listStopword):
  removed = []
  for i in range(ing_list.shape[0]):
    tokens = ing_list[i]
    # menghilangkan angka
    tokens = re.sub(r"\d+", "", tokens)
    
    # mengubah semua karakter menjadi lower case dan menghilangkan tanda baca dalam text
    tokens = tokens.lower()
    tokens = tokens.translate(str.maketrans("","",string.punctuation)).split()
    temp = []
    
    # penghapusan stopwords
    for t in tokens:
      if t not in listStopword:
        temp.append(t)
    kata = " ".join(temp)
    hasil = [kata]
    removed.append(hasil)

  return removed

In [ ]:
# Untuk mencari probability per kata di topic dataset
def term_probability(ldamodel, d, term):
  res = ldamodel[d.doc2bow([term])][0]
  for i in range(1, len(res)+1):
    print(f"Topic {i} = {res[i-1][1]}")

def return_term_probability(ldamodel, d, term):
  res = ldamodel[d.doc2bow([term])][0]
  return res

def make_dictionary_terms_prob(ldamodel, id_map, d):
  prob_allTerms = {}
  word = id_map.values()
  for key in word:
    prob_allTerms[key] = return_term_probability(ldamodel, d, key)
  return prob_allTerms

In [ ]:
def concat_bahan_langkah(df):
  ''' Concat Kolom Bahan2 dan Langkah2 '''
  df["page_bahan_langkah"] = df["page_bahan"]
  for i in range(df.shape[0]):
    df["page_bahan_langkah"][i] = [*df["page_bahan"][i], *df["page_langkah"][i]] 
  return df

# Subprogram untuk Counter

In [ ]:
def wait_for_change(widget, value):
  future = asyncio.Future()
  def getvalue(change):
    # make the new value available
    future.set_result(change.new)
    widget.unobserve(getvalue, value)
  widget.observe(getvalue, value)
  return future

In [ ]:
# Tabel 1981
async def f():
  global tabel_1981, benar, salah, all_cat, all_suggestion

  for i in range(tabel_1981.shape[0]):
    clear_output()
    display(button)
    print()
    print(f"Row: {i+1}/{tabel_1981.shape[0]}")
    print(tabel_1981["Title"][i])
    temp = str(tabel_1981["Category"][i])
    temp = temp.translate(str.maketrans("","",string.punctuation))
    category_1981 = temp.split()

    suggestion = widgets.Text(
      value="",
      description='Usulan Class',
      placeholder = "input class disini dipisahkan dengan spasi",
      disabled=False,
      layout = widgets.Layout(width='600px', height='40px')
    )

    cat = widgets.SelectMultiple(
      options= category_1981,
      value=[],
      #rows=10,
      description='category',
      disabled=False
    )
    display(cat)
    display(suggestion)
    x = await wait_for_change(button, "value")

    temp = str(cat.value)
    temp = temp.translate(str.maketrans("","",string.punctuation))

    all_cat.append(temp.split())
    all_suggestion.append(suggestion.value.split())
    benar+= len(cat.value)
    salah+= len(cat.options) - len(cat.value)
  button.layout.display = 'none'
  print()
  print("=== END ===")
  print()
  print(f"benar: {benar}\nsalah: {salah}")

In [ ]:
# Wiki
async def f_wiki():
  global wiki_df, idx, all_cat
  for i in range(wiki_df.shape[0]):
    clear_output()
    display(button)
    print()
    print(f"Row: {i+1}/{wiki_df.shape[0]}")
    print(wiki_df["Title"][i])
    
    print("\n======Definition========\n")
    print(f"Indonesia: {wiki_df.Def_IND[i]}")
    print(f"Malaysia: {wiki_df.Def_MS[i]}")
    print(f"English: {wiki_df.Def_ENG[i]}")

    print("\n")
    class_suggest = widgets.Text(
      value="",
      description='Usulan Class',
      placeholder = "input class disini dipisahkan dengan spasi",
      disabled=False,
      layout = widgets.Layout(width='600px', height='40px')
    )
    display(class_suggest)
    x = await wait_for_change(button, "value")
    
    all_cat.append(class_suggest.value.split())
    idx.append(i)
    # if len(cat.value) > 0:
    #   benar+=1
    # else:
    #   salah+=1
    # benar+= len(cat.value)
    # salah+= len(cat.options) - len(cat.value)
  button.layout.display = 'none'
  print()
  print("=== END ===")
  print()
  print("hasil dari juri telah tersimpan di variabel all_cat")
  # print(f"benar: {benar}\nsalah: {salah}")

In [ ]:
# Wordnet
def submitRadioClicked(b):
  global catStatus, changesList, benarWordnet, salahWordnet, wnIdx, textBox
  print("Success")
  print()
  if catStatus.value == 'False':
    textBox = widgets.Text(
      placeholder='Masukkan Hypernym',
      description='Masukkan Hypernym:',
      disabled=False
    )
    display(textBox)
    salahWordnet += 1

    submitBtn = widgets.Button(description = "Submit")
    display(submitBtn)
    
    submitBtn.on_click(submitBtnClicked)
  else:
    changesList.append([wnIdx, "None"])
    benarWordnet += 1

def submitBtnClicked(b):
  global changesList, wnIdx, textBox
  print("Success")
  print()
  changesList.append([wnIdx, textBox.value])

async def fWordnet():
  global changesList, catStatus, tableWordnet, benarWordnet, salahWordnet, wnIdx
  changesList = []
  for wnIdx in range(tableWordnet.shape[0]):
    clear_output()
    display(button)
    print()
    print(f"Row: {wnIdx+1}/{tableWordnet.shape[0]}")
    print()
    print("Word:", tableWordnet["Word"][wnIdx])
    print("Synonym:", tableWordnet["Synonym_clean"][wnIdx])
    print("Hypernim:", tableWordnet["Hypernim_clean"][wnIdx])
    print("Hyponim:", tableWordnet["Hyponim_clean"][wnIdx])
    print()
    catStatus = widgets.RadioButtons (
      options=['True', 'False'],
      description='Hypernym (True/False?)',
      disabled=False
    )
    display(catStatus)

    submitBtnRadio = widgets.Button(description = "Submit")
    display(submitBtnRadio)

    submitBtnRadio.on_click(submitRadioClicked)
    x = await wait_for_change(button, "value")
  button.layout.display = 'none'
  print()
  print("=== END ===")

  # List = [[idx, hypernym], [idx, hypernym], ...]
  print("Changes:")
  print(changesList)

  print()
  print(f"Status Benar: {benarWordnet}\nStatus salah: {salahWordnet}")

def listToDataframe(resultList, dfWN):
  dfResult = pd.DataFrame(resultList, columns =['Index', 'Suggestion'])
  dfResult = dfResult.drop(columns=['Index'])
  concatDf = pd.concat([dfWN, dfResult], axis=1, join='inner')
  return concatDf

# Subprogram untuk GUI

In [ ]:
def on_button_clicked(b):
  '''
  Button add word and clean
  untuk menghapus kata sesuai dengan inputan user
  '''
  global listSW # nama stopwords baru
  global data #nama dataset
  # Display the message within the output widget.
  addition = set(teks.value.split())
  listSW = listSW | addition

  data["cleaned_description"] = cleaning_text(data["page_description"], listSW)
  with output:
    print("Success")

def run_button_clicked(b, lang="zsm"):
  '''button run model'''
  clear_output()
  display(teks)
  display(n_topics)
  display(iteration)

  # run_button.on_click(run_button_clicked)
  # button.on_click(on_button_clicked)

  display(button, output)
  display(run_button, output_run)
  global dfWordnet
  global ldamodel, vis, corpus, id_map, d
  print("Running..")
  ldamodel, vis, corpus, id_map, d = make_LDAmodel(data["cleaned_description"], n_topics.value, passes = iteration.value)
  
  display(vis)
  listTerms = modelToText(ldamodel, 30)
  dfWordnet = make_dataframe_wordnet(listTerms, lang)

  # Probabilitas Topic
  display(ldamodel.print_topics(num_topics = n_topics.value, num_words = 10))

# Subprogram untuk Wordnet

In [ ]:
def termsToWordnet(wordList, lang):
  ''' Mengambil hypo, syno, hyper terms/kata2 topiclda dari wordnet '''
  topicAll = []
  synonymAll = []
  hypernimAll = []
  hyponymAll = []
  definitionAll = []
  wordAll = []
  for topic in range(len(wordList)):
    for word in wordList[topic]:
      wordSyn = wn.synsets(word, lang=lang)
      topicAll.append(topic+1)
      wordAll.append(word)
      if len(wordSyn) > 0:
        synonymAll.append(wordSyn[0])
        hypernimAll.append(wordSyn[0].hypernyms())
        hyponymAll.append(wordSyn[0].hyponyms())
        definitionAll.append(wordSyn[0].definition())
      else:
        synonymAll.append("None")
        hypernimAll.append("None")
        hyponymAll.append("None")
        definitionAll.append("None")
  return topicAll, wordAll, synonymAll, hypernimAll, hyponymAll, definitionAll

def make_dataframe_wordnet(wordList, lang="zsm"):
  ''' membuat dataframe utk wordnet '''
  topic, wrd, syno, hype, hypo, definition = termsToWordnet(wordList, lang)

  finalList = zip(topic, wrd, syno, hype, hypo, definition)
  return pd.DataFrame(finalList, columns = ["No. Topic", "Word", "Synonym", "Hypernim", "Hyponim", "Definition"])

In [ ]:
def cleanWN(nym):
  '''mengembalikan -nym yang udah engga ada .n yang gitu gitu'''
  pattern_wn = r"('.+?\.)"
  result = []
  for i in nym:
    temp = []
    if i != "None" and len(i) > 0:
      for kata in i:
        temp.append(re.findall(pattern_wn, str(kata))[0][1:-1])
      result.append(temp)
    else:
      result.append("None")
  
  return result

def cleanWN_Synonym(syno):
  '''mengembalikan synonym yang udah engga ada .n yang gitu gitu'''
  pattern_wn = r"('.+?\.)"
  result = []
  for i in syno:
    strSyn = str(i)
    if strSyn != "None":
      result.append(re.findall(pattern_wn, str(strSyn))[0][1:-1])
    else:
      result.append("None")
  
  return result

# Subprogram untuk Ontology

In [ ]:
def translate(categories):
  res = []
  for i in range(len(categories)):
    categories[i] = convertStringListToList(categories[i])
    for j in range(len(categories[i])):
      if categories[i][j] == "buah": categories[i][j] = "Fruit"
      if categories[i][j] == "bumbu": categories[i][j] = "Spice"
      if categories[i][j] == "daging": categories[i][j] = "MeatAndPoultry"
      if categories[i][j] == "gula": categories[i][j] = "SugarAndSyrup"
      if categories[i][j] == "ikan": categories[i][j] = "Fish"
      if categories[i][j] == "kacang": categories[i][j] = "Nut"
      if categories[i][j] == "lemak": categories[i][j] = "FatAndOil"
      if categories[i][j] == "sayur": categories[i][j] = "Vegetable"
      if categories[i][j] == "serealia": categories[i][j] = "Cereal"
      if categories[i][j] == "susu": categories[i][j] = "Milk"
      if categories[i][j] == "telur": categories[i][j] = "Egg"
      if categories[i][j] == "umbi": categories[i][j] = "Tuber"
  return categories

def convertStringListToList(string):
  string = string.translate({ ord(c): None for c in "'[]," })
  li = list(string.split(" "))
  return li

def concatListInColumn(df):
  df["Category_New"] = df["Category"]
  for i in range(df.shape[0]):
    listRow = df["correct_category"][i] + df["suggestion"][i]
    df["Category_New"][i] = listRow
  return df

In [ ]:
def remove_space(sentence):
  return sentence.title().replace(" ", "")

In [ ]:
## Make Ontology for wordnet dataset
def makeOntology(onto, dfWordnet):
  with onto:
    for i in range(dfWordnet.shape[0]):
      word = dfWordnet["Word"][i]
      hyp = dfWordnet["Hypernim_clean"][i][0].capitalize()
      if dfWordnet["Hypernim_clean"][i] == "None":
        NewClass = types.new_class("Nothing", (Thing,))
        NewInstance = NewClass(word) # make an instance
      else:
        NewClass = types.new_class(hyp, (Thing,))
        NewInstance = NewClass(word)

In [ ]:
## Make Ontology for wikipedia dataset
def makeOntology_wiki(onto, df):
  with onto:
    for i in range(df.shape[0]):
      title = remove_space(df["Title"][i])
      for j in range(len(df["suggest"][i])):
        clas = df["suggest"][i][j].capitalize()
        if df["suggest"][i] == "None":
          NewClass = types.new_class("Nothing", (Thing,))
          NewInstance = NewClass(title) # make an instance
        else:
          NewClass = types.new_class(clas, (Thing,))
          NewInstance = NewClass(title)

In [ ]:
## Make Ontology for Table 1981 dataset
def makeOntologyTable1981(onto, dfTable):
  with onto:
    BaseClass = types.new_class("Recipe", (Thing,))
    for i in range(dfTable.shape[0]):
      title = remove_space(dfTable["Title"][i])
      for j in range(len(dfTable["Category_New"][i])):
        SubClass = types.new_class(dfTable["Category_New"][i][j], (BaseClass,))
        SubClassInstance = SubClass(title)

# Subprogram untuk Wikipedia

In [ ]:
# Subprogram untuk scraping wikipedia 
def wiki_indo(judul):
  try:
    wikipedia.set_lang("id")
    resultID = wikipedia.summary(judul, sentences=4)
    return resultID
  except:
    return 'Not Found'

def wiki_ms(judul):
  try:
    wikipedia.set_lang("ms")
    resultMS = wikipedia.summary(judul, sentences=4)
    return resultMS
  except:
    return 'Not Found'

def wiki_en(judul):
  try:
    wikipedia.set_lang("en")
    resultEN = wikipedia.summary(judul, sentences=4)
    return resultEN
  except:
    return 'Not Found'

def patRegex(pattern, definisi):
  isi = re.findall(pattern, definisi)
  if len(isi) > 0:
    final_isi = sambung_definisi(isi)
    return final_isi
  return 'Not Found'

In [ ]:
def sambung_definisi(arr_definition):
  '''mengembalikan definisi yang disambung ada di indeks 1'''
  hasil = ""
  for i in arr_definition:
    hasil = hasil + i[1].strip() + " "
  return hasil.strip()

In [ ]:
# Pattern Regex untuk definisi
pattern = r"(ialah|adalah|merupakan|merangkumi|kepada|makanan|consist |is a |consisting |made|is an )(.+?)[+,.;]"

# Subprogram untuk membuat definisi berdasarkan pattern regex
def generate_definition(array_title, pattern):
  cleaned = cleaning_title(array_title, listStopword)
  ind = []
  ms = []
  eng = []
  clean_title = []

  for judul in cleaned:
    splitted_judul = judul[0].split()
    if len(splitted_judul) > 1:
      juduls = splitted_judul[0] + " " + splitted_judul[1]
    elif len(splitted_judul) == 1:
      juduls = splitted_judul[0]
    else:
      juduls = 'None'
    
    if juduls != 'None':
      hasil_ind = wiki_indo(juduls)
      hasil_ms = wiki_ms(juduls)
      hasil_eng = wiki_en(juduls)

      isi_ind = patRegex(pattern, hasil_ind)
      isi_ms = patRegex(pattern, hasil_ms)
      isi_eng = patRegex(pattern, hasil_eng)

      ind.append(isi_ind)
      ms.append(isi_ms)
      eng.append(isi_eng)
      clean_title.append(judul[0])
    else:
      ind.append('Unknown')
      ms.append('Unknown')
      eng.append('Unknown')
      clean_title.append(judul[0])

  return ind, ms, eng, clean_title

# Subprogram untuk membuat dataframe definisi
def make_definition_dataframe(df_title, pattern):
  title = df_title
  ind, ms, eng, cleaned = generate_definition(df_title, pattern)

  final = zip(title, cleaned,ind, ms, eng)
  return pd.DataFrame(final, columns = ["Title", "Cleaned", "Def_IND", "Def_MS", "Def_ENG"])

# Subprogram untuk Tabel 1981

In [ ]:
# Tabel Makanan 1981 versi .txt diubah menjadi set
!gdown --id "1GaXKbBdvrSt52MezMV8nSvIHy5x-fvh2" #buah
!gdown --id "1CpdhUD_DEizVOMCJJbc6dGFOh9xeE_E0" #bumbu
!gdown --id "1Gn2vCNF5kRXYJU2GU7DGIlXiLA81UTtQ" #daging
!gdown --id "1U-O_-OFBEZHR0NTqZ43rQHvW_Pqqoc_R" #gula
!gdown --id "1HctjzZRqOrqTxfGd-cWcqYEoIDS6x0fh" #ikan
!gdown --id "1jZ_ngKTPvL-8DiKQ9YG0UPRTn6S9jZMC" #kacang
!gdown --id "1BE9IIFqIcDrgLJWKOA-vPplNwwe2jZZF" #lemak
!gdown --id "14ETHj-OUpL7QidjiI3pIMfMoOy9hCwd7" #sayur
!gdown --id "1760qlylrkaFX2HfMptJDgyPGEpN1m4Ux" #serealia
!gdown --id "1dFkhjqIuTT3MAWz_ZzCkEL5Zvnn0yxHR" #susu
!gdown --id "1iV756YrD7wJ_yGlwq0CenBZLHnSFn3-2" #telur
!gdown --id "1OtQ1Ij7S_GC-ROdoVzQn4i6cLMELUYAQ" #umbi

buah = set(line.strip() for line in open('buah.txt'))
daging = set(line.strip() for line in open('daging.txt'))
kacang = set(line.strip() for line in open('kacang.txt'))
sayur = set(line.strip() for line in open('sayur.txt'))
serealia = set(line.strip() for line in open('serealia.txt'))
umbi = set(line.strip() for line in open('umbi.txt'))

bumbu = set(line.strip() for line in open('bumbu.txt'))
gula = set(line.strip() for line in open('gula.txt'))
ikan = set(line.strip() for line in open('ikan.txt'))
lemak = set(line.strip() for line in open('lemak.txt'))
susu = set(line.strip() for line in open('susu.txt'))
telur = set(line.strip() for line in open('telur.txt'))

bumbu = set(map(lambda x: x.lower(), bumbu))
gula = set(map(lambda x: x.lower(), gula))
ikan = set(map(lambda x: x.lower(), ikan))
lemak = set(map(lambda x: x.lower(), lemak))
susu = set(map(lambda x: x.lower(), susu))
telur = set(map(lambda x: x.lower(), telur))

dict_cat = {"buah": buah, "bumbu": bumbu, "daging": daging, "gula": gula, "ikan": ikan,
            "kacang": kacang, "lemak": lemak, "sayur": sayur, "serealia": serealia, "susu": susu,
            "telur": telur, "umbi": umbi}

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1GaXKbBdvrSt52MezMV8nSvIHy5x-fvh2
To: /content/buah.txt
100% 590/590 [00:00<00:00, 867kB/s]
/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1CpdhUD_DEizVOMCJJbc6dGFOh9xeE_E0
To: /content/bumbu.txt
100% 657/657 [00:00<00:00, 941kB/s]
/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.goog

In [ ]:
# Subprogram untuk meng-kategorikan semua title pada kolom dataframe
def generate_category(array_title):
  cleaned = cleaning_title(array_title, listStopword)
  ctg = []
  ctg_all = []

  for judul in cleaned:
    splitted_judul = judul[0].split()
    ctg = []
    for jdl in splitted_judul:
      for key, value in dict_cat.items():
        if jdl in dict_cat[key]:
          ctg.append(key)
    ctg = set(ctg)
    ctg = list(ctg)
    ctg_all.append(ctg)

  return ctg_all

# Subprogram untuk meng-append kolom category ke dataframe definisi
def make_append_category_df(df_title, df_definition):
  category = generate_category(df_title)
  df2 = df_definition.assign(Category=category)
  return df2

# Kodingan Scraping & Category (Tabel 1981)

## Scrape Wikipedia & Categorization dari Tabel 1981

Azie Udang - Sapi  
https://colab.research.google.com/drive/1Dj9b42R3czpQZrMvMh6dk-xIeOfx7ED2?usp=sharing

Fiza - Tiffin Biru, cookpad tahu - udang, dan scraping malaysian Cooking Pdf  
https://colab.research.google.com/drive/1XZ08L9TtsJPvS7lo63_SdOQUonx1Z23g?usp=sharing

Cookpad (Ayam - Sapi + Malay Cooking PDF)  
https://colab.research.google.com/drive/13efKsxutJyuV5thDDpvN_qNo5L_OHIOK?authuser=1#scrollTo=3ncLd_YZiHc3 

## Scrape Website (Fiza-Tiffinbiru)

https://drive.google.com/drive/folders/1QVjafdJy7rBgi5pHE3aEJ9HVWPIrdSG9?usp=sharing

# Udang

In [ ]:
!gdown --id "1hg9S6bSgHS3yc6q3xgBDP1e6WxUV8JBw"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hg9S6bSgHS3yc6q3xgBDP1e6WxUV8JBw
To: /content/aziekitchen-udang.jl
100% 1.39M/1.39M [00:00<00:00, 94.1MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
udang=pd.read_json(r'/content/aziekitchen-udang.jl', lines = True)

#save dalam acar biar ngebut loading ntar.
udang.to_pickle(r'/content/raw_df.pkl')

udang = pre_cleaning_text(udang, threshold=20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
udang.head()

,list_scraping_time,url_search,list_item_url,list_item_title,list_item_author_name,list_item_author_url,list_item_post_date,list_item_post_comment_count,list_item_category,list_item_category_link,...,page_item_category,page_item_category_link,page_description,page_description_image_url,page_ulasan_count,page_comment,page_comment_time,page_comment_user,page_comment_user_url,page_comment_user_image_url
0,2021-12-31 06:01:24,https://www.aziekitchen.com/search?q=udang,https://www.aziekitchen.com/2020/02/rendang-pe...,Rendang Pedas Udang Galah Dengan Kacang Buncis,Azie,https://www.blogger.com/profile/00119325245731...,2/18/2020,\n1 Comment\n,\nfeatured\n,https://www.aziekitchen.com/search/label/featu...,...,"[featured, Lauk Pauk, Udang]",[https://www.aziekitchen.com/search/label/feat...,"[-, -, -, -, -, -, -, -, -, -, -, Jom layan re...",[https://1.bp.blogspot.com/-SGT7IwM3w88/XkotWQ...,1 ulasan,"[tengok saja pun dah tau sedap, kak., Ialah, k...","[23 Februari, 2020 14:53]",[Seri Kandi Di Tanah Jauhar],[https://www.blogger.com/profile/0485581192521...,[//2.bp.blogspot.com/-7IpnMVzXwzg/XY2woYDx1UI/...
1,2021-12-31 06:01:24,https://www.aziekitchen.com/search?q=udang,https://www.aziekitchen.com/2013/05/udang-gore...,Udang Goreng Butter Yang Paling Sedap,Azie,https://www.blogger.com/profile/02058252209863...,5/13/2013,\n117\n Comments\...,\nLauk Pauk\n,https://www.aziekitchen.com/search/label/Lauk%...,...,"[Lauk Pauk, Seafood, Udang]",[https://www.aziekitchen.com/search/label/Lauk...,"[-, Saya pernah menyediakan Udang Goreng Butte...",[https://3.bp.blogspot.com/-t1QVzkRla9g/UY8qkn...,117 ulasan,"[Salam Azie, mudah dan menyelerakan, mungkin s...","[13 Mei, 2013 08:52, 13 Mei, 2013 18:18, 09 Ja...","[Engineers Love Cooking, Azie, Unknown, Unknow...",[https://www.blogger.com/profile/1394017965595...,[//2.bp.blogspot.com/-CyoZGHKCgLk/T81X70uH1RI/...
2,2021-12-31 06:01:24,https://www.aziekitchen.com/search?q=udang,https://www.aziekitchen.com/2019/10/masak-loma...,Masak Lomak Cili Api Udang,Azie,https://www.blogger.com/profile/00119325245731...,10/30/2019,\n1 Comment\n,\nfeatured\n,https://www.aziekitchen.com/search/label/featu...,...,"[featured, Lauk Pauk, Masakan Tradisional, Sea...",[https://www.aziekitchen.com/search/label/feat...,"[-, -, -, -, -, -, -, Jom layan resepi masak l...",[https://1.bp.blogspot.com/-3ECL9_raEu8/Xbg33Y...,1 ulasan,[Sodapnyo],"[30 Oktober, 2019 21:33]",[Warisan Petani],[https://www.blogger.com/profile/1515705677504...,[//4.bp.blogspot.com/-I1pNzgenHDw/XO6UCe9q3UI/...
3,2021-12-31 06:01:24,https://www.aziekitchen.com/search?q=udang,https://www.aziekitchen.com/2018/06/sambal-tum...,Sambal Tumis Udang dengan Petai Buat Juita,Azie,https://www.blogger.com/profile/00119325245731...,6/08/2018,\n1 Comment\n,\nLauk Pauk\n,https://www.aziekitchen.com/search/label/Lauk%...,...,"[Lauk Pauk, Seafood, Udang]",[https://www.aziekitchen.com/search/label/Lauk...,"[-, -, Hmm terasa sangat cepat masa berlalu, R...",[https://4.bp.blogspot.com/-o3YDsu_j3rA/WxqXFd...,1 ulasan,[Wah....syoknya dengar Juita dapat beraya lama...,"[09 Jun, 2018 12:40]",[Norma Shah],[https://www.blogger.com/profile/1297446680819...,[//www.blogger.com/img/blogger_logo_round_35.png]
4,2021-12-31 06:01:24,https://www.aziekitchen.com/search?q=udang,https://www.aziekitchen.com/2011/09/mee-udang-...,Mee Udang Istimewa,Azie,https://www.blogger.com/profile/02058252209863...,9/18/2011,\n2\n Comments\n ...,\nKuih Muih\n,https://www.aziekitchen.com/search/label/Kuih%...,...,"[Kuih Muih, Mee/Mee Hoon]",[https://www.aziekitchen.com/search/label/Kuih...,"[\n, -, -, \n, -, \n, OK jom kita layan gambar...",[https://2.bp.blogspot.com/-ispM0mhrQRE/TnV2z6...,2 ulasan,[Assalamualaikum kak azie.. sy selalu jenguk b...,"[21 September, 2015 16:39, 23 Julai, 2017 13:37]","[Unknown, Unknown]",[https://www.blogger.com/profile/0363961842120...,[//www.blogger.com/img/blogger_logo_round_35.p...


In [ ]:
#boleh run sekiranya ingin masukkan column lain dari dataset, kalo ga ya jangan run donk
# convert_to_list = ['page_description']

# for c in convert_to_list:
#     print(data[c])
#     data[c] = data[c].apply(lambda x: eval(x))

## GUI Udang

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = udang.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5     -0.221905 -0.048307       1        1  55.774387
1     -0.156442  0.035967       2        1  20.617019
2      0.045597 -0.289746       3        1   6.325500
3     -0.070228  0.111930       4        1   5.998331
6      0.000154  0.114006       5        1   5.156608
0      0.244939  0.045244       6        1   3.691350
4      0.157885  0.030907       7        1   2.436806, topic_info=               Term        Freq       Total Category  logprob  loglift
23             gula  176.000000  176.000000  Default  30.0000  30.0000
28           bawang  462.000000  462.000000  Default  29.0000  29.0000
6             udang  439.000000  439.000000  Default  28.0000  28.0000
11           santan  111.000000  111.000000  Default  27.0000  27.0000
143             mee   97.000000   97.000000  Default  26.0000  26.0000
24            garam  291.000000  291.000000  Default  25.0000  25.0000
51           goreng  148.000000  148.000000  Default  24.0000  24.0000
136             air  148.000000  148.000000  Default  23.0000  23.0000
31            putih  191.000000  191.000000  Default  22.0000  22.0000
199           kicap   49.000000   49.000000  Default  21.0000  21.0000
16             daun  146.000000  146.000000  Default  20.0000  20.0000
35            hidup   58.000000   58.000000  Default  19.0000  19.0000
55            telur   85.000000   85.000000  Default  18.0000  18.0000
9            kacang   72.000000   72.000000  Default  17.0000  17.0000
29            merah  222.000000  222.000000  Default  16.0000  16.0000
131             sos   94.000000   94.000000  Default  15.0000  15.0000
20           batang  100.000000  100.000000  Default  14.0000  14.0000
96             kuah   57.000000   57.000000  Default  13.0000  13.0000
166            gram   65.000000   65.000000  Default  12.0000  12.0000
198           karot   40.000000   40.000000  Default  11.0000  11.0000
167           nenas   29.000000   29.000000  Default  10.0000  10.0000
27             biji  231.000000  231.000000  Default   9.0000   9.0000
37             cili  334.000000  334.000000  Default   8.0000   8.0000
221           panas   40.000000   40.000000  Default   7.0000   7.0000
206           sayur   55.000000   55.000000  Default   6.0000   6.0000
429          rendam   30.000000   30.000000  Default   5.0000   5.0000
208            ikan   83.000000   83.000000  Default   4.0000   4.0000
190            nasi   48.000000   48.000000  Default   3.0000   3.0000
87           keping  119.000000  119.000000  Default   2.0000   2.0000
72             kari   48.000000   48.000000  Default   1.0000   1.0000
88             asam  101.335533  101.961995   Topic1  -4.1633   0.5777
108          sambal   63.793925   64.420960   Topic1  -4.6260   0.5741
184            jawa   47.405843   48.032023   Topic1  -4.9229   0.5707
122           petai   35.014788   35.640580   Topic1  -5.2259   0.5661
207           lemak   26.657558   27.285940   Topic1  -5.4986   0.5606
226         belacan   24.971958   25.598273   Topic1  -5.5639   0.5591
502           kubis   24.859154   25.535807   Topic1  -5.5685   0.5570
608          terung   22.302356   22.930066   Topic1  -5.6770   0.5561
227           wangi   19.816392   20.444982   Topic1  -5.7952   0.5526
250             pes   17.508539   18.135962   Topic1  -5.9190   0.5486
161           gulai   17.088412   17.732642   Topic1  -5.9433   0.5468
10           buncis   16.268109   16.894024   Topic1  -5.9925   0.5461
299          tengok   16.131431   16.758306   Topic1  -6.0009   0.5457
97           cantik   15.199088   15.826258   Topic1  -6.0605   0.5434
117            saiz   15.052386   15.687663   Topic1  -6.0702   0.5425
703           bayam   13.065630   13.692239   Topic1  -6.2117   0.5370
254        ketumbar   13.061360   13.691431   Topic1  -6.2120   0.5367
169           halba   12.270400   12.896274   Topic1  -6.2745  

Topic: 1
kicap, teow, kue, taugeh, kucai, sos, kangkung, talam, ikan, manis, prego, isi, kerang, bilis, telur, cap, kuey, peket, ikat, bihun, kukus, tiram, mee, baby, mas, berlauk, gaul, beras, salam, hancur

Topic: 2
bawang, mee, cili, daun, sos, udang, goreng, biji, telur, garam, air, limau, kuning, kisar, kering, tomato, hoon, kicap, putih, sotong, ulas, hiris, ayam, merah, sayur, tumis, sawi, ekor, tumbuk, buah

Topic: 3
nenas, hidup, kuah, santan, rebus, rendam, maggi, batang, kacang, goreng, tauhu, biji, kari, tempe, perencah, secukup, ketuk, nasi, mendidih, serai, periuk, kotak, letak, rempah, tos, pajeri, kentang, biarkan, timun, sengkuang

Topic: 4
bawang, goreng, daun, purut, putih, merah, gram, karot, telur, sayur, tauhu, sayuran, ayam, udang, ketepikan, tangkai, cili, lada, gaul, digoreng, ulas, tabur, sup, serbuk, tepung, cincang, panaskan, hiris, dihiris, lobak

Topic: 5
tepung, gula, mentega, air, susu, telur, masin, cucur, butter, dikacau, tuang, direbus, gandum, pencic

[(0,
  '0.043*"kicap" + 0.031*"teow" + 0.025*"kue" + 0.024*"taugeh" + 0.021*"kucai" + 0.017*"sos" + 0.014*"kangkung" + 0.012*"talam" + 0.012*"ikan" + 0.011*"manis"'),
 (1,
  '0.043*"bawang" + 0.039*"mee" + 0.033*"cili" + 0.028*"daun" + 0.028*"sos" + 0.028*"udang" + 0.024*"goreng" + 0.022*"biji" + 0.017*"telur" + 0.017*"garam"'),
 (2,
  '0.038*"nenas" + 0.028*"hidup" + 0.026*"kuah" + 0.025*"santan" + 0.023*"rebus" + 0.023*"rendam" + 0.022*"maggi" + 0.021*"batang" + 0.021*"kacang" + 0.019*"goreng"'),
 (3,
  '0.039*"bawang" + 0.033*"goreng" + 0.023*"daun" + 0.023*"purut" + 0.022*"putih" + 0.022*"merah" + 0.022*"gram" + 0.022*"karot" + 0.020*"telur" + 0.019*"sayur"'),
 (4,
  '0.034*"tepung" + 0.026*"gula" + 0.023*"mentega" + 0.021*"air" + 0.020*"susu" + 0.019*"telur" + 0.017*"masin" + 0.015*"cucur" + 0.013*"butter" + 0.011*"dikacau"'),
 (5,
  '0.052*"udang" + 0.049*"bawang" + 0.038*"cili" + 0.035*"garam" + 0.028*"merah" + 0.027*"kisar" + 0.025*"biji" + 0.024*"gula" + 0.022*"ulas" + 0.021*"

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
udangProb = make_dictionary_terms_prob(ldamodel, id_map, d)
udangProb["mee"] #probabilitas kata mee pada setiap topik

[(0, 0.017503638),
 (1, 0.5931365),
 (2, 0.029725397),
 (3, 0.032974172),
 (4, 0.013196567),
 (5, 0.28782016),
 (6, 0.02564358)]

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
udangWN = dfWordnet.copy()

udangWN["Synonym_clean"] = cleanWN_Synonym(udangWN["Synonym"])
udangWN["Hypernim_clean"] = cleanWN(udangWN["Hypernim"])
udangWN["Hyponim_clean"] = cleanWN(udangWN["Hyponim"])
udangWN

### Make Ontology

In [ ]:
# Make Ontology
azieUdangOnto = get_ontology("http://test.org/azieUdang.owl")
makeOntology(azieUdangOnto, udangWN)
azieUdangOnto.save(file = "azieUdang.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "16ApevKhMErVji7vBeTB2cSZnWW46Scwa"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=16ApevKhMErVji7vBeTB2cSZnWW46Scwa
To: /content/AzieUdangFinal.xlsx
100% 22.8k/22.8k [00:00<00:00, 28.7MB/s]


In [ ]:
wiki_udang = pd.read_excel("AzieUdangFinal.xlsx").iloc[:, 1:]

# Translate
wiki_udang["Category"] = translate(wiki_udang["Category"])
wiki_udang

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_udang[["Title", "Category"]].iloc[:].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

ToggleButton(value=False, button_style='success', description='Next row', icon='check', tooltip='Description')


Row: 1/194
Rendang Pedas Udang Galah Dengan Kacang Buncis


SelectMultiple(description='category', options=('Vegetable', 'Nut', 'Fish', 'MeatAndPoultry'), value=())

Text(value='', description='Usulan Class', layout=Layout(height='40px', width='600px'), placeholder='input cla…

In [ ]:
udangTabel_new = tabel_1981.copy()
udangTabel_new["correct_category"] = all_cat
udangTabel_new["suggestion"] = all_suggestion

# Export Result to File
udangTabel_new.to_excel("udangTabel_Result.xlsx")
udangTabel_new

,Title,Category,correct_category,suggestion
0,Rendang Pedas Udang Galah Dengan Kacang Buncis,"[Vegetable, Nut, Fish, MeatAndPoultry]","[Vegetable, Fish]",[CuisineMeatAndPoultry]
1,Udang Goreng Butter Yang Paling Sedap,[Fish],[Fish],[]
2,Masak Lomak Cili Api Udang,"[Vegetable, Fish]",[Vegetable],[]


In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "udangTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/googleapiclient/discovery_cache/file_cache.py", line 33, in <module>
    from oauth2client.contrib.locked_file import LockedFile
ModuleNotFoundError: No module named 'oauth2client.contrib.locked_file'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/googleapiclient/discovery_cache/file_cache.py", line 37, in <module>
    from oauth2client.locked_file import LockedFile
ModuleNotFoundError: No module named 'oauth2client.locked_file'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/googleapiclient/discovery_cache/__init__.py", line 44, in autodetect
    from . import file_cache
  File "/usr/local/lib/python3.7/dist-packages/googleapiclient/discovery_cache/file_cache.py", line 41, in <module>
    "file_cach

### Update Ontology NFO

In [ ]:
# NFO Ontology
!gdown "1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru"
table1981_Onto = get_ontology(r'/content/NFOversion2April22.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
udangTabel_new = concatListInColumn(udangTabel_new)
udangTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, udangTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

## Counter Wikipedia

In [ ]:
wiki_df = wiki_udang.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
udangWiki_new = wiki_df.copy()
udangWiki_new["suggest"] = all_cat

# Export Result to file
udangWiki_new.to_excel("udangWiki_Result.xlsx")
udangWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "udangWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieUdangOntoWiki = get_ontology("http://test.org/azieUdang_onto_wiki.owl")
makeOntology_wiki(azieUdangOntoWiki, udangWiki_new)
azieUdangOntoWiki.save(file = "azieUdang_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = udangWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

ToggleButton(value=False, button_style='success', description='Next row', icon='check', tooltip='Description')


Row: 1/210

Word: kicap
Synonym: soy_sauce
Hypernim: ['condiment']
Hyponim: None



RadioButtons(description='Hypernym (True/False?)', options=('True', 'False'), value='True')

Button(description='Submit', style=ButtonStyle())

In [ ]:
# Export Result to File
udangWN_new = listToDataframe(changesList, tableWordnet)
udangWN_new.to_excel("udangWN_Result.xlsx")
udangWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "udangWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Ayam

In [ ]:
!gdown --id "1hDDpuyUsZ9HyuG5tDM7LGzTMVifNhkrK"

In [ ]:
ayam=pd.read_json(r'/content/aziekitchen-ayam.jl', lines = True)

ayam.to_pickle(r'/content/raw_df.pkl')
ayam = pre_cleaning_text(ayam, threshold=20)
# ayam["cleaned_description"] = cleaning_text(ayam["page_description"])

# ingredients_all = ayam["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Ayam

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = ayam.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
ayamProb = make_dictionary_terms_prob(ldamodel, id_map, d)
ayamProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
ayamWN = dfWordnet.copy()

ayamWN["Synonym_clean"] = cleanWN_Synonym(ayamWN["Synonym"])
ayamWN["Hypernim_clean"] = cleanWN(ayamWN["Hypernim"])
ayamWN["Hyponim_clean"] = cleanWN(ayamWN["Hyponim"])
ayamWN

### Make Ontology

In [ ]:
# Make Ontology
azieAyamOnto = get_ontology("http://test.org/azieAyam.owl")
makeOntology(azieAyamOnto, ayamWN)
azieAyamOnto.save(file = "azieAyam.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1GdNFDSXOMR9C4TRWfGn7N8Zr_IaH0-Ur"

In [ ]:
wiki_ayam = pd.read_excel("AzieAyamFinal.xlsx").iloc[:, 1:]

# Translate
wiki_ayam["Category"] = translate(wiki_ayam["Category"])
wiki_ayam.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_ayam[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
ayamTabel_new = tabel_1981.copy()
ayamTabel_new["correct_category"] = all_cat
ayamTabel_new["suggestion"] = all_suggestion

# Export Result to File
ayamTabel_new.to_excel("ayamTabel_Result.xlsx")
ayamTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
ayamTabel_new = concatListInColumn(ayamTabel_new)
ayamTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, ayamTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

## Counter Wikipedia

In [ ]:
wiki_df = wiki_ayam.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
ayamWiki_new = wiki_df.copy()
ayamWiki_new["suggest"] = all_cat

# Export Result to file
ayamWiki_new.to_excel("ayamWiki_Result.xlsx")
ayamWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieAyamOntoWiki = get_ontology("http://test.org/azieAyam_onto_wiki.owl")
makeOntology_wiki(azieAyamOntoWiki, ayamWiki_new)
azieAyamOntoWiki.save(file = "azieAyam_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = ayamWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
ayamWN_new = listToDataframe(changesList, tableWordnet)
ayamWN_new.to_excel("ayamWN_Result.xlsx")
ayamWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Ikan

In [ ]:
!gdown --id "1h_hNQO3qLqUyvQ5v3GwPGk8YGbTpBrmd"

In [ ]:
ikan=pd.read_json(r'/content/aziekitchen-ikan.jl', lines = True)

#save dalam acar biar ngebut loading ntar.
ikan.to_pickle(r'/content/raw_df.pkl')

ikan = pre_cleaning_text(ikan, threshold=20)
# ikan["cleaned_description"] = cleaning_text(ikan["page_description"])
# ingredients_all = ikan["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Ikan

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = ikan.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
ikanProb = make_dictionary_terms_prob(ldamodel, id_map, d)
ikanProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
ikanWN = dfWordnet.copy()

ikanWN["Synonym_clean"] = cleanWN_Synonym(ikanWN["Synonym"])
ikanWN["Hypernim_clean"] = cleanWN(ikanWN["Hypernim"])
ikanWN["Hyponim_clean"] = cleanWN(ikanWN["Hyponim"])
ikanWN

### Make Ontology

In [ ]:
# Make Ontology
azieIkanOnto = get_ontology("http://test.org/azieIkan.owl")
makeOntology(azieIkanOnto, ikanWN)
azieIkanOnto.save(file = "azieIkan.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "12lE3RokN12jUUuDyAbl0_QNmKIv1IUrd"

In [ ]:
wiki_ikan = pd.read_excel("AzieIkanFinal.xlsx").iloc[:, 1:]

# Translate
wiki_ikan["Category"] = translate(wiki_ikan["Category"])
wiki_ikan.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_ikan[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
ikanTabel_new = tabel_1981.copy()
ikanTabel_new["correct_category"] = all_cat
ikanTabel_new["suggestion"] = all_suggestion

# Export Result to File
ikanTabel_new.to_excel("ikanTabel_Result.xlsx")
ikanTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
ikanTabel_new = concatListInColumn(ikanTabel_new)
ikanTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, ikanTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

## Counter Wikipedia

In [ ]:
wiki_df = wiki_ikan.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
ikanWiki_new = wiki_df.copy()
ikanWiki_new["suggest"] = all_cat

# Export Result to file
ikanWiki_new.to_excel("ikanWiki_Result.xlsx")
ikanWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieIkanOntoWiki = get_ontology("http://test.org/azieIkan_onto_wiki.owl")
makeOntology_wiki(azieIkanOntoWiki, ikanWiki_new)
azieIkanOntoWiki.save(file = "azieIkan_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = ikanWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
ikanWN_new = listToDataframe(changesList, tableWordnet)
ikanWN_new.to_excel("ikanWN_Result.xlsx")
ikanWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Kambing

In [ ]:
!gdown --id "1hPTtLXu7jgsmWegZwxU6M-Swbb_zvhgM"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hPTtLXu7jgsmWegZwxU6M-Swbb_zvhgM
To: /content/aziekitchen-kambing.jl
100% 916k/916k [00:00<00:00, 142MB/s]


In [ ]:
kambing=pd.read_json(r'/content/aziekitchen-kambing.jl', lines = True)

#save dalam acar biar ngebut loading ntar.
kambing.to_pickle(r'/content/raw_df.pkl')

kambing = pre_cleaning_text(kambing)

# kambing["cleaned_description"] = cleaning_text(kambing["page_description"])
# ingredients_all = kambing["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Kambing

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = kambing.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.143016  0.048356       1        1  42.802709
5      0.173085 -0.083628       2        1  23.400268
2      0.023307  0.069393       3        1   8.776384
3     -0.091653  0.101684       4        1   7.537965
6     -0.154680 -0.101018       5        1   7.291537
0     -0.002055 -0.039683       6        1   6.657953
1     -0.091020  0.004898       7        1   3.533184, topic_info=                Term        Freq       Total Category  logprob  loglift
63               air   51.000000   51.000000  Default  30.0000  30.0000
28            bawang  135.000000  135.000000  Default  29.0000  29.0000
65            daging   55.000000   55.000000  Default  28.0000  28.0000
117           rempah   35.000000   35.000000  Default  27.0000  27.0000
27            kunyit   29.000000   29.000000  Default  26.0000  26.0000
50             garam   78.000000   78.000000  Default  25.0000  25.0000
53            goreng   25.000000   25.000000  Default  24.0000  24.0000
126              sup   13.000000   13.000000  Default  23.0000  23.0000
146             asam   33.000000   33.000000  Default  22.0000  22.0000
55             hiris   54.000000   54.000000  Default  21.0000  21.0000
145          kerisik   17.000000   17.000000  Default  20.0000  20.0000
56              cili   55.000000   55.000000  Default  19.0000  19.0000
51              gula   63.000000   63.000000  Default  18.0000  18.0000
139              mee    8.000000    8.000000  Default  17.0000  17.0000
9             keping   32.000000   32.000000  Default  16.0000  16.0000
123             ulas   43.000000   43.000000  Default  15.0000  15.0000
313             ayam   44.000000   44.000000  Default  14.0000  14.0000
54              daun   57.000000   57.000000  Default  13.0000  13.0000
34             halia   39.000000   39.000000  Default  12.0000  12.0000
23            serbuk   42.000000   42.000000  Default  11.0000  11.0000
26             putih   52.000000   52.000000  Default  10.0000  10.0000
74            kering   26.000000   26.000000  Default   9.0000   9.0000
30            tomato   33.000000   33.000000  Default   8.0000   8.0000
33            tumbuk   16.000000   16.000000  Default   7.0000   7.0000
122            merah   52.000000   52.000000  Default   6.0000   6.0000
17             manis   29.000000   29.000000  Default   5.0000   5.0000
82             tabur   11.000000   11.000000  Default   4.0000   4.0000
144           santan   28.000000   28.000000  Default   3.0000   3.0000
18              biji   82.000000   82.000000  Default   2.0000   2.0000
40             telur   14.000000   14.000000  Default   1.0000   1.0000
482              sos   17.964280   18.572085   Topic1  -4.6203   0.8153
76           lapisan   17.039831   17.636349   Topic1  -4.6731   0.8142
29              dadu   14.158284   14.799927   Topic1  -4.8584   0.8042
45              susu   12.343233   12.938914   Topic1  -4.9956   0.8014
48           pewarna   11.406404   11.999922   Topic1  -5.0745   0.7978
36             helai   10.465937   11.060100   Topic1  -5.1606   0.7934
35           yoghurt   10.465690   11.060159   Topic1  -5.1606   0.7933
97             halus    9.526472   10.120629   Topic1  -5.2546   0.7881
12            kuntum    8.585812    9.180713   Topic1  -5.3586   0.7816
96             bulat    7.646523    8.241199   Topic1  -5.4744   0.7737
30            tomato   31.127446   33.723992   Topic1  -4.0706   0.7684
488             cair    6.707657    7.302113   Topic1  -5.6054   0.7637
313             ayam   40.520666   44.398706   Topic1  -3.8069   0.7572
151             lauk    5.770212    6.362956   Topic1  -5.7560   0.7508
155          beriani    5.770012    6.362911   Topic1  -5.7560   0.7508
93        kesemuanya    5.769673    6.362847   Topic1  -5.7561   0.7507
78          menutupi    5.769630    6.362864   Topic1  -5.7561   0.7507
24          ket

Topic: 1
daging, kunyit, garam, bawang, separuh, kering, putih, hiris, panas, rendang, hati, air, ulas, cili, serbuk, klik, sambal, itik, paru, lupa, kisar, kerutuk, batang, tabur, kerisik, nasi, serai, dihiris, nak, hidup

Topic: 2
air, garam, abang, asam, manis, biji, lembu, lupa, serbuk, arwah, ingatan, arahan, perbanyakkan, kunyit, ubi, keledek, ditumis, penyediaan, tuang, kampung, kelantan, gula, daging, bunga, cili, limau, rebus, padi, kambing, rebusan

Topic: 3
bawang, sup, goreng, merah, hiris, ulas, daun, garam, tumbuk, telur, putih, tabur, cili, biji, entri, pucuk, sedia, lada, kelapa, periuk, gula, tulang, ketepikan, lembu, hati, hitam, panaskan, layan, tumis, tauhu

Topic: 4
mee, labu, gula, kampung, layan, merah, cili, dihiris, rebus, kisar, kicap, stick, gambar, santan, tumbuk, garam, bawang, biji, ulas, tumis, buah, kambing, mencuba, nasi, selamat, kuah, kangkung, entri, talam, tinggal

Topic: 5
bawang, biji, daun, cili, ayam, garam, serbuk, tomato, nasi, air, putih, mer

[(0,
  '0.021*"daging" + 0.020*"kunyit" + 0.018*"garam" + 0.016*"bawang" + 0.012*"separuh" + 0.012*"kering" + 0.011*"putih" + 0.010*"hiris" + 0.010*"panas" + 0.009*"rendang"'),
 (1,
  '0.046*"air" + 0.015*"garam" + 0.011*"abang" + 0.008*"asam" + 0.008*"manis" + 0.007*"biji" + 0.007*"lembu" + 0.007*"lupa" + 0.007*"serbuk" + 0.007*"arwah"'),
 (2,
  '0.041*"bawang" + 0.027*"sup" + 0.022*"goreng" + 0.019*"merah" + 0.017*"hiris" + 0.014*"ulas" + 0.013*"daun" + 0.013*"garam" + 0.012*"tumbuk" + 0.012*"telur"'),
 (3,
  '0.022*"mee" + 0.013*"labu" + 0.011*"gula" + 0.011*"kampung" + 0.011*"layan" + 0.009*"merah" + 0.008*"cili" + 0.007*"dihiris" + 0.007*"rebus" + 0.007*"kisar"'),
 (4,
  '0.035*"bawang" + 0.033*"biji" + 0.025*"daun" + 0.024*"cili" + 0.022*"ayam" + 0.021*"garam" + 0.017*"serbuk" + 0.017*"tomato" + 0.015*"nasi" + 0.013*"air"'),
 (5,
  '0.049*"bawang" + 0.035*"daging" + 0.032*"gula" + 0.030*"rempah" + 0.027*"garam" + 0.024*"hiris" + 0.023*"keping" + 0.023*"halia" + 0.022*"asam" + 0.0

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
kambingProb = make_dictionary_terms_prob(ldamodel, id_map, d)
kambingProb["bawang"] #probabilitas kata bawang pada setiap topik

[(0, 0.020178424),
 (1, 0.054989588),
 (2, 0.031142846),
 (3, 0.03367496),
 (4, 0.01622466),
 (5, 0.812446),
 (6, 0.03134356)]

## Wordnet

In [ ]:
kambingWN = dfWordnet.copy()

kambingWN["Synonym_clean"] = cleanWN_Synonym(kambingWN["Synonym"])
kambingWN["Hypernim_clean"] = cleanWN(kambingWN["Hypernim"])
kambingWN["Hyponim_clean"] = cleanWN(kambingWN["Hyponim"])
kambingWN

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,daging,Synset('fiber.n.05'),[Synset('fabric.n.01')],[],a leatherlike material made by compressing lay...,fiber,[fabric],None
1,1,kunyit,Synset('turmeric.n.02'),[Synset('flavorer.n.01')],[],ground dried rhizome of the turmeric plant use...,turmeric,[flavorer],None
2,1,garam,Synset('strategic_arms_limitation_talks.n.01'),[],[],negotiations between the United States and the...,strategic_arms_limitation_talks,None,None
3,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
4,1,separuh,Synset('partially.r.01'),[],[],in part; in some degree; not wholly,partially,None,None
5,1,kering,Synset('dead.s.17'),[],[],devoid of activity,dead,None,None
6,1,putih,Synset('canescent.s.02'),[],[],covered with fine whitish hairs or down,canescent,None,None
7,1,hiris,Synset('piece.n.08'),[Synset('helping.n.01')],"[Synset('cutlet.n.01'), Synset('fillet.n.02')]",a serving that has been cut from a larger portion,piece,[helping],"[cutlet, fillet]"
8,1,panas,Synset('warm.v.02'),[Synset('change.v.01')],[Synset('chafe.v.06')],make warm or warmer,warm,[change],[chafe]
9,1,rendang,Synset('shady.s.04'),[],[],filled with shade,shady,None,None


### Make Ontology

In [ ]:
# Make Ontology
azieKambingOnto = get_ontology("http://test.org/azieKambing.owl")
makeOntology(azieKambingOnto, kambingWN)
azieKambingOnto.save(file = "azieKambing.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1i4ibWRh3bkxlOW72pJfF1g9rLER-4gbt"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1i4ibWRh3bkxlOW72pJfF1g9rLER-4gbt
To: /content/AzieKambingFinal.xlsx
100% 16.3k/16.3k [00:00<00:00, 23.3MB/s]


In [ ]:
wiki_kambing = pd.read_excel("AzieKambingFinal.xlsx").iloc[:, 1:]

# Translate
wiki_kambing["Category"] = translate(wiki_kambing["Category"])
wiki_kambing.head()

,Title,Cleaned,Def_IND,Def_MS,Def_ENG,Category
0,Nasi Beriani Kambing Pakistan Yang Memang Supe...,nasi beriani kambing pakistan super duper,Not Found,hidangan nasi campur dari Benua Kecil India sa...,Not Found,"['serealia', 'daging']"
1,Sup Kambing Berempah Yang Sangat Sedap,sup kambing berempah,hidangan sup daging kambing yang lazim ditemuk...,sup kambing yang biasa ditemui dalam masakan I...,Southeast Asian mutton soup,['daging']
2,Gulai Kambing Kelantan,gulai kambing kelantan,masakan berbahan baku daging ayam bumbunya yan...,masakan mengandungi daging,type of food containing rich common name to re...,['daging']
3,Beriani Daging Kambing,beriani daging kambing,Not Found,Not Found,Not Found,['daging']
4,Gulai Kampung Daging Kambing,gulai kampung daging kambing,Not Found,gulai itik khas dari Aceh gulai itik yang dibu...,type of food containing rich common name to re...,['daging']


## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_kambing[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

<Task pending coro=<f() running at <ipython-input-15-e5a452d805f8>:2>>

In [ ]:
kambingTabel_new = tabel_1981.copy()
kambingTabel_new["correct_category"] = all_cat
kambingTabel_new["suggestion"] = all_suggestion

# Export Result to File
kambingTabel_new.to_excel("kambingTabel_Result.xlsx")
kambingTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
kambingTabel_new = concatListInColumn(kambingTabel_new)
kambingTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, kambingTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_kambing.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

<Task pending coro=<f_wiki() running at <ipython-input-16-4627bd525651>:2>>

In [ ]:
kambingWiki_new = wiki_df.copy()
kambingWiki_new["suggest"] = all_cat

# Export Result to file
kambingWiki_new.to_excel("kambingWiki_Result.xlsx")
kambingWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieKambingOntoWiki = get_ontology("http://test.org/azieKambing_onto_wiki.owl")
makeOntology_wiki(azieKambingOntoWiki, kambingWiki_new)
azieKambingOntoWiki.save(file = "azieKambing_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = kambingWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

<Task pending coro=<fWordnet() running at <ipython-input-17-59b16f86cfbe>:28>>

In [ ]:
# Export Result to File
kambingWN_new = listToDataframe(changesList, tableWordnet)
kambingWN_new.to_excel("kambingWN_Result.xlsx")
kambingWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Tempe

In [ ]:
!gdown --id "1hXhZXEAiqEeWn5_0SiC3JLTnQiMSDYb3"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hXhZXEAiqEeWn5_0SiC3JLTnQiMSDYb3
To: /content/aziekitchen-tempe.jl
100% 195k/195k [00:00<00:00, 89.7MB/s]


In [ ]:
tempe=pd.read_json(r'/content/aziekitchen-tempe.jl', lines = True)
tempe.to_pickle(r'/content/raw_df.pkl')
tempe = pre_cleaning_text(tempe, threshold=20)
# tempe["cleaned_description"] = cleaning_text(tempe["page_description"])
# ingredients_all = tempe["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Tempe

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = tempe.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.199445 -0.050918       1        1  40.551482
2     -0.082372 -0.162198       2        1  26.668087
6     -0.099134 -0.004899       3        1   9.610349
3      0.003299  0.044380       4        1   9.437549
0      0.037760  0.009881       5        1   7.900587
4     -0.018298  0.109559       6        1   5.804739
5     -0.040700  0.054195       7        1   0.027207, topic_info=              Term       Freq      Total Category  logprob  loglift
66           bilis  27.000000  27.000000  Default  30.0000  30.0000
65            ikan  28.000000  28.000000  Default  29.0000  29.0000
41          bawang  34.000000  34.000000  Default  28.0000  28.0000
132         batang  30.000000  30.000000  Default  27.0000  27.0000
126         kacang  22.000000  22.000000  Default  26.0000  26.0000
144          udang  20.000000  20.000000  Default  25.0000  25.0000
136         rendam  21.000000  21.000000  Default  24.0000  24.0000
76            gula  20.000000  20.000000  Default  23.0000  23.0000
145          hidup  17.000000  17.000000  Default  22.0000  22.0000
12           tempe  29.000000  29.000000  Default  21.0000  21.0000
83          garing  11.000000  11.000000  Default  20.0000  20.0000
141          tauhu  15.000000  15.000000  Default  19.0000  19.0000
71           kisar  34.000000  34.000000  Default  18.0000  18.0000
146         santan  21.000000  21.000000  Default  17.0000  17.0000
139           hoon   8.000000   8.000000  Default  16.0000  16.0000
138            soo   8.000000   8.000000  Default  15.0000  15.0000
46          kunyit  15.000000  15.000000  Default  14.0000  14.0000
156          sayur  11.000000  11.000000  Default  13.0000  13.0000
68          keping  26.000000  26.000000  Default  12.0000  12.0000
13          goreng  35.000000  35.000000  Default  11.0000  11.0000
72           merah  36.000000  36.000000  Default  10.0000  10.0000
134         terung  10.000000  10.000000  Default   9.0000   9.0000
45          serbuk   8.000000   8.000000  Default   8.0000   8.0000
142          serai  11.000000  11.000000  Default   7.0000   7.0000
69            cili  36.000000  36.000000  Default   6.0000   6.0000
42           putih  13.000000  13.000000  Default   5.0000   5.0000
60          sambal  14.000000  14.000000  Default   4.0000   4.0000
149           inci  12.000000  12.000000  Default   3.0000   3.0000
133          karot   9.000000   9.000000  Default   2.0000   2.0000
143          ketuk  10.000000  10.000000  Default   1.0000   1.0000
166           kuah   9.275083   9.959489   Topic1  -4.2447   0.8314
136         rendam  19.661497  21.211749   Topic1  -3.4934   0.8267
137           ikat   6.682819   7.222620   Topic1  -4.5725   0.8249
130      sengkuang   6.681123   7.222252   Topic1  -4.5727   0.8247
148          kotak   5.749108   6.285786   Topic1  -4.7230   0.8134
129          kubis   7.276487   8.042498   Topic1  -4.4874   0.8025
173          impit   4.803465   5.346866   Topic1  -4.9027   0.7954
156          sayur  10.214095  11.517409   Topic1  -4.1483   0.7825
135          fucuk   6.270940   7.082051   Topic1  -4.6361   0.7810
126         kacang  19.825752  22.411033   Topic1  -3.4850   0.7800
141          tauhu  13.982218  15.999944   Topic1  -3.8342   0.7678
131          letak   8.568182   9.843717   Topic1  -4.3240   0.7638
70          kering  14.701560  17.219250   Topic1  -3.7841   0.7445
252      kepekatan   2.945747   3.475455   Topic1  -5.3917   0.7372
145          hidup  15.133024  17.869191   Topic1  -3.7552   0.7364
132         batang  25.065669  30.053217   Topic1  -3.2505   0.7211
144          udang  16.374183  20.141708   Topic1  -3.6763   0.6955
162           basi   2.008186   2.537883   Topic1  -5.7748   0.6685
151     sepenuhnya   2.008160   2.537877   Topic1  -5.7748   0.6685
152      tambahkan   2.008151   2.537885   Topic1  -5.7748   0.6685
163

Topic: 1
merah, tangkai, cili, hoon, soo, santan, bawang, batang, ketuk, inci, biji, bilis, ikan, ulas, lengkuas, keping, karot, terung, serai, tempe, kacang, kisar, kentang, jintan, talian, hayat, halia, petai, putih, selamat

Topic: 2
biji, batang, kacang, keping, rendam, merah, goreng, udang, hidup, kering, kisar, santan, tauhu, cili, garam, kunyit, sambal, sayur, kuah, serai, tempe, letak, terung, hiris, kubis, bawang, ikat, sengkuang, karot, inci

Topic: 3
ikan, bilis, goreng, bawang, cili, tempe, garam, garing, biji, kisar, gula, merah, ulas, kentang, putih, rangup, mencuba, hiris, air, keping, nipis, tos, tumbuk, gaul, nota, digoreng, dimasukkan, asam, tumis, tutup

Topic: 4
biji, bawang, cili, garam, sos, timun, ulas, kunyit, gula, tangkai, hidup, putih, udang, kulit, merah, hiris, santan, kisar, inci, mendidih, kena, cuka, tomato, aji, kicap, tiram, hijau, helai, daun, jari

Topic: 5
bawang, sayuran, opsyenal, kisar, gula, putih, suuhun, layu, pes, jawa, biji, udang, sambal, h

[(0,
  '0.026*"merah" + 0.023*"tangkai" + 0.022*"cili" + 0.022*"hoon" + 0.022*"soo" + 0.020*"santan" + 0.020*"bawang" + 0.020*"batang" + 0.020*"ketuk" + 0.020*"inci"'),
 (1,
  '0.044*"biji" + 0.039*"batang" + 0.031*"kacang" + 0.030*"keping" + 0.030*"rendam" + 0.027*"merah" + 0.026*"goreng" + 0.025*"udang" + 0.023*"hidup" + 0.023*"kering"'),
 (2,
  '0.054*"ikan" + 0.054*"bilis" + 0.039*"goreng" + 0.035*"bawang" + 0.035*"cili" + 0.034*"tempe" + 0.026*"garam" + 0.024*"garing" + 0.023*"biji" + 0.021*"kisar"'),
 (3,
  '0.032*"biji" + 0.027*"bawang" + 0.026*"cili" + 0.023*"garam" + 0.020*"sos" + 0.020*"timun" + 0.019*"ulas" + 0.018*"kunyit" + 0.015*"gula" + 0.015*"tangkai"'),
 (4,
  '0.044*"bawang" + 0.021*"sayuran" + 0.020*"opsyenal" + 0.020*"kisar" + 0.018*"gula" + 0.014*"putih" + 0.014*"suuhun" + 0.014*"layu" + 0.014*"pes" + 0.014*"jawa"'),
 (5,
  '0.003*"bawang" + 0.003*"sos" + 0.003*"hiris" + 0.003*"biji" + 0.003*"ulas" + 0.003*"garam" + 0.003*"cili" + 0.003*"kisar" + 0.003*"kicap" + 0.

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
tempeProb = make_dictionary_terms_prob(ldamodel, id_map, d)
tempeProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
tempeWN = dfWordnet.copy()

tempeWN["Synonym_clean"] = cleanWN_Synonym(tempeWN["Synonym"])
tempeWN["Hypernim_clean"] = cleanWN(tempeWN["Hypernim"])
tempeWN["Hyponim_clean"] = cleanWN(tempeWN["Hyponim"])
tempeWN

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,merah,Synset('red.s.01'),[],[],of a color at the end of the color spectrum (n...,red,None,None
1,1,tangkai,Synset('stalk.n.04'),[Synset('pursuit.n.01')],[],the act of following prey stealthily,stalk,[pursuit],None
2,1,cili,Synset('pepper.n.04'),[Synset('solanaceous_vegetable.n.01')],"[Synset('hot_pepper.n.02'), Synset('sweet_pepp...",sweet and hot varieties of fruits of plants of...,pepper,[solanaceous_vegetable],"[hot_pepper, sweet_pepper]"
3,1,hoon,None,None,None,None,None,None,None
4,1,soo,None,None,None,None,None,None,None
5,1,santan,Synset('coconut_milk.n.02'),[Synset('milk.n.04')],[],clear to whitish fluid from within a fresh coc...,coconut_milk,[milk],None
6,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
7,1,batang,Synset('bar.n.03'),[Synset('implement.n.01')],"[Synset('belaying_pin.n.01'), Synset('bolt.n.0...",a rigid piece of metal or wood; usually used a...,bar,[implement],"[belaying_pin, bolt, bolt, carpenter's_level, ..."
8,1,ketuk,Synset('tap.n.08'),[Synset('touch.n.05')],[],a light touch or stroke,tap,[touch],None
9,1,inci,Synset('inch.n.01'),[Synset('linear_unit.n.01')],[],a unit of length equal to one twelfth of a foot,inch,[linear_unit],None


### Make Ontology

In [ ]:
# Make Ontology
azieTempeOnto = get_ontology("http://test.org/azieTempe.owl")
makeOntology(azieTempeOnto, tempeWN)
azieTempeOnto.save(file = "azieTempe.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1HsXWNDNuv483p7U-DjlR6cjD_4DUFunD"

In [ ]:
wiki_tempe = pd.read_excel("AzieTempeFinal.xlsx").iloc[:, 1:]

# Translate
wiki_tempe["Category"] = translate(wiki_tempe["Category"])
wiki_tempe.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_tempe[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
tempeTabel_new = tabel_1981.copy()
tempeTabel_new["correct_category"] = all_cat
tempeTabel_new["suggestion"] = all_suggestion

# Export Result to File
tempeTabel_new.to_excel("tempeTabel_Result.xlsx")
tempeTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tempeTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
tempeTabel_new = concatListInColumn(tempeTabel_new)
tempeTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, tempeTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_tempe.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
tempeWiki_new = wiki_df.copy()
tempeWiki_new["suggest"] = all_cat
tempeWiki_new


# Export Result to file
tempeWiki_new.to_excel("tempeWiki_Result.xlsx")
tempeWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tempeWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieTempeOntoWiki = get_ontology("http://test.org/azieTempe_onto_wiki.owl")
makeOntology_wiki(azieTempeOntoWiki, tempeWiki_new)
azieTempeOntoWiki.save(file = "azieTempe_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = tempeWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
tempeWN_new = listToDataframe(changesList, tableWordnet)
tempeWN_new.to_excel("tempeWN_Result.xlsx")
tempeWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tempeWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Telur

In [ ]:
!gdown --id "1hWA53v9OWJtCqT-QOEtszdTvtVtsNv0F"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hWA53v9OWJtCqT-QOEtszdTvtVtsNv0F
To: /content/aziekitchen-telur.jl
100% 3.07M/3.07M [00:00<00:00, 232MB/s]


In [ ]:
telur=pd.read_json(r'/content/aziekitchen-telur.jl', lines = True)
telur.to_pickle(r'/content/raw_df.pkl')

telur = pre_cleaning_text(telur, threshold=20)

# telur["cleaned_description"] = cleaning_text(telur["page_description"])
# ingredients_all = telur["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Telur

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = telur.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.225612  0.034817       1        1  29.325068
5      0.181723  0.050457       2        1  20.931190
6     -0.172845  0.119034       3        1  14.447273
4     -0.170914  0.094648       4        1  12.882242
3     -0.059055  0.126442       5        1  12.547131
1      0.134837 -0.117029       6        1   6.695483
0     -0.139358 -0.308368       7        1   3.171613, topic_info=               Term        Freq       Total Category  logprob  loglift
62           bawang  601.000000  601.000000  Default  30.0000  30.0000
1              cili  410.000000  410.000000  Default  29.0000  29.0000
226          goreng  272.000000  272.000000  Default  28.0000  28.0000
261             mee   97.000000   97.000000  Default  27.0000  27.0000
54             nasi  218.000000  218.000000  Default  26.0000  26.0000
152          tepung  255.000000  255.000000  Default  25.0000  25.0000
314             sos   89.000000   89.000000  Default  24.0000  24.0000
59             gula  402.000000  402.000000  Default  23.0000  23.0000
16           santan  149.000000  149.000000  Default  22.0000  22.0000
492             kek   88.000000   88.000000  Default  21.0000  21.0000
18             daun  236.000000  236.000000  Default  20.0000  20.0000
265           kicap   63.000000   63.000000  Default  19.0000  19.0000
210            susu  108.000000  108.000000  Default  18.0000  18.0000
158          pandan   80.000000   80.000000  Default  17.0000  17.0000
40              air  315.000000  315.000000  Default  16.0000  16.0000
37             ikan  200.000000  200.000000  Default  15.0000  15.0000
13             asam   88.000000   88.000000  Default  14.0000  14.0000
19            hiris  190.000000  190.000000  Default  13.0000  13.0000
63            merah  290.000000  290.000000  Default  12.0000  12.0000
14            garam  433.000000  433.000000  Default  11.0000  11.0000
61             ulas  211.000000  211.000000  Default  10.0000  10.0000
43            kisar  233.000000  233.000000  Default   9.0000   9.0000
399           bilis   99.000000   99.000000  Default   8.0000   8.0000
313          kuning   87.000000   87.000000  Default   7.0000   7.0000
168          adunan  117.000000  117.000000  Default   6.0000   6.0000
299          tomato   45.000000   45.000000  Default   5.0000   5.0000
234           udang   80.000000   80.000000  Default   4.0000   4.0000
188            esen   77.000000   77.000000  Default   3.0000   3.0000
189         vanilla   77.000000   77.000000  Default   2.0000   2.0000
494          baking   51.000000   51.000000  Default   1.0000   1.0000
54             nasi  218.213229  218.908771   Topic1  -3.4515   1.2235
399           bilis   98.404874   99.098864   Topic1  -4.2479   1.2197
243          garing   46.627202   47.326492   Topic1  -4.9948   1.2118
718           kubis   29.611049   30.305254   Topic1  -5.4488   1.2036
296           karot   67.824298   69.475499   Topic1  -4.6200   1.2027
363           bendi   20.002667   20.696918   Topic1  -5.8411   1.1926
426           sulah   15.346558   16.040964   Topic1  -6.1061   1.1825
865        perencah   14.274353   14.969333   Topic1  -6.1785   1.1792
553           halba   12.452624   13.150863   Topic1  -6.3150   1.1722
795           blend   13.067676   13.801603   Topic1  -6.2668   1.1721
548           petai   11.851527   12.545875   Topic1  -6.3645   1.1698
522        hidangan   11.001189   11.702197   Topic1  -6.4390   1.1650
774      kekuningan   10.937702   11.644506   Topic1  -6.4447   1.1641
697       tambahkan   10.483114   11.193999   Topic1  -6.4872   1.1611
1694           stok    9.909755   10.613164   Topic1  -6.5434   1.1582
866         seriaji    9.596772   10.290216   Topic1  -6.5755   1.1570
694        ditumbuk    9.484264   10.178674   Topic1  -6.5873   1.1561
867             pot    8.822430    9.516570   Topic1  -6.6597  

Topic: 1
salmon, cheese, cream, camca, lemon, pie, buttermilk, teh, flour, oil, burger, garam, jus, cake, spatula, begedil, hadiah, keju, link, gantikan, shell, vegetable, red, potongan, berdurasi, parutan, gliserin, jenama, quiche, spaghetti

Topic: 2
mee, sos, cili, kicap, daun, tomato, bawang, tiram, udang, biji, limau, nipis, kuning, goreng, air, telur, daging, fish, kisar, tabur, ikat, manis, sup, makan, bok, pek, gram, kering, sayur, kue

Topic: 3
bawang, telur, goreng, garam, cili, putih, nasi, merah, biji, ikan, ulas, hiris, tumbuk, bilis, mangkuk, kisar, tumis, panaskan, tangkai, serbuk, layan, daun, ayam, karot, kering, selamat, mencuba, kacang, batang, dikacau

Topic: 4
air, gula, santan, pandan, daun, garam, tepung, telur, biji, kuih, pulut, adunan, gandum, helai, loyang, mendidih, beras, pekat, pengukus, acuan, lapisan, nota, periuk, kelapa, mencuba, kukus, layan, gram, sejuk, kisar

Topic: 5
telur, tepung, susu, gula, biji, gram, butter, kuning, buah, air, roti, putar, no

[(0,
  '0.017*"salmon" + 0.013*"cheese" + 0.013*"cream" + 0.010*"camca" + 0.009*"lemon" + 0.009*"pie" + 0.008*"buttermilk" + 0.008*"teh" + 0.008*"flour" + 0.008*"oil"'),
 (1,
  '0.062*"mee" + 0.050*"sos" + 0.036*"cili" + 0.033*"kicap" + 0.023*"daun" + 0.022*"tomato" + 0.020*"bawang" + 0.016*"tiram" + 0.016*"udang" + 0.015*"biji"'),
 (2,
  '0.061*"bawang" + 0.039*"telur" + 0.036*"goreng" + 0.035*"garam" + 0.033*"cili" + 0.032*"putih" + 0.032*"nasi" + 0.030*"merah" + 0.025*"biji" + 0.023*"ikan"'),
 (3,
  '0.033*"air" + 0.031*"gula" + 0.030*"santan" + 0.025*"pandan" + 0.021*"daun" + 0.020*"garam" + 0.020*"tepung" + 0.019*"telur" + 0.016*"biji" + 0.014*"kuih"'),
 (4,
  '0.034*"telur" + 0.027*"tepung" + 0.022*"susu" + 0.021*"gula" + 0.019*"biji" + 0.018*"gram" + 0.016*"butter" + 0.015*"kuning" + 0.015*"buah" + 0.013*"air"'),
 (5,
  '0.031*"bawang" + 0.026*"cili" + 0.024*"biji" + 0.023*"kisar" + 0.023*"air" + 0.022*"garam" + 0.018*"asam" + 0.017*"gula" + 0.015*"inci" + 0.014*"keping"'),
 (6,

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
telurProb = make_dictionary_terms_prob(ldamodel, id_map, d)
telurProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
telurWN = dfWordnet.copy()

telurWN["Synonym_clean"] = cleanWN_Synonym(telurWN["Synonym"])
telurWN["Hypernim_clean"] = cleanWN(telurWN["Hypernim"])
telurWN["Hyponim_clean"] = cleanWN(telurWN["Hyponim"])
telurWN

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,salmon,Synset('salmon.n.01'),"[Synset('food_fish.n.01'), Synset('salmonid.n....","[Synset('atlantic_salmon.n.02'), Synset('black...",any of various large food and game fishes of n...,salmon,"[food_fish, salmonid]","[atlantic_salmon, blackfish, chinook, chum_sal..."
1,1,cheese,None,None,None,None,None,None,None
2,1,cream,None,None,None,None,None,None,None
3,1,camca,Synset('spoon.n.01'),"[Synset('container.n.01'), Synset('cutlery.n.0...","[Synset('dessert_spoon.n.01'), Synset('runcibl...",a piece of cutlery with a shallow bowl-shaped ...,spoon,"[container, cutlery]","[dessert_spoon, runcible_spoon, soupspoon, sug..."
4,1,lemon,Synset('lemony.s.01'),[],[],tasting sour like a lemon,lemony,None,None
5,1,pie,None,None,None,None,None,None,None
6,1,buttermilk,None,None,None,None,None,None,None
7,1,teh,Synset('tea.n.02'),[Synset('meal.n.01')],[],a light midafternoon meal of tea and sandwiche...,tea,[meal],None
8,1,flour,None,None,None,None,None,None,None
9,1,oil,None,None,None,None,None,None,None


### Make Ontology

In [ ]:
# Make Ontology
azieTelurOnto = get_ontology("http://test.org/azieTelur.owl")
makeOntology(azieTelurOnto, telurWN)
azieTelurOnto.save(file = "azieTelur.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1g-pUsOrn9SVSd9yvoTgNN5JrlVEIU0pA"

In [ ]:
wiki_telur = pd.read_excel("AzieTelurFinal.xlsx").iloc[:, 1:]

# Translate
wiki_telur["Category"] = translate(wiki_telur["Category"])
wiki_telur.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_telur[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
telurTabel_new = tabel_1981.copy()
telurTabel_new["correct_category"] = all_cat
telurTabel_new["suggestion"] = all_suggestion

# Export Result to File
telurTabel_new.to_excel("telurTabel_Result.xlsx")
telurTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "telurTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
telurTabel_new = concatListInColumn(telurTabel_new)
telurTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, telurTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_telur.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
telurWiki_new = wiki_df.copy()
telurWiki_new["suggest"] = all_cat

# Export Result to file
telurWiki_new.to_excel("telurWiki_Result.xlsx")
telurWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "telurWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieTelurgOntoWiki = get_ontology("http://test.org/azieTelurg_onto_wiki.owl")
makeOntology_wiki(azieTelurgOntoWiki, telurgWiki_new)
azieTelurgOntoWiki.save(file = "azieTelurg_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = telurWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
telurWN_new = listToDataframe(changesList, tableWordnet)
telurWN_new.to_excel("telurWN_Result.xlsx")
telurWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "telurWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Tahu

In [ ]:
!gdown --id "1hdmUEtcYw_oPbDjARWqq3_muGGgIOvSP"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hdmUEtcYw_oPbDjARWqq3_muGGgIOvSP
To: /content/aziekitchen-tahu.jl
100% 5.99M/5.99M [00:00<00:00, 19.1MB/s]


In [ ]:
tahu=pd.read_json(r'/content/aziekitchen-tahu.jl', lines = True)

tahu.to_pickle(r'/content/raw_df.pkl')
tahu = pre_cleaning_text(tahu, threshold=20)

#tahu["cleaned_description"] = cleaning_text(tahu["page_description"])
#ingredients_all = tahu["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Tahu

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = tahu.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5      0.221312  0.033289       1        1  39.595308
6      0.196397 -0.050944       2        1  18.861479
4     -0.003690  0.303936       3        1  16.978463
0      0.233395 -0.129605       4        1  13.182474
2     -0.224273 -0.072141       5        1   5.447509
3     -0.209072 -0.053514       6        1   3.070367
1     -0.214069 -0.031022       7        1   2.864400, topic_info=                           Term         Freq        Total Category  logprob  \
20                       bawang  1080.000000  1080.000000  Default  30.0000   
67                         cili   694.000000   694.000000  Default  29.0000   
884                      sotong   144.000000   144.000000  Default  28.0000   
171                      tepung   232.000000   232.000000  Default  27.0000   
74                         gula   674.000000   674.000000  Default  26.0000   
163                       layan   227.000000   227.000000  Default  25.0000   
504                         sos   155.000000   155.000000  Default  24.0000   
69                       santan   487.000000   487.000000  Default  23.0000   
76                       keping   328.000000   328.000000  Default  22.0000   
60                         daun   495.000000   495.000000  Default  21.0000   
21                        putih   463.000000   463.000000  Default  20.0000   
29                        tumis   204.000000   204.000000  Default  19.0000   
482                      goreng   225.000000   225.000000  Default  18.0000   
235                       telur   309.000000   309.000000  Default  17.0000   
19                         ulas   392.000000   392.000000  Default  16.0000   
688                      tomato   150.000000   150.000000  Default  15.0000   
22                        hiris   313.000000   313.000000  Default  14.0000   
84                        merah   430.000000   430.000000  Default  13.0000   
89                        pulut   240.000000   240.000000  Default  12.0000   
23                       serbuk   292.000000   292.000000  Default  11.0000   
27                        garam   899.000000   899.000000  Default  10.0000   
534                      perasa   125.000000   125.000000  Default   9.0000   
501                       udang   160.000000   160.000000  Default   8.0000   
280                      daging   141.000000   141.000000  Default   7.0000   
1226                        mee    95.000000    95.000000  Default   6.0000   
61                       kunyit   312.000000   312.000000  Default   5.0000   
172                       beras   214.000000   214.000000  Default   4.0000   
4104                   catilisa    50.000000    50.000000  Default   3.0000   
126                        kari   111.000000   111.000000  Default   2.0000   
169                        jawa   109.000000   109.000000  Default   1.0000   
76                       keping   327.799591   328.541061   Topic1  -3.8250   
89                        pulut   240.043601   240.785697   Topic1  -4.1365   
172                       beras   213.686650   214.429406   Topic1  -4.2528   
164               penyediaannya    93.250799    93.992221   Topic1  -5.0821   
222                     youtube    83.071533    83.812523   Topic1  -5.1977   
83                     lengkuas    62.010977    62.751783   Topic1  -5.4900   
1150                    dikisar    56.382213    57.130292   Topic1  -5.5852   
217                         jer    51.443797    52.195168   Topic1  -5.6769   
479                     dikacau    49.960674    50.704498   Topic1  -5.7061   
1260                     kepala    47.273433    48.015584   Topic1  -5.7614   
600                    direndam    44.536574    45.278943   Topic1  -5.8210   
55                      rendang    44.301328    45.042389   Topic1  -5.8263   
1216                      masin    38.592743    39.334040   Topic1  -5.964

Topic: 1
bawang, cili, sos, garam, putih, daun, goreng, serbuk, tomato, tumis, perasa, mee, merah, telur, udang, ulas, ayam, kisar, hidup, maggi, buah, padi, kicap, hiris, halia, disukai, air, limau, dihiris, lada

Topic: 2
catilisa, pisang, layan, sebiji, cerita, httpwwwcatilisacatterycom, httpcatilisablogspotcom, cattery, teow, kue, kucai, jala, dibelah, dah, versi, kegelapan, jantung, bancuhan, keledek, laksa, percik, susun, balikkan, dibalik, penerangnya, kaup, nak, gelas, laksam, tisu

Topic: 3
gambar, letakkan, freezer, entri, doh, pastri, pastikan, pasta, kebakaran, klik, simpan, pepper, disediakan, buah, pulak, cuba, soket, karipap, hidangan, parsley, sesuai, membuatnya, paste, unsalted, icing, perkakasan, malaysia, memasak, olive, lampu

Topic: 4
sotong, bilis, aween, makanan, ketupat, mahal, sumbatkan, tarik, dimakan, nasi, selera, menjamu, ikan, resepinya, makan, syarat, memasak, dah, garfu, tembok, zakat, keropok, dibeli, kangkung, hidangan, kongsikan, flakes, dikongsikan, 

[(0,
  '0.086*"bawang" + 0.056*"cili" + 0.031*"sos" + 0.030*"garam" + 0.029*"putih" + 0.028*"daun" + 0.025*"goreng" + 0.024*"serbuk" + 0.022*"tomato" + 0.021*"tumis"'),
 (1,
  '0.045*"catilisa" + 0.026*"pisang" + 0.020*"layan" + 0.017*"sebiji" + 0.017*"cerita" + 0.015*"httpcatilisablogspotcom" + 0.015*"httpwwwcatilisacatterycom" + 0.015*"cattery" + 0.008*"teow" + 0.007*"kue"'),
 (2,
  '0.018*"gambar" + 0.013*"letakkan" + 0.008*"freezer" + 0.007*"entri" + 0.006*"doh" + 0.006*"pastri" + 0.006*"pastikan" + 0.005*"pasta" + 0.005*"kebakaran" + 0.005*"klik"'),
 (3,
  '0.123*"sotong" + 0.023*"bilis" + 0.017*"aween" + 0.017*"makanan" + 0.016*"ketupat" + 0.015*"mahal" + 0.014*"sumbatkan" + 0.014*"tarik" + 0.013*"dimakan" + 0.012*"nasi"'),
 (4,
  '0.036*"tepung" + 0.034*"gula" + 0.026*"air" + 0.022*"telur" + 0.014*"biji" + 0.013*"susu" + 0.012*"gandum" + 0.011*"adunan" + 0.011*"pandan" + 0.011*"butter"'),
 (5,
  '0.039*"garam" + 0.031*"santan" + 0.027*"gula" + 0.022*"keping" + 0.021*"cili" + 0.0

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
tahuProb = make_dictionary_terms_prob(ldamodel, id_map, d)
tahuProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
tahuWN = dfWordnet.copy()

tahuWN["Synonym_clean"] = cleanWN_Synonym(tahuWN["Synonym"])
tahuWN["Hypernim_clean"] = cleanWN(tahuWN["Hypernim"])
tahuWN["Hyponim_clean"] = cleanWN(tahuWN["Hyponim"])
tahuWN

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
1,1,cili,Synset('pepper.n.04'),[Synset('solanaceous_vegetable.n.01')],"[Synset('hot_pepper.n.02'), Synset('sweet_pepp...",sweet and hot varieties of fruits of plants of...,pepper,[solanaceous_vegetable],"[hot_pepper, sweet_pepper]"
2,1,sos,Synset('sauce.n.01'),[Synset('condiment.n.01')],"[Synset('aioli.n.01'), Synset('allemande.n.01'...",flavorful relish or dressing or topping served...,sauce,[condiment],"[aioli, allemande, anchovy_sauce, apricot_sauc..."
3,1,garam,Synset('strategic_arms_limitation_talks.n.01'),[],[],negotiations between the United States and the...,strategic_arms_limitation_talks,None,None
4,1,putih,Synset('canescent.s.02'),[],[],covered with fine whitish hairs or down,canescent,None,None
5,1,daun,Synset('blade.n.08'),[Synset('rotating_mechanism.n.01')],"[Synset('fan_blade.n.01'), Synset('impeller.n....",flat surface that rotates and pushes against a...,blade,[rotating_mechanism],"[fan_blade, impeller, paddle, rudder_blade]"
6,1,goreng,Synset('fried.s.01'),[],[],cooked by frying in fat,fried,None,None
7,1,serbuk,Synset('crumb.n.03'),[Synset('morsel.n.02')],"[Synset('breadcrumb.n.01'), Synset('cracker_cr...",small piece of e.g. bread or cake,crumb,[morsel],"[breadcrumb, cracker_crumbs]"
8,1,tomato,Synset('tomato.n.01'),[Synset('solanaceous_vegetable.n.01')],"[Synset('beefsteak_tomato.n.01'), Synset('cher...",mildly acid red or yellow pulpy fruit eaten as...,tomato,[solanaceous_vegetable],"[beefsteak_tomato, cherry_tomato]"
9,1,tumis,None,None,None,None,None,None,None


### Make Ontology

In [ ]:
# Make Ontology
azieTahuOnto = get_ontology("http://test.org/azieTahu.owl")
makeOntology(azieTahuOnto, tahuWN)
azieTahuOnto.save(file = "azieTahu.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1gwlbGvmECGXWAAcKeTOCbrZcR3XGSjoA"

In [ ]:
wiki_tahu = pd.read_excel("AzieTahuFinal.xlsx").iloc[:, 1:]

# Translate
wiki_tahu["Category"] = translate(wiki_tahu["Category"])
wiki_tahu.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_tahu[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
tahuTabel_new = tabel_1981.copy()
tahuTabel_new["correct_category"] = all_cat
tahuTabel_new["suggestion"] = all_suggestion

# Export Result to File
tahuTabel_new.to_excel("tahuTabel_Result.xlsx")
tahuTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tahuTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
tahuTabel_new = concatListInColumn(tahuTabel_new)
tahuTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, tahuTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_tahu.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
tahuWiki_new = wiki_df.copy()
tahuWiki_new["suggest"] = all_cat

# Export Result to file
tahuWiki_new.to_excel("tahuWiki_Result.xlsx")
tahuWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tahuWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieTahuOntoWiki = get_ontology("http://test.org/azieTahu_onto_wiki.owl")
makeOntology_wiki(azieTahuOntoWiki, tahuWiki_new)
azieTahuOntoWiki.save(file = "azieTahu_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = tahuWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
tahuWN_new = listToDataframe(changesList, tableWordnet)
tahuWN_new.to_excel("tahuWN_Result.xlsx")
tahuWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tahuWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Sapi

In [ ]:
!gdown --id "1hQOC2z9SX-wILlMSPn2ottwBLjZTqOEU"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1hQOC2z9SX-wILlMSPn2ottwBLjZTqOEU
To: /content/aziekitchen-sapi.jl
100% 264k/264k [00:00<00:00, 85.3MB/s]


In [ ]:
sapi=pd.read_json(r'/content/aziekitchen-sapi.jl', lines = True)

sapi.to_pickle(r'/content/raw_df.pkl')
sapi = pre_cleaning_text(sapi, threshold=20)

# sapi["cleaned_description"] = cleaning_text(sapi["page_description"])
# ingredients_all = sapi["cleaned_description"].apply(lambda x: treat_ingredients(x))

## GUI Sapi

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = sapi.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4     -0.067647  0.037148       1        1  46.623386
1     -0.131794 -0.004537       2        1  18.574887
0     -0.078108  0.037132       3        1  11.525455
3      0.152823  0.032354       4        1   7.683538
5      0.016459 -0.154635       5        1   5.764095
6      0.007403  0.030849       6        1   5.049435
2      0.100862  0.021690       7        1   4.779204, topic_info=              Term       Freq      Total Category  logprob  loglift
21          bawang  75.000000  75.000000  Default  30.0000  30.0000
54            ayam  43.000000  43.000000  Default  29.0000  29.0000
5             nasi  49.000000  49.000000  Default  28.0000  28.0000
26           kisar  24.000000  24.000000  Default  27.0000  27.0000
58          batang  15.000000  15.000000  Default  26.0000  26.0000
30           garam  35.000000  35.000000  Default  25.0000  25.0000
20            biji  53.000000  53.000000  Default  24.0000  24.0000
83          goreng  21.000000  21.000000  Default  23.0000  23.0000
190         tomato  28.000000  28.000000  Default  22.0000  22.0000
151            sup   9.000000   9.000000  Default  21.0000  21.0000
65            cili  26.000000  26.000000  Default  20.0000  20.0000
73           putih  34.000000  34.000000  Default  19.0000  19.0000
22           merah  34.000000  34.000000  Default  18.0000  18.0000
23           hiris  29.000000  29.000000  Default  17.0000  17.0000
79            daun  37.000000  37.000000  Default  16.0000  16.0000
158         tepung   8.000000   8.000000  Default  15.0000  15.0000
33           telur  13.000000  13.000000  Default  14.0000  14.0000
183         rempah  31.000000  31.000000  Default  13.0000  13.0000
47          kering  23.000000  23.000000  Default  12.0000  12.0000
82       ketepikan  10.000000  10.000000  Default  11.0000  11.0000
181         daging  23.000000  23.000000  Default  10.0000  10.0000
102          tumis  20.000000  20.000000  Default   9.0000   9.0000
3          selamat  11.000000  11.000000  Default   8.0000   8.0000
157           gram   8.000000   8.000000  Default   7.0000   7.0000
68            gula  22.000000  22.000000  Default   6.0000   6.0000
152         serbuk  10.000000  10.000000  Default   5.0000   5.0000
11            sapi  35.000000  35.000000  Default   4.0000   4.0000
77          kacang   6.000000   6.000000  Default   3.0000   3.0000
160         santan   7.000000   7.000000  Default   2.0000   2.0000
188         jintan   9.000000   9.000000  Default   1.0000   1.0000
309         puding   8.515218   9.114457   Topic1  -4.9342   0.6951
312         pisang   6.652547   7.251548   Topic1  -5.1811   0.6769
86            kuah   6.652211   7.251429   Topic1  -5.1811   0.6768
80        ketumbar  13.170652  14.549095   Topic1  -4.4981   0.6635
28             jus   5.720696   6.319950   Topic1  -5.3320   0.6634
52           gajus   3.858301   4.457105   Topic1  -5.7258   0.6188
338         mayang   3.858082   4.457058   Topic1  -5.7259   0.6187
12         majerin   3.858048   4.457021   Topic1  -5.7259   0.6187
29        sunquick   3.858052   4.457029   Topic1  -5.7259   0.6187
13          planta   3.858052   4.457041   Topic1  -5.7259   0.6187
284          basah   3.857942   4.456948   Topic1  -5.7259   0.6187
182         pinang   2.927026   3.525631   Topic1  -6.0021   0.5770
487        kastard   2.927016   3.525656   Topic1  -6.0021   0.5770
61           serai   2.927001   3.525649   Topic1  -6.0021   0.5770
323        aturkan   2.926941   3.525641   Topic1  -6.0021   0.5770
49         gaulkan   2.926879   3.525615   Topic1  -6.0021   0.5769
75           bulat   2.926842   3.525597   Topic1  -6.0021   0.5769
330       rosemary   2.926862   3.525627   Topic1  -6.0021   0.5769
27           ambil   2.926832   3.525604   Topic1  -6.0021   0.5769
87      disediakan   2.926855   3.525661   Topic1  -6.0021   0.5769
188

Topic: 1
ayam, bawang, nasi, goreng, rempah, tomato, biji, merah, putih, serbuk, kiub, susu, cair, blend, maggi, cili, buah, air, beras, daun, garam, sapi, halia, dimakan, kuzi, dsb, kari, kunyit, campur, hati

Topic: 2
nasi, hiris, air, bawang, biji, beras, tomato, daging, cair, rempah, susu, sapi, bunga, daun, tumis, pandan, buah, halia, merah, ayam, putih, pewarna, inci, basmathi, kuning, pelaga, garam, lawang, kayu, ulas

Topic: 3
bawang, daun, merah, ayam, garam, ketepikan, selamat, tepung, talam, berlauk, putih, cili, sapi, mencuba, gula, digoreng, gram, sulah, serbuk, lada, sup, lauk, cina, kering, biji, tumis, air, panaskan, beras, ulas

Topic: 4
bawang, biji, santan, gram, tepung, daun, merah, gula, sapi, garam, labu, adunan, cwn, telur, air, udang, tuang, gandum, bakar, tart, ayam, ulas, cili, putih, kering, serbuk, panaskan, sup, butter, lada

Topic: 5
bawang, biji, daun, nasi, garam, kisar, ayam, putih, sapi, kering, merah, manis, buah, susu, gula, rempah, air, cair, cili, 

[(0,
  '0.047*"ayam" + 0.038*"bawang" + 0.031*"nasi" + 0.024*"goreng" + 0.020*"rempah" + 0.017*"tomato" + 0.017*"biji" + 0.014*"merah" + 0.014*"putih" + 0.014*"serbuk"'),
 (1,
  '0.037*"nasi" + 0.026*"hiris" + 0.023*"air" + 0.021*"bawang" + 0.020*"biji" + 0.020*"beras" + 0.020*"tomato" + 0.018*"daging" + 0.018*"cair" + 0.016*"rempah"'),
 (2,
  '0.035*"bawang" + 0.020*"daun" + 0.020*"merah" + 0.015*"ayam" + 0.015*"garam" + 0.015*"ketepikan" + 0.015*"selamat" + 0.015*"tepung" + 0.015*"talam" + 0.015*"berlauk"'),
 (3,
  '0.022*"bawang" + 0.019*"biji" + 0.019*"santan" + 0.019*"gram" + 0.019*"tepung" + 0.015*"daun" + 0.015*"merah" + 0.015*"gula" + 0.015*"sapi" + 0.015*"garam"'),
 (4,
  '0.031*"bawang" + 0.024*"biji" + 0.019*"daun" + 0.018*"nasi" + 0.017*"garam" + 0.017*"kisar" + 0.016*"ayam" + 0.016*"putih" + 0.015*"sapi" + 0.015*"kering"'),
 (5,
  '0.036*"sup" + 0.027*"bawang" + 0.027*"batang" + 0.023*"biji" + 0.018*"hiris" + 0.014*"rempah" + 0.014*"ulas" + 0.014*"merah" + 0.014*"dadu" + 0

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sapiProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sapiProb["bawang"] #probabilitas kata bawang pada setiap topik

## Wordnet

In [ ]:
sapiWN = dfWordnet.copy()

sapiWN["Synonym_clean"] = cleanWN_Synonym(sapiWN["Synonym"])
sapiWN["Hypernim_clean"] = cleanWN(sapiWN["Hypernim"])
sapiWN["Hyponim_clean"] = cleanWN(sapiWN["Hyponim"])
sapiWN

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,ayam,Synset('domestic_fowl.n.01'),[Synset('gallinaceous_bird.n.01')],"[Synset('bantam.n.01'), Synset('chicken.n.02')...",a domesticated gallinaceous bird thought to be...,domestic_fowl,[gallinaceous_bird],"[bantam, chicken, cochin, cornish, dorking, ga..."
1,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
2,1,nasi,Synset('rice.n.01'),"[Synset('grain.n.02'), Synset('starches.n.01')]","[Synset('brown_rice.n.01'), Synset('paddy.n.03...",grains used as food either unpolished or more ...,rice,"[grain, starches]","[brown_rice, paddy, white_rice]"
3,1,goreng,Synset('fried.s.01'),[],[],cooked by frying in fat,fried,None,None
4,1,rempah,Synset('spiciness.n.01'),[Synset('taste_property.n.01')],"[Synset('hotness.n.03'), Synset('nip.n.05'), S...",the property of being seasoned with spice and ...,spiciness,[taste_property],"[hotness, nip, pungency]"
5,1,tomato,Synset('tomato.n.01'),[Synset('solanaceous_vegetable.n.01')],"[Synset('beefsteak_tomato.n.01'), Synset('cher...",mildly acid red or yellow pulpy fruit eaten as...,tomato,[solanaceous_vegetable],"[beefsteak_tomato, cherry_tomato]"
6,1,biji,Synset('semen.n.01'),[Synset('liquid_body_substance.n.01')],[Synset('milt.n.02')],the thick white fluid containing spermatozoa t...,semen,[liquid_body_substance],[milt]
7,1,merah,Synset('red.s.01'),[],[],of a color at the end of the color spectrum (n...,red,None,None
8,1,putih,Synset('canescent.s.02'),[],[],covered with fine whitish hairs or down,canescent,None,None
9,1,serbuk,Synset('crumb.n.03'),[Synset('morsel.n.02')],"[Synset('breadcrumb.n.01'), Synset('cracker_cr...",small piece of e.g. bread or cake,crumb,[morsel],"[breadcrumb, cracker_crumbs]"


### Make Ontology

In [ ]:
# Make Ontology
azieSapiOnto = get_ontology("http://test.org/azieSapi.owl")
makeOntology(azieSapiOnto, sapiWN)
azieSapiOnto.save(file = "azieSapi.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "129XYWHaftY_wNGcOIsI6xW6UvQMjKNT7"

In [ ]:
wiki_sapi = pd.read_excel("AzieSapiFinal.xlsx").iloc[:, 1:]

# Translate
wiki_sapi["Category"] = translate(wiki_sapi["Category"])
wiki_sapi.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_sapi[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
sapiTabel_new = tabel_1981.copy()
sapiTabel_new["correct_category"] = all_cat
sapiTabel_new["suggestion"] = all_suggestion

# Export Result to File
sapiTabel_new.to_excel("sapiTabel_Result.xlsx")
sapiTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
sapiTabel_new = concatListInColumn(sapiTabel_new)
sapiTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, sapiTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_sapi.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
sapiWiki_new = wiki_df.copy()
sapiWiki_new["suggest"] = all_cat

# Export Result to file
sapiWiki_new.to_excel("sapiWiki_Result.xlsx")
sapiWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
azieSapiOntoWiki = get_ontology("http://test.org/azieSapi_onto_wiki.owl")
makeOntology_wiki(azieSapiOntoWiki, sapiWiki_new)
azieSapiOntoWiki.save(file = "azieSapi_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = sapiWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
sapiWN_new = listToDataframe(changesList, tableWordnet)
sapiWN_new.to_excel("sapiWN_Result.xlsx")
sapiWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Apakah Performa Meningkat?
Ya, karena setelah stopword dihapus, pada visualisasi data, topik terlihat memiliki makna lebih jelas dan mudah untuk di klasifikasikan.

# Fiza

In [ ]:
!gdown --id "11Yf4HVqR_pLKHYP3jPsB9eL2M7Qtzjdp"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=11Yf4HVqR_pLKHYP3jPsB9eL2M7Qtzjdp
To: /content/Fiza.jl
100% 1.07M/1.07M [00:00<00:00, 145MB/s]


In [ ]:
fiza=pd.read_json(r'/content/Fiza.jl', lines = True)
fiza.rename(columns = {"Page_description":"page_description"}, inplace=True)
fiza["page_description"] = delist(fiza, "page_description")
fiza = pre_cleaning_text(fiza, threshold=30)

In [ ]:
fiza.head()

,URL,Page_Title,Publish_Date,Number_of_Commentary,page_description
0,http://fizalgk.blogspot.com/2017/01/6-kelebiha...,6 KELEBIHAN ACUAN SILIKON BERBANDING LOYANG BIASA,Khamis Januari 19 2017,0,"[-, , Semoga perkongsian dibawah ini memberi m..."
1,http://fizalgk.blogspot.com/2015/06/sayur-camp...,SAYUR CAMPUR BERGAJUS,Isnin Jun 22 2015,0,"[-, , -, Maggi® CukupRasa™ , Maggi® CukupRasa™..."
2,http://fizalgk.blogspot.com/2015/01/aiskrim-pa...,AISKRIM PASU,Khamis Januari 15 2015,0,"[Assalamualaikum, , , -, , -, AISKRIM PASU, Ba..."
3,http://fizalgk.blogspot.com/2013/09/aiskrim-bu...,AISKRIM BUAH CAMPURAN,Rabu September 25 2013,0,"[Assalamualaikum, , , -, , Aiskrim buah campur..."
4,http://fizalgk.blogspot.com/2012/10/pumkin-ais...,PUMPKIN ICE CREAM,Rabu Oktober 03 2012,2,"[-, , Sabar je lah...dalam cuaca panas begini ..."


## GUI Fiza

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = fiza.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.213945 -0.028162       1        1  48.063741
2     -0.148415  0.199010       2        1  16.881908
4      0.104777  0.016820       3        1   9.431461
5     -0.065762  0.053686       4        1   8.480387
1     -0.117095  0.067976       5        1   7.052502
6      0.190214 -0.026280       6        1   6.875081
3     -0.177664 -0.283050       7        1   3.214920, topic_info=                     Term        Freq       Total Category  logprob  loglift
130                   air  714.000000  714.000000  Default  30.0000  30.0000
300                bawang  900.000000  900.000000  Default  29.0000  29.0000
394                 telur  463.000000  463.000000  Default  28.0000  28.0000
111                  cili  529.000000  529.000000  Default  27.0000  27.0000
92                    sos  248.000000  248.000000  Default  26.0000  26.0000
95                 tepung  356.000000  356.000000  Default  25.0000  25.0000
40                 adunan  214.000000  214.000000  Default  24.0000  24.0000
197                  gula  587.000000  587.000000  Default  23.0000  23.0000
514               mentega  192.000000  192.000000  Default  22.0000  22.0000
530                 udang  294.000000  294.000000  Default  21.0000  21.0000
85                 serbuk  325.000000  325.000000  Default  20.0000  20.0000
301                 putih  455.000000  455.000000  Default  19.0000  19.0000
84                  garam  555.000000  555.000000  Default  18.0000  18.0000
66                kedalam  406.000000  406.000000  Default  17.0000  17.0000
395                   teh  331.000000  331.000000  Default  16.0000  16.0000
234              mendidih  202.000000  202.000000  Default  15.0000  15.0000
165                  daun  390.000000  390.000000  Default  14.0000  14.0000
119                 merah  356.000000  356.000000  Default  13.0000  13.0000
337                  ikan  324.000000  324.000000  Default  12.0000  12.0000
476                coklat   85.000000   85.000000  Default  11.0000  11.0000
576                 pulut   91.000000   91.000000  Default  10.0000  10.0000
229                  susu  167.000000  167.000000  Default   9.0000   9.0000
261                 halus  160.000000  160.000000  Default   8.0000   8.0000
299                  ulas  305.000000  305.000000  Default   7.0000   7.0000
6                  loyang  148.000000  148.000000  Default   6.0000   6.0000
252                kacang  145.000000  145.000000  Default   5.0000   5.0000
110               tangkai  257.000000  257.000000  Default   4.0000   4.0000
113                   isi  136.000000  136.000000  Default   3.0000   3.0000
397                gandum  136.000000  136.000000  Default   2.0000   2.0000
618                  kari  182.000000  182.000000  Default   1.0000   1.0000
337                  ikan  323.524229  324.300598   Topic1  -4.0563   0.7302
525                  asam  177.495737  178.272074   Topic1  -4.6566   0.7283
340                sambal  154.531064  155.307240   Topic1  -4.7952   0.7276
593                   sup  151.340217  152.116967   Topic1  -4.8161   0.7275
583                 halia  150.999642  151.836818   Topic1  -4.8183   0.7271
835                 bilis  126.335019  127.111089   Topic1  -4.9967   0.7265
714                  jawa  118.069121  118.845323   Topic1  -5.0643   0.7261
661                  padi  103.489327  104.266654   Topic1  -5.1961   0.7252
777                 hiris   95.512245   96.289084   Topic1  -5.2763   0.7245
545                tumbuk   84.138064   84.923964   Topic1  -5.4031   0.7233
637               kentang   72.066923   72.844210   Topic1  -5.5580   0.7219
1382              holland   66.593103   67.372258   Topic1  -5.6370   0.7210
535               belacan   65.746678   66.522896   Topic1  -5.6498   0.7209
701                 serai   64.069470   64.845344   Topic1  -5.6756   0.7206
560   

Topic: 1
bawang, cili, garam, air, biji, ikan, putih, merah, daun, ulas, gula, tangkai, secukup, goreng, kering, ayam, tumis, serbuk, kisar, bahanbahan, udang, asam, inci, nasi, lada, kari, sambal, sup, halia, santan

Topic: 2
tepung, coklat, gula, garam, doh, teh, serbuk, air, masakan, secubit, pizza, gandum, cairkan, bebola, membuatnya, uli, cincang, ubi, telur, biji, keluarkan, sagu, campurkan, sebiji, ketepikan, greentea, double, badam, betik, didalam

Topic: 3
tepung, adunan, mentega, telur, gula, kedalam, loyang, susu, teh, gandum, oven, bakar, esen, membuatnya, biji, roti, bahanbahan, suhu, keju, vanilla, tuang, didalam, kek, cair, sebati, kastor, dibawah, panaskan, mangkuk, muffin

Topic: 4
coleslaw, mayonis, gula, cuka, biskut, singapore, halus, makan, bawang, digaul, foto, merah, menjaga, lobak, diperap, gaul, nestum, gram, kesegaran, perap, canai, jem, pastikan, toskan, untukmengelakkan, minitperah, benarbenar, udara, kedap, garam

Topic: 5
air, telur, sos, mendidih, pulut, 

[(0,
  '0.044*"bawang" + 0.024*"cili" + 0.023*"garam" + 0.020*"air" + 0.019*"biji" + 0.017*"ikan" + 0.016*"putih" + 0.016*"merah" + 0.016*"daun" + 0.015*"ulas"'),
 (1,
  '0.030*"tepung" + 0.030*"coklat" + 0.023*"gula" + 0.022*"garam" + 0.020*"doh" + 0.020*"teh" + 0.016*"serbuk" + 0.014*"air" + 0.014*"masakan" + 0.013*"secubit"'),
 (2,
  '0.031*"tepung" + 0.031*"adunan" + 0.029*"mentega" + 0.028*"telur" + 0.025*"gula" + 0.022*"kedalam" + 0.019*"loyang" + 0.019*"susu" + 0.016*"teh" + 0.016*"gandum"'),
 (3,
  '0.027*"coleslaw" + 0.018*"mayonis" + 0.017*"gula" + 0.017*"cuka" + 0.016*"biskut" + 0.016*"singapore" + 0.015*"halus" + 0.013*"makan" + 0.012*"bawang" + 0.011*"digaul"'),
 (4,
  '0.047*"air" + 0.045*"telur" + 0.031*"sos" + 0.026*"mendidih" + 0.025*"pulut" + 0.022*"putih" + 0.019*"udang" + 0.017*"kobis" + 0.013*"kuah" + 0.013*"kuning"'),
 (5,
  '0.041*"air" + 0.027*"kedalam" + 0.021*"gula" + 0.015*"agaragar" + 0.014*"isi" + 0.013*"ais" + 0.012*"tuang" + 0.012*"kelapa" + 0.011*"susu" 

### Inspect Probability

In [ ]:
fizaProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata bawang pada setiap topik
fizaProb["garam"]

## Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
fizaWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"
fizaWN["Synonym_clean"] = cleanWN_Synonym(fizaWN["Synonym"])
fizaWN["Hypernim_clean"] = cleanWN(fizaWN["Hypernim"])
fizaWN["Hyponim_clean"] = cleanWN(fizaWN["Hyponim"])

# output 5 data teratas
fizaWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
1,1,cili,Synset('pepper.n.04'),[Synset('solanaceous_vegetable.n.01')],"[Synset('hot_pepper.n.02'), Synset('sweet_pepp...",sweet and hot varieties of fruits of plants of...,pepper,[solanaceous_vegetable],"[hot_pepper, sweet_pepper]"
2,1,garam,Synset('strategic_arms_limitation_talks.n.01'),[],[],negotiations between the United States and the...,strategic_arms_limitation_talks,None,None
3,1,air,Synset('aquatic.a.02'),[],[],operating or living or growing in water,aquatic,None,None
4,1,biji,Synset('semen.n.01'),[Synset('liquid_body_substance.n.01')],[Synset('milt.n.02')],the thick white fluid containing spermatozoa t...,semen,[liquid_body_substance],[milt]


### Make Ontology

In [ ]:
# Make Ontology
fizaOnto = get_ontology("http://test.org/fiza.owl")
makeOntology(fizaOnto, fizaWN)
fizaOnto.save(file = "fiza.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1yNT1pPMobdn7jzP3ggI-GrGZWrjvV0y4"

In [ ]:
wiki_fiza = pd.read_excel("Fiza_final.xlsx").iloc[:, 1:]
wiki_fiza.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_fiza["Category"] = translate(wiki_fiza["Category"])
wiki_fiza.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_fiza[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
fizaTabel_new = tabel_1981.copy()
fizaTabel_new["correct_category"] = all_cat
fizaTabel_new["suggestion"] = all_suggestion

# Export Result to File
fizaTabel_new.to_excel("fizaTabel_Result.xlsx")
fizaTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "fizaTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
fizaTabel_new = concatListInColumn(fizaTabel_new)
fizaTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, fizaTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_fiza.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
fizaWiki_new = wiki_df.copy()
fizaWiki_new["suggest"] = all_cat

# Export Result to file
fizaWiki_new.to_excel("fizaWiki_Result.xlsx")
fizaWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "fizaWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
fizaOntoWiki = get_ontology("http://test.org/fiza_onto_wiki.owl")
makeOntology_wiki(fizaOntoWiki, fizaWiki_new)
fizaOntoWiki.save(file = "fiza_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = fizaWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
fizaWN_new = listToDataframe(changesList, tableWordnet)
fizaWN_new.to_excel("fizaWN_Result.xlsx")
fizaWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "fizaWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Sajian Dapur Bonda

In [ ]:
!gdown --id "1FaTkf1nWCzEegMyeorB3g8ydRbHNjniy"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1FaTkf1nWCzEegMyeorB3g8ydRbHNjniy
To: /content/SajianDapurBonda.jl
100% 721k/721k [00:00<00:00, 70.8MB/s]


In [ ]:
bonda=pd.read_json(r'/content/SajianDapurBonda.jl', lines = True)
bonda.rename(columns = {"Page_description":"page_description"}, inplace=True)
bonda["page_description"] = delist(bonda, "page_description")
bonda = pre_cleaning_text(bonda, threshold=30)

## GUI Sajian Dapur Bonda

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = bonda.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.274821 -0.038605       1        1  55.830144
0      0.019467  0.121251       2        1  21.016001
5     -0.131465 -0.261862       3        1   7.481161
2      0.104037 -0.052523       4        1   5.069288
1     -0.193096  0.105438       5        1   4.817668
6      0.084154  0.080937       6        1   3.535355
3     -0.157918  0.045365       7        1   2.250383, topic_info=                  Term        Freq       Total Category  logprob  loglift
293             bawang  360.000000  360.000000  Default  30.0000  30.0000
367             periuk  111.000000  111.000000  Default  29.0000  29.0000
809             daging   94.000000   94.000000  Default  28.0000  28.0000
298                sos  108.000000  108.000000  Default  27.0000  27.0000
3                  kak  120.000000  120.000000  Default  26.0000  26.0000
4                 noor  119.000000  119.000000  Default  25.0000  25.0000
137              putih  176.000000  176.000000  Default  24.0000  24.0000
306               cili  199.000000  199.000000  Default  23.0000  23.0000
83            dipotong  105.000000  105.000000  Default  22.0000  22.0000
294              tumis  124.000000  124.000000  Default  21.0000  21.0000
288              garam  125.000000  125.000000  Default  20.0000  20.0000
30                gula  223.000000  223.000000  Default  19.0000  19.0000
109                nak   67.000000   67.000000  Default  18.0000  18.0000
360                air  196.000000  196.000000  Default  17.0000  17.0000
735               lada   39.000000   39.000000  Default  16.0000  16.0000
424             tomato   50.000000   50.000000  Default  15.0000  15.0000
609              noxxa   37.000000   37.000000  Default  14.0000  14.0000
371           pressure   36.000000   36.000000  Default  13.0000  13.0000
1302          agaragar   28.000000   28.000000  Default  12.0000  12.0000
459             kering   89.000000   89.000000  Default  11.0000  11.0000
411              kisar   72.000000   72.000000  Default  10.0000  10.0000
40              serbuk  114.000000  114.000000  Default   9.0000   9.0000
304               daun  144.000000  144.000000  Default   8.0000   8.0000
290              kuali   58.000000   58.000000  Default   7.0000   7.0000
364             fungsi   34.000000   34.000000  Default   6.0000   6.0000
37              tepung   54.000000   54.000000  Default   5.0000   5.0000
60              adunan   46.000000   46.000000  Default   4.0000   4.0000
127              manis   80.000000   80.000000  Default   3.0000   3.0000
303              tutup   73.000000   73.000000  Default   2.0000   2.0000
94                gram   53.000000   53.000000  Default   1.0000   1.0000
310               ikan  149.510845  150.287561   Topic1  -3.9522   0.5777
414               kuah  146.268483  147.046318   Topic1  -3.9741   0.5776
323               ulas   70.215257   70.992220   Topic1  -4.7080   0.5719
286               enaq   63.091778   63.868473   Topic1  -4.8150   0.5706
355              masin   58.891342   59.668124   Topic1  -4.8839   0.5698
1008            buncis   53.537002   54.314054   Topic1  -4.9792   0.5684
328           perencah   53.210040   53.986728   Topic1  -4.9853   0.5684
326             sambal   50.256075   51.032740   Topic1  -5.0425   0.5675
409               asam   49.919093   50.695720   Topic1  -5.0492   0.5674
300              pedas   44.684666   45.461947   Topic1  -5.1600   0.5656
299              tiram   43.682490   44.460453   Topic1  -5.1826   0.5652
305                sup   41.817612   42.595334   Topic1  -5.2263   0.5644
692              empuk   41.742449   42.521087   Topic1  -5.2281   0.5644
311              bilis   38.335412   39.112114   Topic1  -5.3132   0.5628
313             garing   37.129383   37.906993   Topic1  -5.3452   0.5621
810             kunyit   33.474043   34.251683   Topic1  -5.4488   

Topic: 1
gula, susu, tepung, gram, bahanbahan, kak, noor, telur, sederhana, pandan, sebati, adunan, darat, nanas, cair, kacang, biji, caracaranya, dapur, air, jagung, kismis, rebus, sajian, merah, daun, pekat, gandum, loyang, baking

Topic: 2
agaragar, coklat, doh, peti, puding, biskut, adunan, sejuk, kak, penuh, larut, cream, noor, diblend, cheese, mixer, acuan, pengisar, lapisan, oven, anakanak, balang, butter, tali, badam, chocolate, susun, eksperimen, suhu, dikeluarkan

Topic: 3
bawang, daging, mee, putih, bunga, bau, manis, daun, serbuk, pelaga, cili, ala, tsp, masakan, air, lawang, memanjang, kisar, tbsp, buah, cengkih, kayu, kering, hidangan, harum, jintan, kandar, batang, secukup, goreng

Topic: 4
alhamdulillah, kak, noor, order, make, nak, syukur, limau, cerita, ambil, lengkuas, hadrat, bendi, pleasure, kindly, please, reading, fund, sebalik, today, family, serba, enjoy, haha, resepinya, toskan, stirfry, cubalah, kasturi, nipis

Topic: 5
bawang, biji, cili, putih, ikan, kuah, 

[(0,
  '0.031*"gula" + 0.019*"susu" + 0.018*"tepung" + 0.018*"gram" + 0.017*"bahanbahan" + 0.015*"kak" + 0.015*"noor" + 0.015*"telur" + 0.014*"sederhana" + 0.014*"pandan"'),
 (1,
  '0.041*"agaragar" + 0.022*"coklat" + 0.017*"doh" + 0.017*"peti" + 0.016*"puding" + 0.016*"biskut" + 0.016*"adunan" + 0.015*"sejuk" + 0.013*"kak" + 0.013*"penuh"'),
 (2,
  '0.034*"bawang" + 0.032*"daging" + 0.017*"mee" + 0.014*"putih" + 0.013*"bunga" + 0.013*"bau" + 0.013*"manis" + 0.012*"daun" + 0.012*"serbuk" + 0.012*"pelaga"'),
 (3,
  '0.037*"alhamdulillah" + 0.028*"kak" + 0.027*"noor" + 0.023*"make" + 0.023*"order" + 0.021*"nak" + 0.019*"syukur" + 0.016*"limau" + 0.015*"cerita" + 0.014*"ambil"'),
 (4,
  '0.042*"bawang" + 0.026*"biji" + 0.024*"cili" + 0.020*"putih" + 0.019*"ikan" + 0.019*"kuah" + 0.019*"air" + 0.018*"bahanbahan" + 0.017*"kicap" + 0.016*"gula"'),
 (5,
  '0.071*"periuk" + 0.035*"noxxa" + 0.034*"pressure" + 0.030*"fungsi" + 0.028*"tombol" + 0.023*"tekan" + 0.022*"selamat" + 0.020*"arah" + 0.0

###Inspect Probability

In [ ]:
bondaProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata garam pada setiap topik
bondaProb["garam"]

##Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
bondaWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"

bondaWN["Synonym_clean"] = cleanWN_Synonym(bondaWN["Synonym"])
bondaWN["Hypernim_clean"] = cleanWN(bondaWN["Hypernim"])
bondaWN["Hyponim_clean"] = cleanWN(bondaWN["Hyponim"])

# output 5 data teratas
bondaWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,gula,Synset('pecan_pie.n.01'),[Synset('pie.n.01')],[],pie made of pecans and sugar and corn syrup an...,pecan_pie,[pie],None
1,1,susu,Synset('white.s.09'),[],[],(of coffee) having cream or milk added,white,None,None
2,1,tepung,Synset('meal.n.03'),[Synset('foodstuff.n.02')],"[Synset('cornmeal.n.01'), Synset('farina.n.01'...",coarsely ground foodstuff; especially seeds of...,meal,[foodstuff],"[cornmeal, farina, kibble, matzo_meal, oatmeal..."
3,1,gram,Synset('gram.n.01'),[Synset('metric_weight_unit.n.01')],[],a metric unit of weight equal to one thousandt...,gram,[metric_weight_unit],None
4,1,bahanbahan,None,None,None,None,None,None,None


### Make Ontology

In [ ]:
# Make Ontology
bondaOnto = get_ontology("http://test.org/bonda.owl")
makeOntology(bondaOnto, bondaWN)
bondaOnto.save(file = "bonda.owl")

##Wikipedia & Category

In [ ]:
!gdown --id "1qI339gOiXnnxiutzZJ5Th288DSvBFNX9"

In [ ]:
wiki_bonda = pd.read_excel("Bonda_final.xlsx").iloc[:, 1:]
wiki_bonda.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_bonda["Category"] = translate(wiki_bonda["Category"])
wiki_bonda.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_bonda[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
bondaTabel_new = tabel_1981.copy()
bondaTabel_new["correct_category"] = all_cat
bondaTabel_new["suggestion"] = all_suggestion

# Export Result to File
bondaTabel_new.to_excel("bondaTabel_Result.xlsx")
bondaTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "bondaTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
bondaTabel_new = concatListInColumn(bondaTabel_new)
bondaTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, bondaTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_bonda.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
bondaWiki_new = wiki_df.copy()
bondaWiki_new["suggest"] = all_cat

# Export Result to file
bondaWiki_new.to_excel("bondaWiki_Result.xlsx")
bondaWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "bondaWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
bondaOntoWiki = get_ontology("http://test.org/bonda_onto_wiki.owl")
makeOntology_wiki(bondaOntoWiki, bondaWiki_new)
bondaOntoWiki.save(file = "bonda_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = bondaWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()


benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
bondaWN_new = listToDataframe(changesList, tableWordnet)
bondaWN_new.to_excel("bondaWN_Result.xlsx")
bondaWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "bondaWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# banyak resepi

In [ ]:
!gdown --id "1Ti4mXlJY-shdnNp9-kcxoLgyQlZBpZFN"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1Ti4mXlJY-shdnNp9-kcxoLgyQlZBpZFN
To: /content/banyakResepi.jl
100% 131k/131k [00:00<00:00, 85.2MB/s]


In [ ]:
resepi = pd.read_json(r'/content/banyakResepi.jl', lines = True)

resepi.rename(columns = {"Page_description":"page_description"}, inplace=True)
resepi["page_description"] = delist(resepi, "page_description")
resepi = pre_cleaning_text(resepi, threshold=30)

## GUI Banyak Resepi

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = resepi.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4     -0.066144 -0.033108       1        1  23.659652
6     -0.137865 -0.007465       2        1  21.673622
5     -0.032099  0.088333       3        1  18.165464
0      0.007361 -0.021776       4        1  16.466621
1      0.066014 -0.132593       5        1  15.121684
2      0.066601  0.058206       6        1   4.162422
3      0.096133  0.048403       7        1   0.750534, topic_info=                 Term       Freq      Total Category  logprob  loglift
327             udang  20.000000  20.000000  Default  30.0000  30.0000
4                ikan  35.000000  35.000000  Default  29.0000  29.0000
288              ayam  37.000000  37.000000  Default  28.0000  28.0000
349             pastu  23.000000  23.000000  Default  27.0000  27.0000
373               mee  10.000000  10.000000  Default  26.0000  26.0000
226               sos  23.000000  23.000000  Default  25.0000  25.0000
8               merah  23.000000  23.000000  Default  24.0000  24.0000
384             lemak   3.000000   3.000000  Default  23.0000  23.0000
411             lazat   4.000000   4.000000  Default  22.0000  22.0000
337             telur  20.000000  20.000000  Default  21.0000  21.0000
537             beras   8.000000   8.000000  Default  20.0000  20.0000
785              roti  13.000000  13.000000  Default  19.0000  19.0000
6                biji  42.000000  42.000000  Default  18.0000  18.0000
10               cili  38.000000  38.000000  Default  17.0000  17.0000
254          tempoyak   4.000000   4.000000  Default  16.0000  16.0000
44                nak  26.000000  26.000000  Default  15.0000  15.0000
162            kunyit  11.000000  11.000000  Default  14.0000  14.0000
78            masukan   4.000000   4.000000  Default  13.0000  13.0000
145              nasi  11.000000  11.000000  Default  12.0000  12.0000
54             serbuk  15.000000  15.000000  Default  11.0000  11.0000
186             kesum   4.000000   4.000000  Default  10.0000  10.0000
11               padi  11.000000  11.000000  Default   9.0000   9.0000
35               step  32.000000  32.000000  Default   8.0000   8.0000
76               kuah  13.000000  13.000000  Default   7.0000   7.0000
94              tutup  12.000000  12.000000  Default   6.0000   6.0000
259             hidup   4.000000   4.000000  Default   5.0000   5.0000
328            maggie   3.000000   3.000000  Default   4.0000   4.0000
542           rebusan   5.000000   5.000000  Default   3.0000   3.0000
522            tunggu   5.000000   5.000000  Default   2.0000   2.0000
935             petai   4.000000   4.000000  Default   1.0000   1.0000
349             pastu  21.141921  23.231491   Topic1  -3.6529   1.3471
121               pes   5.164185   5.710947   Topic1  -5.0624   1.3408
102              pari   4.323283   4.870050   Topic1  -5.2401   1.3223
859            tomyam   3.482767   4.029018   Topic1  -5.4563   1.2957
352          kuayteaw   2.642626   3.188104   Topic1  -5.7324   1.2537
28               hehe   2.642401   3.188021   Topic1  -5.7325   1.2537
455           holland   2.642367   3.188104   Topic1  -5.7325   1.2536
873             botol   2.642183   3.188088   Topic1  -5.7325   1.2536
973           topping   2.641798   3.187975   Topic1  -5.7327   1.2535
706           kentang   2.641790   3.187995   Topic1  -5.7327   1.2535
1038          brokoli   1.801930   2.347116   Topic1  -6.1153   1.1771
1032          mariana   1.801926   2.347115   Topic1  -6.1153   1.1771
1033           ismail   1.801912   2.347103   Topic1  -6.1153   1.1771
1036        spaghetti   1.801893   2.347086   Topic1  -6.1153   1.1771
1031              ana   1.801858   2.347061   Topic1  -6.1153   1.1771
24             tulang   1.801778   2.347069   Topic1  -6.1154   1.1770
109               bji   1.801692   2.347187   Topic1  -6.1154   1.1769
453             kobis   1.801637   2.347163   Topic1  -6.1155  

Topic: 1
biji, nak, air, udang, ikan, bahanbahan, cili, lupa, step, letak, mee, selamat, sebati, mencuba, bawang, garam, hiris, telur, halus, petai, kembang, pekat, roti, dah, sos, takde, tau, pandan, sbs, susu

Topic: 2
telur, udang, step, roti, dah, garam, bawang, kunyit, cili, sikit, tomato, disediakan, kuning, sosej, daging, daun, sambal, air, gambar, serdak, butter, kat, padi, cantik, susu, slice, letakkan, cheese, puri, mendidih

Topic: 3
sos, cili, ikan, merah, tempoyak, kueyteow, masukan, char, patin, dah, air, garam, daun, step, kuah, kicap, tutup, maggie, kuey, kesum, teow, pekat, kunyit, hidup, padi, temerloh, disukai, asli, menggelegak, versi

Topic: 4
lemak, joha, hassan, udang, mee, lazat, lobster, berkrim, ketam, laksa, honey, rempah, bakar, kunyit, ikan, bawang, biji, cili, air, garam, hidupkan, step, pekat, padi, pastu, serbuk, hidup, serai, daun, lupa

Topic: 5
pastu, bawang, dah, biji, air, letak, cili, step, ayam, garam, daun, limau, mencuba, selamat, lupa, putih, g

[(0,
  '0.024*"biji" + 0.021*"nak" + 0.017*"air" + 0.014*"udang" + 0.013*"ikan" + 0.013*"bahanbahan" + 0.011*"cili" + 0.011*"lupa" + 0.011*"step" + 0.010*"letak"'),
 (1,
  '0.021*"telur" + 0.016*"udang" + 0.015*"step" + 0.014*"roti" + 0.013*"dah" + 0.012*"garam" + 0.012*"bawang" + 0.010*"kunyit" + 0.010*"cili" + 0.009*"sikit"'),
 (2,
  '0.017*"sos" + 0.017*"cili" + 0.014*"ikan" + 0.014*"merah" + 0.014*"tempoyak" + 0.011*"kueyteow" + 0.011*"masukan" + 0.011*"char" + 0.011*"patin" + 0.007*"dah"'),
 (3,
  '0.023*"lemak" + 0.017*"joha" + 0.017*"hassan" + 0.015*"udang" + 0.012*"mee" + 0.012*"lazat" + 0.012*"berkrim" + 0.012*"lobster" + 0.012*"ketam" + 0.012*"laksa"'),
 (4,
  '0.026*"pastu" + 0.026*"bawang" + 0.017*"dah" + 0.017*"biji" + 0.015*"air" + 0.015*"letak" + 0.011*"cili" + 0.010*"step" + 0.010*"ayam" + 0.010*"garam"'),
 (5,
  '0.032*"ikan" + 0.018*"air" + 0.015*"bawang" + 0.011*"garam" + 0.011*"daun" + 0.010*"tomato" + 0.010*"step" + 0.010*"lupa" + 0.010*"mencuba" + 0.010*"selamat"'

###Inspect Probability

In [ ]:
resepiProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata bawang pada setiap topik
resepiProb["garam"]

##Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
resepiWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"

resepiWN["Synonym_clean"] = cleanWN_Synonym(resepiWN["Synonym"])
resepiWN["Hypernim_clean"] = cleanWN(resepiWN["Hypernim"])
resepiWN["Hyponim_clean"] = cleanWN(resepiWN["Hyponim"])

# output 5 data teratas
resepiWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,biji,Synset('semen.n.01'),[Synset('liquid_body_substance.n.01')],[Synset('milt.n.02')],the thick white fluid containing spermatozoa t...,semen,[liquid_body_substance],[milt]
1,1,nak,Synset('cub.n.02'),[Synset('male_child.n.01')],[],a male child (a familiar term of address to a ...,cub,[male_child],None
2,1,air,Synset('aquatic.a.02'),[],[],operating or living or growing in water,aquatic,None,None
3,1,udang,Synset('shrimp.n.03'),[Synset('decapod_crustacean.n.01')],[Synset('snapping_shrimp.n.01')],small slender-bodied chiefly marine decapod cr...,shrimp,[decapod_crustacean],[snapping_shrimp]
4,1,ikan,Synset('fish.n.01'),[Synset('aquatic_vertebrate.n.01')],"[Synset('bony_fish.n.01'), Synset('bottom-feed...",any of various mostly cold-blooded aquatic ver...,fish,[aquatic_vertebrate],"[bony_fish, bottom-feeder, bottom_lurkers, car..."


### Make Ontology

In [ ]:
# Make Ontology
resepiOnto = get_ontology("http://test.org/resepi.owl")
makeOntology(resepiOnto, resepiWN)
resepiOnto.save(file = "resepi.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1kHTJ2BC4j55z421dEi7iN01UNEFLtSFg"

In [ ]:
wiki_resepi = pd.read_excel("Resepi_final.xlsx").iloc[:, 1:]
wiki_resepi.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_resepi["Category"] = translate(wiki_resepi["Category"])
wiki_resepi.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_resepi[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
resepiTabel_new = tabel_1981.copy()
resepiTabel_new["correct_category"] = all_cat
resepiTabel_new["suggestion"] = all_suggestion

# Export Result to File
resepiTabel_new.to_excel("resepiTabel_Result.xlsx")
resepiTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "resepiTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
resepiTabel_new = concatListInColumn(resepiTabel_new)
resepiTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, resepiTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_resepi.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
resepiWiki_new = wiki_df.copy()
resepiWiki_new["suggest"] = all_cat

# Export Result to file
resepiWiki_new.to_excel("resepiWiki_Result.xlsx")
resepiWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "resepiWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
resepiOntoWiki = get_ontology("http://test.org/resepi_onto_wiki.owl")
makeOntology_wiki(resepiOntoWiki, resepiWiki_new)
resepiOntoWiki.save(file = "resepi_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = resepiWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
resepiWN_new = listToDataframe(changesList, tableWordnet)
resepiWN_new.to_excel("resepiWN_Result.xlsx")
resepiWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "resepiWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Mamawandiha

In [ ]:
!gdown --id "1ke2lx4sNHgMh7Es9f1_PA3hw7ITKR-54"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1ke2lx4sNHgMh7Es9f1_PA3hw7ITKR-54
To: /content/Mamawandiha.jl
100% 2.78M/2.78M [00:00<00:00, 133MB/s]


In [ ]:
mawa = pd.read_json(r'/content/Mamawandiha.jl', lines = True)

mawa.rename(columns = {"Page_description":"page_description"}, inplace=True)
mawa["page_description"] = delist(mawa, "page_description")
mawa = pre_cleaning_text(mawa, threshold=30)

## GUI Mawawandiha

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = mawa.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
6      0.210252 -0.136258       1        1  31.722263
1      0.210743 -0.077546       2        1  23.928347
0      0.023532  0.190887       3        1  10.831769
3      0.019344  0.085002       4        1  10.459616
2     -0.264202 -0.159911       5        1   8.456646
4     -0.129022 -0.060243       6        1   7.948684
5     -0.070646  0.158069       7        1   6.652675, topic_info=                 Term         Freq        Total Category  logprob  loglift
220            bawang   886.000000   886.000000  Default  30.0000  30.0000
311            tengok   255.000000   255.000000  Default  29.0000  29.0000
232              cili   673.000000   673.000000  Default  28.0000  28.0000
398             ambil   272.000000   272.000000  Default  27.0000  27.0000
91                kak   247.000000   247.000000  Default  26.0000  26.0000
179          bahannya   146.000000   146.000000  Default  25.0000  25.0000
19             tepung   193.000000   193.000000  Default  24.0000  24.0000
2             madihaa  1712.000000  1712.000000  Default  23.0000  23.0000
365         resepinya   207.000000   207.000000  Default  22.0000  22.0000
1099            pulut   123.000000   123.000000  Default  21.0000  21.0000
156               nak   388.000000   388.000000  Default  20.0000  20.0000
25              telur   276.000000   276.000000  Default  19.0000  19.0000
1218             rima   124.000000   124.000000  Default  18.0000  18.0000
24               biji   996.000000   996.000000  Default  17.0000  17.0000
279           mentega   130.000000   130.000000  Default  16.0000  16.0000
480              ikan   255.000000   255.000000  Default  15.0000  15.0000
3                 dah   384.000000   384.000000  Default  14.0000  14.0000
197           kentang   234.000000   234.000000  Default  13.0000  13.0000
30               gula   547.000000   547.000000  Default  12.0000  12.0000
0               salam   734.000000   734.000000  Default  11.0000  11.0000
205              ayam   335.000000   335.000000  Default  10.0000  10.0000
259              susu   185.000000   185.000000  Default   9.0000   9.0000
72             tepong   185.000000   185.000000  Default   8.0000   8.0000
1           sejahtera   688.000000   688.000000  Default   7.0000   7.0000
20             gandum   162.000000   162.000000  Default   6.0000   6.0000
276            gambar   162.000000   162.000000  Default   5.0000   5.0000
246              ulas   397.000000   397.000000  Default   4.0000   4.0000
791            sotong   208.000000   208.000000  Default   3.0000   3.0000
64            mencuba   657.000000   657.000000  Default   2.0000   2.0000
48             sebati   210.000000   210.000000  Default   1.0000   1.0000
480              ikan   255.119905   255.870138   Topic1  -4.3356   1.1452
341            kunyit   154.946485   155.749977   Topic1  -4.8343   1.1430
93               nasi   123.268278   124.018899   Topic1  -5.0630   1.1421
1288            petai   111.012208   111.762138   Topic1  -5.1677   1.1414
524              jawa    98.885730    99.635368   Topic1  -5.2834   1.1406
494             serai    92.856810    93.606448   Topic1  -5.3463   1.1401
759           belacan    85.083586    85.833476   Topic1  -5.4337   1.1394
3378            kucai    62.492360    63.241973   Topic1  -5.7423   1.1362
646           gelugur    59.027046    59.776474   Topic1  -5.7994   1.1355
1294           senduk    54.687769    55.439834   Topic1  -5.8757   1.1345
2236            pucuk    49.822028    50.576800   Topic1  -5.9689   1.1331
516             empuk    49.487151    50.237532   Topic1  -5.9757   1.1331
250             masin    47.442366    48.193048   Topic1  -6.0179   1.1325
485               ala    47.065091    47.815118   Topic1  -6.0258   1.1323
342            jintan    45.675188    46.426698   Topic1  -6.0558   1.1318
492           diketuk

Topic: 1
tepung, mentega, madihaa, gandum, susu, gula, sebati, oven, garam, telur, cheese, biji, cream, salam, panaskan, sejahtera, gaul, doh, air, cheddar, bakar, selamat, roti, adunan, cair, suhu, mencuba, butter, pekat, suam

Topic: 2
madihaa, bawang, garam, ayam, biji, sos, putih, mencuba, lada, salam, selamat, halus, letak, air, serbuk, sejahtera, sotong, telur, isi, hitam, rebus, ratna, sup, carrot, tomato, kering, cincang, aka, goreng, spaghetti

Topic: 3
kak, madihaa, ambil, nak, resepinya, rima, dah, ayu, internet, kongsikan, berkongsi, gambar, makan, silap, asalnya, pulak, mencuba, idea, hehe, tengok, kat, kuih, susah, cuba, selamat, western, tkasih, ketandusan, meenmar, masaknya

Topic: 4
mencuba, madihaa, selamat, salam, sejahtera, gula, air, parut, ubi, aka, ratna, santan, sejuk, kelapa, campur, jagung, pisang, sotong, tepong, garam, sejukkan, beras, pandan, saiz, mangga, buah, susu, tuangkan, sikit, kuning

Topic: 5
pulut, madihaa, salam, sejahtera, nak, selamat, air, dah

[(0,
  '0.027*"tepung" + 0.019*"mentega" + 0.019*"madihaa" + 0.017*"gandum" + 0.017*"susu" + 0.016*"gula" + 0.016*"sebati" + 0.013*"oven" + 0.012*"garam" + 0.011*"telur"'),
 (1,
  '0.056*"madihaa" + 0.021*"bawang" + 0.018*"garam" + 0.016*"ayam" + 0.016*"biji" + 0.014*"sos" + 0.013*"putih" + 0.012*"mencuba" + 0.012*"lada" + 0.012*"salam"'),
 (2,
  '0.037*"kak" + 0.035*"madihaa" + 0.032*"ambil" + 0.026*"nak" + 0.025*"resepinya" + 0.024*"rima" + 0.023*"dah" + 0.015*"ayu" + 0.014*"internet" + 0.013*"kongsikan"'),
 (3,
  '0.026*"mencuba" + 0.025*"madihaa" + 0.025*"selamat" + 0.024*"salam" + 0.023*"sejahtera" + 0.022*"gula" + 0.016*"air" + 0.015*"parut" + 0.014*"ubi" + 0.014*"aka"'),
 (4,
  '0.025*"pulut" + 0.025*"madihaa" + 0.017*"salam" + 0.014*"sejahtera" + 0.012*"nak" + 0.012*"selamat" + 0.011*"air" + 0.010*"dah" + 0.010*"kasih" + 0.010*"kampong"'),
 (5,
  '0.043*"tengok" + 0.036*"bahannya" + 0.017*"biji" + 0.015*"salam" + 0.015*"telur" + 0.015*"sejahtera" + 0.014*"tepong" + 0.013*"bilis

### Inspect Probability

In [ ]:
mawaProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata garam pada setiap topik
mawaProb["garam"]

## Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
mawaWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"
mawaWN["Synonym_clean"] = cleanWN_Synonym(mawaWN["Synonym"])
mawaWN["Hypernim_clean"] = cleanWN(mawaWN["Hypernim"])
mawaWN["Hyponim_clean"] = cleanWN(mawaWN["Hyponim"])

# output 5 data teratas
mawaWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,tepung,Synset('meal.n.03'),[Synset('foodstuff.n.02')],"[Synset('cornmeal.n.01'), Synset('farina.n.01'...",coarsely ground foodstuff; especially seeds of...,meal,[foodstuff],"[cornmeal, farina, kibble, matzo_meal, oatmeal..."
1,1,mentega,Synset('penuche.n.01'),[Synset('fudge.n.01')],[],fudge made with brown sugar and butter and mil...,penuche,[fudge],None
2,1,madihaa,None,None,None,None,None,None,None
3,1,gandum,Synset('grain.n.02'),[Synset('foodstuff.n.02')],"[Synset('barley.n.01'), Synset('buckwheat.n.02...",foodstuff prepared from the starchy grains of ...,grain,[foodstuff],"[barley, buckwheat, corn, grist, groats, malt,..."
4,1,susu,Synset('white.s.09'),[],[],(of coffee) having cream or milk added,white,None,None


### Make Ontology

In [ ]:
# Make Ontology
mawaOnto = get_ontology("http://test.org/mamawandiha.owl")
makeOntology(mawaOnto, mawaWN)
mawaOnto.save(file = "mamawandiha.owl")

## Wikipedia & category

In [ ]:
!gdown --id "1mz7_P5JERgWh2IYZhGqUUlzs9RPrXRt_"

In [ ]:
wiki_mawa = pd.read_excel("Mawa_final.xlsx").iloc[:, 1:]
wiki_mawa.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_mawa["Category"] = translate(wiki_mawa["Category"])
wiki_mawa.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_mawa[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
mawaTabel_new = tabel_1981.copy()
mawaTabel_new["correct_category"] = all_cat
mawaTabel_new["suggestion"] = all_suggestion

# Export Result to File
mawaTabel_new.to_excel("mawaTabel_Result.xlsx")
mawaTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "mawaTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
mawaTabel_new = concatListInColumn(mawaTabel_new)
mawaTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, mawaTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_mawa.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
mawaWiki_new = wiki_df.copy()
mawaWiki_new["suggest"] = all_cat

# Export Result to file
mawaWiki_new.to_excel("mawaWiki_Result.xlsx")
mawaWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "mawaWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
mawaOntoWiki = get_ontology("http://test.org/mawa_onto_wiki.owl")
makeOntology_wiki(mawaOntoWiki, mawaWiki_new)
mawaOntoWiki.save(file = "mawa_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = mawaWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
mawaWN_new = listToDataframe(changesList, tableWordnet)
mawaWN_new.to_excel("mawaWN_Result.xlsx")
mawaWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "mawaWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Salamisimon

In [ ]:
!gdown --id "14W7cUo_TjgMeFbkWtYIpcwprR-lkWG6o"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=14W7cUo_TjgMeFbkWtYIpcwprR-lkWG6o
To: /content/Salamisimon.jl
100% 1.64M/1.64M [00:00<00:00, 181MB/s]


In [ ]:
misimon = pd.read_json(r'/content/Salamisimon.jl', lines = True)

misimon.rename(columns = {"Page_description":"page_description"}, inplace=True)
misimon["page_description"] = delist(misimon, "page_description")
misimon = pre_cleaning_text(misimon, threshold=30)

## GUI Salamisimon

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = misimon.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.221748  0.038121       1        1  29.199262
4     -0.239962  0.044908       2        1  27.069562
6     -0.100714 -0.221139       3        1  14.573332
3      0.101160  0.163806       4        1   9.491792
2      0.230226 -0.145260       5        1   8.012747
1      0.118744  0.062544       6        1   6.647604
5      0.112294  0.057020       7        1   5.005700, topic_info=                   Term        Freq       Total Category  logprob  loglift
6                  ikan  610.000000  610.000000  Default  30.0000  30.0000
55               bawang  815.000000  815.000000  Default  29.0000  29.0000
71                 gula  510.000000  510.000000  Default  28.0000  28.0000
63                makan  480.000000  480.000000  Default  27.0000  27.0000
58                 cili  504.000000  504.000000  Default  26.0000  26.0000
1771                kek  157.000000  157.000000  Default  25.0000  25.0000
39                 biji  723.000000  723.000000  Default  24.0000  24.0000
30               tepung  191.000000  191.000000  Default  23.0000  23.0000
25                 ayam  408.000000  408.000000  Default  22.0000  22.0000
331                asam  230.000000  230.000000  Default  21.0000  21.0000
13                 gram  275.000000  275.000000  Default  20.0000  20.0000
28               goreng  412.000000  412.000000  Default  19.0000  19.0000
61                  nak  236.000000  236.000000  Default  18.0000  18.0000
106              perasa  358.000000  358.000000  Default  17.0000  17.0000
47                  air  610.000000  610.000000  Default  16.0000  16.0000
251                susu  145.000000  145.000000  Default  15.0000  15.0000
1042               sate   77.000000   77.000000  Default  14.0000  14.0000
115               telur  338.000000  338.000000  Default  13.0000  13.0000
44                garam  543.000000  543.000000  Default  12.0000  12.0000
14                putih  524.000000  524.000000  Default  11.0000  11.0000
250              serbuk  224.000000  224.000000  Default  10.0000  10.0000
390             firdaus  143.000000  143.000000  Default   9.0000   9.0000
225              adunan  109.000000  109.000000  Default   8.0000   8.0000
363                 sos  154.000000  154.000000  Default   7.0000   7.0000
112              kacang  177.000000  177.000000  Default   6.0000   6.0000
124               encik  130.000000  130.000000  Default   5.0000   5.0000
4                  amie  620.000000  620.000000  Default   4.0000   4.0000
148             cincang  133.000000  133.000000  Default   3.0000   3.0000
540              daging  216.000000  216.000000  Default   2.0000   2.0000
96                bunga  183.000000  183.000000  Default   1.0000   1.0000
148             cincang  132.769746  133.520328   Topic1  -4.7636   1.2254
521               betik   83.636995   84.386259   Topic1  -5.2258   1.2221
997                itik   56.940501   57.693787   Topic1  -5.6102   1.2179
1050             kepala   52.842357   53.594806   Topic1  -5.6849   1.2169
1115              cocos   43.888862   44.639267   Topic1  -5.8706   1.2141
97               lawang   42.136314   42.885419   Topic1  -5.9113   1.2134
99               pelaga   37.034060   37.783049   Topic1  -6.0404   1.2110
2922              lodeh   35.689361   36.438329   Topic1  -6.0774   1.2103
584             cengkih   35.517141   36.266460   Topic1  -6.0822   1.2101
827             tradisi   31.799849   32.551119   Topic1  -6.1928   1.2077
362                madu   30.509775   31.273175   Topic1  -6.2342   1.2063
2081               inci   28.560551   29.311637   Topic1  -6.3002   1.2051
386          kekuningan   26.733579   27.483113   Topic1  -6.3663   1.2034
153             perahan   26.435466   27.185167   Topic1  -6.3775   1.2031
601             penumis   25.357354   26.106440   Topic1  -6.4192   1.2019
1706           shiita

Topic: 1
bawang, putih, amie, biji, cili, garam, merah, serbuk, air, daging, goreng, udang, kulit, bunga, perasa, tomato, ayam, tumbuk, manis, cincang, tumis, panaskan, gaul, bahanbahan, dapur, ulas, nasi, halia, lada, sos

Topic: 2
makan, biskut, nak, pie, kak, ketuhar, oven, kacang, yong, shepherds, panas, sarapan, belajar, dadar, buah, kering, selamat, durian, masakan, puff, popia, minum, anakanak, merah, kelas, daun, amie, kelapa, lupa, tengahari

Topic: 3
sate, amie, encik, kasih, terima, suami, firdaus, roti, makan, kuih, gambar, food, ingredients, chicken, post, malaysia, hosted, submitting, penang, water, set, nampak, oil, farid, fest, add, tbsp, pisang, pesan, milk

Topic: 4
makan, nak, ayam, amie, goreng, beli, ikan, nasi, firdaus, hidangan, sos, pizza, panggang, makanan, keju, fairuz, kentang, encik, bakar, suami, abang, jenis, nampak, anakanak, dah, telur, selera, ahaks, zaitun, fuad

Topic: 5
ikan, biji, bawang, cili, garam, air, daun, asam, gula, perasa, sambal, kisar, pu

[(0,
  '0.034*"bawang" + 0.021*"putih" + 0.018*"amie" + 0.016*"biji" + 0.015*"cili" + 0.014*"garam" + 0.014*"merah" + 0.013*"serbuk" + 0.012*"air" + 0.012*"daging"'),
 (1,
  '0.017*"makan" + 0.014*"biskut" + 0.009*"nak" + 0.009*"pie" + 0.008*"kak" + 0.007*"ketuhar" + 0.006*"oven" + 0.006*"kacang" + 0.006*"yong" + 0.006*"shepherds"'),
 (2,
  '0.018*"sate" + 0.014*"amie" + 0.012*"encik" + 0.011*"kasih" + 0.010*"terima" + 0.010*"suami" + 0.008*"firdaus" + 0.008*"roti" + 0.007*"makan" + 0.007*"kuih"'),
 (3,
  '0.026*"makan" + 0.018*"nak" + 0.015*"ayam" + 0.013*"amie" + 0.011*"goreng" + 0.010*"beli" + 0.010*"ikan" + 0.009*"nasi" + 0.009*"firdaus" + 0.009*"hidangan"'),
 (4,
  '0.036*"ikan" + 0.026*"biji" + 0.020*"bawang" + 0.019*"cili" + 0.018*"garam" + 0.018*"air" + 0.015*"daun" + 0.015*"asam" + 0.014*"gula" + 0.014*"perasa"'),
 (5,
  '0.013*"makan" + 0.013*"amie" + 0.013*"tepung" + 0.013*"ayam" + 0.013*"jagung" + 0.012*"firdaus" + 0.012*"goreng" + 0.011*"air" + 0.011*"hati" + 0.010*"soto"'

### Inspect Probability

In [ ]:
misimonProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata bawang pada setiap topik
misimonProb["bawang"]

## Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
misimonWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"

misimonWN["Synonym_clean"] = cleanWN_Synonym(misimonWN["Synonym"])
misimonWN["Hypernim_clean"] = cleanWN(misimonWN["Hypernim"])
misimonWN["Hyponim_clean"] = cleanWN(misimonWN["Hyponim"])

# output 5 data teratas
misimonWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
1,1,putih,Synset('canescent.s.02'),[],[],covered with fine whitish hairs or down,canescent,None,None
2,1,amie,None,None,None,None,None,None,None
3,1,biji,Synset('semen.n.01'),[Synset('liquid_body_substance.n.01')],[Synset('milt.n.02')],the thick white fluid containing spermatozoa t...,semen,[liquid_body_substance],[milt]
4,1,cili,Synset('pepper.n.04'),[Synset('solanaceous_vegetable.n.01')],"[Synset('hot_pepper.n.02'), Synset('sweet_pepp...",sweet and hot varieties of fruits of plants of...,pepper,[solanaceous_vegetable],"[hot_pepper, sweet_pepper]"


### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/salamisimon.owl")
makeOntology(ontoName, misimonWN)
ontoName.save(file = "salamisimon.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1B-ClnWPpGgnTGjM1VrHHizVVm3IP8giP"

In [ ]:
wiki_misimon = pd.read_excel("Salamisimon_final.xlsx").iloc[:, 1:]
wiki_misimon.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_misimon["Category"] = translate(wiki_misimon["Category"])
wiki_misimon.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_misimon[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
misimonTabel_new = tabel_1981.copy()
misimonTabel_new["correct_category"] = all_cat
misimonTabel_new["suggestion"] = all_suggestion

# Export Result to File
misimonTabel_new.to_excel("misimonTabel_Result.xlsx")
misimonTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "misimonTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
misimonTabel_new = concatListInColumn(misimonTabel_new)
misimonTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, misimonTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_misimon.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
misimonWiki_new = wiki_df.copy()
misimonWiki_new["suggest"] = all_cat

# Export Result to file
misimonWiki_new.to_excel("misimonWiki_Result.xlsx")
misimonWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "misimonWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
misimonOntoWiki = get_ontology("http://test.org/misimon_onto_wiki.owl")
makeOntology_wiki(misimonOntoWiki, misimonWiki_new)
misimonOntoWiki.save(file = "misimon_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = misimonWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
misimonWN_new = listToDataframe(changesList, tableWordnet)
misimonWN_new.to_excel("misimonWN_Result.xlsx")
misimonWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "misimonWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Qasey Honey

In [ ]:
!gdown --id "1V5yx_lco3yEGCWOnCoerYr2-U2rrSYw5"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1V5yx_lco3yEGCWOnCoerYr2-U2rrSYw5
To: /content/Qasey.jl
100% 2.78M/2.78M [00:00<00:00, 193MB/s]


In [ ]:
qasey=pd.read_json(r'/content/Qasey.jl', lines = True)
qasey = pre_cleaning_text(qasey, threshold=20)

## GUI Qasey Honey

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = qasey.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.231454  0.105580       1        1  38.712002
5      0.220322 -0.137553       2        1  19.284725
1      0.024289  0.150646       3        1  13.921810
0      0.036914  0.022359       4        1   9.319309
4     -0.234421  0.096124       5        1   8.574352
3     -0.114883 -0.305341       6        1   5.106577
6     -0.163676  0.068186       7        1   5.081226, topic_info=                    Term        Freq       Total Category  logprob  loglift
98                tepung  898.000000  898.000000  Default  30.0000  30.0000
274                  kek  575.000000  575.000000  Default  29.0000  29.0000
129                 ayam  451.000000  451.000000  Default  28.0000  28.0000
112               bawang  735.000000  735.000000  Default  27.0000  27.0000
179                 biji  809.000000  809.000000  Default  26.0000  26.0000
95                adunan  307.000000  307.000000  Default  25.0000  25.0000
116                 cili  464.000000  464.000000  Default  24.0000  24.0000
5438                 tel  171.000000  171.000000  Default  23.0000  23.0000
263                telur  474.000000  474.000000  Default  22.0000  22.0000
289                  teh  165.000000  165.000000  Default  21.0000  21.0000
271              mentega  265.000000  265.000000  Default  20.0000  20.0000
27                   air  724.000000  724.000000  Default  19.0000  19.0000
44                  gula  811.000000  811.000000  Default  18.0000  18.0000
283               biskut  214.000000  214.000000  Default  17.0000  17.0000
463               butter  264.000000  264.000000  Default  16.0000  16.0000
458               coklat  231.000000  231.000000  Default  15.0000  15.0000
216                  sos  264.000000  264.000000  Default  14.0000  14.0000
21                 garam  652.000000  652.000000  Default  13.0000  13.0000
180                merah  395.000000  395.000000  Default  12.0000  12.0000
259               gandum  179.000000  179.000000  Default  11.0000  11.0000
22                  daun  333.000000  333.000000  Default  10.0000  10.0000
0                  salam  513.000000  513.000000  Default   9.0000   9.0000
2040            membakar  114.000000  114.000000  Default   8.0000   8.0000
41                goreng  305.000000  305.000000  Default   7.0000   7.0000
11                  ikan  292.000000  292.000000  Default   6.0000   6.0000
270               loyang  161.000000  161.000000  Default   5.0000   5.0000
268                 suhu  159.000000  159.000000  Default   4.0000   4.0000
109                sejuk  139.000000  139.000000  Default   3.0000   3.0000
412                 lada  221.000000  221.000000  Default   2.0000   2.0000
358                 apam  121.000000  121.000000  Default   1.0000   1.0000
112               bawang  735.265674  735.937276   Topic1  -3.3945   0.9481
116                 cili  464.234085  464.905654   Topic1  -3.8544   0.9476
22                  daun  332.889584  333.561400   Topic1  -4.1869   0.9470
11                  ikan  291.469972  292.141293   Topic1  -4.3198   0.9467
181                 ulas  212.868055  213.539719   Topic1  -4.6341   0.9459
136                 inci  171.890585  172.563560   Topic1  -4.8479   0.9451
119                tumis  166.280950  166.952542   Topic1  -4.8811   0.9450
115               kunyit  160.900253  161.571589   Topic1  -4.9140   0.9449
408                hiris  159.481563  160.153072   Topic1  -4.9228   0.9448
719                udang  158.362681  159.034251   Topic1  -4.9299   0.9448
18                  asam  158.334530  159.006743   Topic1  -4.9300   0.9448
182               batang  157.125606  157.797090   Topic1  -4.9377   0.9448
381                 kari  151.911035  152.582756   Topic1  -4.9714   0.9446
114                halia  137.723297  138.394743   Topic1  -5.0695   0.9442
535              dihiris  123.646588  124.320106   

Topic: 1
ayam, membakar, sos, jus, roti, telur, lada, garing, wangi, dadu, foil, aluminium, goreng, yea, tepung, hangus, tuk, tertanggal, kalbu, penganut, hindu, kekurangan, lemon, hitam, tauhu, panas, mee, maggi, nak, perasa

Topic: 2
coklat, salam, sejuk, apam, doh, air, gula, nak, masakan, lapisan, bakar, semua, sediakan, jenis, peti, ambil, cair, sejukkan, gaul, keluarkan, satukan, pandan, ais, ketepikan, membuatnya, panaspanas, buah, letakkan, tuang, kasih

Topic: 3
biji, bawang, air, garam, cili, gula, merah, daun, putih, ikan, secukup, kisar, serbuk, goreng, salam, ayam, ulas, sejahtera, panaskan, sos, inci, lada, tumis, kunyit, hiris, udang, asam, gaul, batang, kari

Topic: 4
tepung, teh, berprotein, gandum, chef, sesuai, asma, oktober, jagung, ubi, acuan, roti, bancuhan, serba, fungsinya, pulut, yis, dimakan, kuih, kukus, serbaguna, ros, hidangkan, pau, berbeza, krimer, berwarna, sempurna, memekatkan, penggunaan

Topic: 5
tel, darul, taman, selangor, qasey, pensonic, ehsan, ba

[(0,
  '0.042*"ayam" + 0.022*"membakar" + 0.017*"sos" + 0.014*"jus" + 0.014*"roti" + 0.012*"telur" + 0.010*"lada" + 0.010*"garing" + 0.010*"wangi" + 0.010*"dadu"'),
 (1,
  '0.025*"coklat" + 0.019*"salam" + 0.016*"sejuk" + 0.015*"apam" + 0.014*"doh" + 0.013*"air" + 0.013*"gula" + 0.012*"nak" + 0.011*"masakan" + 0.011*"lapisan"'),
 (2,
  '0.034*"biji" + 0.034*"bawang" + 0.028*"air" + 0.027*"garam" + 0.021*"cili" + 0.018*"gula" + 0.018*"merah" + 0.015*"daun" + 0.015*"putih" + 0.013*"ikan"'),
 (3,
  '0.165*"tepung" + 0.026*"teh" + 0.024*"berprotein" + 0.024*"gandum" + 0.022*"chef" + 0.019*"sesuai" + 0.018*"asma" + 0.018*"oktober" + 0.014*"jagung" + 0.014*"ubi"'),
 (4,
  '0.035*"tel" + 0.017*"darul" + 0.012*"taman" + 0.010*"selangor" + 0.010*"qasey" + 0.009*"pensonic" + 0.009*"ehsan" + 0.008*"bakery" + 0.008*"youtube" + 0.007*"fax"'),
 (5,
  '0.050*"kek" + 0.028*"gula" + 0.028*"adunan" + 0.028*"tepung" + 0.027*"telur" + 0.023*"mentega" + 0.022*"butter" + 0.020*"biskut" + 0.015*"loyang" + 0.

### Inspect Probability

In [ ]:
qaseyProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata garam pada setiap topik
qaseyProb["garam"]

## Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
qaseyWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"

qaseyWN["Synonym_clean"] = cleanWN_Synonym(qaseyWN["Synonym"])
qaseyWN["Hypernim_clean"] = cleanWN(qaseyWN["Hypernim"])
qaseyWN["Hyponim_clean"] = cleanWN(qaseyWN["Hyponim"])

# output 5 data teratas
qaseyWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,ayam,Synset('domestic_fowl.n.01'),[Synset('gallinaceous_bird.n.01')],"[Synset('bantam.n.01'), Synset('chicken.n.02')...",a domesticated gallinaceous bird thought to be...,domestic_fowl,[gallinaceous_bird],"[bantam, chicken, cochin, cornish, dorking, ga..."
1,1,membakar,Synset('inflame.v.05'),[Synset('worsen.v.01')],[],become inflamed; get sore,inflame,[worsen],None
2,1,sos,Synset('sauce.n.01'),[Synset('condiment.n.01')],"[Synset('aioli.n.01'), Synset('allemande.n.01'...",flavorful relish or dressing or topping served...,sauce,[condiment],"[aioli, allemande, anchovy_sauce, apricot_sauc..."
3,1,jus,Synset('juice.n.04'),[Synset('liquid_body_substance.n.01')],"[Synset('cancer_juice.n.01'), Synset('digestiv...",any of several liquids of the body,juice,[liquid_body_substance],"[cancer_juice, digestive_juice]"
4,1,roti,Synset('bread.n.01'),"[Synset('baked_goods.n.01'), Synset('starches....","[Synset('anadama_bread.n.01'), Synset('bap.n.0...",food made from dough of flour or meal and usua...,bread,"[baked_goods, starches]","[anadama_bread, bap, barmbrack, breadstick, br..."


### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/QaseyHoney.owl")
makeOntology(ontoName, qaseyWN)
ontoName.save(file = "QaseyHoney.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1z804RUur9y4XZwXg-gq9Wi9L9GG07Ed-"

In [ ]:
wiki_qasey = pd.read_excel("Qasey Honey_final.xlsx").iloc[:, 1:]
wiki_qasey.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_qasey["Category"] = translate(wiki_qasey["Category"])
wiki_qasey.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_qasey[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
qaseyTabel_new = tabel_1981.copy()
qaseyTabel_new["correct_category"] = all_cat
qaseyTabel_new["suggestion"] = all_suggestion

# Export Result to File
qaseyTabel_new.to_excel("qaseyTabel_Result.xlsx")
qaseyTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "qaseyTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
qaseyTabel_new = concatListInColumn(qaseyTabel_new)
qaseyTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, qaseyTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_qasey.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
qaseyWiki_new = wiki_df.copy()
qaseyWiki_new["suggest"] = all_cat

# Export Result to file
qaseyWiki_new.to_excel("qaseyWiki_Result.xlsx")
qaseyWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "qaseyWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
qaseyOntoWiki = get_ontology("http://test.org/qasey_onto_wiki.owl")
makeOntology_wiki(qaseyOntoWiki, qaseyWiki_new)
qaseyOntoWiki.save(file = "qasey_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = qaseyWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
qaseyWN_new = listToDataframe(changesList, tableWordnet)
qaseyWN_new.to_excel("qaseyWN_Result.xlsx")
qaseyWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "qaseyWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Tiffin Biru

In [ ]:
!gdown --id "1e-4GSAgdEsp9DotoqZj1niqXozT4BRIS"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1e-4GSAgdEsp9DotoqZj1niqXozT4BRIS
To: /content/Tiffinbiru.jl
100% 11.8M/11.8M [00:00<00:00, 41.9MB/s]


In [ ]:
tiffin=pd.read_json(r'/content/Tiffinbiru.jl', lines = True)
tiffin = pre_cleaning_text(tiffin, threshold=20)

## GUI TiffinBiru

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = tiffin.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5      0.229652 -0.048033       1        1  38.097012
0      0.175423 -0.046796       2        1  20.375566
2      0.093547  0.227229       3        1  15.226402
6      0.153967 -0.136797       4        1  10.203275
4     -0.384911 -0.024462       5        1   6.277035
3     -0.103751  0.238644       6        1   5.143543
1     -0.163927 -0.209785       7        1   4.677167, topic_info=                 Term          Freq         Total Category  logprob  loglift
33              camca  11331.000000  11331.000000  Default  30.0000  30.0000
45             bawang   7965.000000   7965.000000  Default  29.0000  29.0000
144            tepung   1766.000000   1766.000000  Default  28.0000  28.0000
49               inci   3719.000000   3719.000000  Default  27.0000  27.0000
34                teh   5004.000000   5004.000000  Default  26.0000  26.0000
38             santan   1848.000000   1848.000000  Default  25.0000  25.0000
85               gula   4454.000000   4454.000000  Default  24.0000  24.0000
110               sos   1917.000000   1917.000000  Default  23.0000  23.0000
74             kunyit   1943.000000   1943.000000  Default  22.0000  22.0000
11             rendam   1134.000000   1134.000000  Default  21.0000  21.0000
8                biji  10954.000000  10954.000000  Default  20.0000  20.0000
17              udang   1171.000000   1171.000000  Default  19.0000  19.0000
30               ikan   2545.000000   2545.000000  Default  18.0000  18.0000
53               cili   3838.000000   3838.000000  Default  17.0000  17.0000
27             toskan   1924.000000   1924.000000  Default  16.0000  16.0000
46              merah   4014.000000   4014.000000  Default  15.0000  15.0000
14              hiris   2760.000000   2760.000000  Default  14.0000  14.0000
40                mat   2129.000000   2129.000000  Default  13.0000  13.0000
20             kering    623.000000    623.000000  Default  12.0000  12.0000
124             telur   1640.000000   1640.000000  Default  11.0000  11.0000
36               asam   1425.000000   1425.000000  Default  10.0000  10.0000
47               ulas   3729.000000   3729.000000  Default   9.0000   9.0000
433              roti    533.000000    533.000000  Default   8.0000   8.0000
505               kak    621.000000    621.000000  Default   7.0000   7.0000
1              keping   1616.000000   1616.000000  Default   6.0000   6.0000
75              hidup   1102.000000   1102.000000  Default   5.0000   5.0000
29               pati    810.000000    810.000000  Default   4.0000   4.0000
57            hirisan   1555.000000   1555.000000  Default   3.0000   3.0000
50              halia   1484.000000   1484.000000  Default   2.0000   2.0000
148            gandum    751.000000    751.000000  Default   1.0000   1.0000
50              halia   1483.793174   1484.575154   Topic1  -4.2532   0.9645
92             kuntum   1091.416696   1092.198584   Topic1  -4.5604   0.9643
93              bunga   1090.981076   1091.762930   Topic1  -4.5608   0.9643
164              kari    866.945850    867.727609   Topic1  -4.7906   0.9641
89           lengkuas    671.355057    672.136612   Topic1  -5.0463   0.9639
345               jus    648.318059    649.101628   Topic1  -5.0812   0.9638
37               jawa    604.826483    605.608865   Topic1  -5.1507   0.9637
163            rempah    568.150573    568.932487   Topic1  -5.2132   0.9637
205            jintan    549.559721    550.341041   Topic1  -5.2465   0.9636
204          ketumbar    400.035999    400.817415   Topic1  -5.5640   0.9631
640             lumat    394.685373    395.468534   Topic1  -5.5775   0.9631
218            lawang    369.861723    370.642558   Topic1  -5.6425   0.9629
500            keladi    369.576608    370.395730   Topic1  -5.6432   0.9628
201             bulat    329.686279    330.471630   Topic1  -5.7575   0.9627
302   

Topic: 1
camca, bawang, biji, sos, hiris, cili, hirisan, putih, halus, ulas, membuatnya, daun, merah, kicap, batang, bahanbahannya, lada, goreng, toskan, air, hiasan, tomato, teh, dihiris, tiram, garam, udang, ayam, didadu, tapis

Topic: 2
kering, rendam, udang, toskan, kacang, ketepikan, cekak, kembang, panaskan, kerat, ambil, sambal, kuah, nie, laksa, cuci, keperluan, lembut, goreng, thai, hiris, tauhu, sayur, yea, tumbuk, isi, garing, air, bilis, timun

Topic: 3
camca, teh, gula, tepung, mat, telur, membuatnya, halus, mentega, gandum, air, biji, susu, bahanbahannya, gred, garam, esen, vanilla, cream, diayak, gram, baking, pandan, kuning, pasir, segar, kek, manis, serbuk, sebati

Topic: 4
roti, doh, tin, halus, bread, flour, mat, telur, sosej, lecek, teh, dimakan, keju, serbuk, parutan, camca, kaedah, keping, lemon, hitam, cincang, didadu, sayur, yeast, instant, membuatnya, pilih, kecuali, tuna, improver

Topic: 5
kak, nak, kasih, terima, dah, kat, gambar, salah, nama, ben, tau, tag,

[(0,
  '0.062*"camca" + 0.045*"bawang" + 0.041*"biji" + 0.034*"sos" + 0.023*"hiris" + 0.021*"cili" + 0.020*"hirisan" + 0.019*"putih" + 0.019*"halus" + 0.017*"ulas"'),
 (1,
  '0.046*"kering" + 0.042*"rendam" + 0.038*"udang" + 0.023*"toskan" + 0.020*"kacang" + 0.018*"ketepikan" + 0.016*"cekak" + 0.015*"kembang" + 0.013*"panaskan" + 0.012*"kerat"'),
 (2,
  '0.073*"camca" + 0.054*"teh" + 0.043*"gula" + 0.042*"tepung" + 0.026*"mat" + 0.024*"telur" + 0.024*"membuatnya" + 0.022*"halus" + 0.019*"mentega" + 0.018*"gandum"'),
 (3,
  '0.038*"roti" + 0.025*"doh" + 0.024*"tin" + 0.018*"halus" + 0.016*"bread" + 0.015*"flour" + 0.015*"mat" + 0.013*"telur" + 0.011*"sosej" + 0.011*"lecek"'),
 (4,
  '0.031*"kak" + 0.024*"nak" + 0.015*"kasih" + 0.013*"terima" + 0.013*"dah" + 0.011*"kat" + 0.010*"gambar" + 0.009*"salah" + 0.008*"nama" + 0.007*"ben"'),
 (5,
  '0.063*"biji" + 0.045*"bawang" + 0.045*"camca" + 0.029*"inci" + 0.026*"halus" + 0.023*"merah" + 0.023*"ulas" + 0.021*"putih" + 0.020*"cili" + 0.020*"

### Inspect Probability

In [ ]:
tiffinProb = make_dictionary_terms_prob(ldamodel, id_map, d)
#probabilitas kata garam pada setiap topik
tiffinProb["garam"]

## Wordnet

In [ ]:
# dfWN = make_dataframe_wordnet(modelToText(ldamodel, 10)) # mengambil 10 kata dari tiap topik
tiffinWN = dfWordnet.copy()
pattern_wn = r"('.+?\.)"

tiffinWN["Synonym_clean"] = cleanWN_Synonym(tiffinWN["Synonym"])
tiffinWN["Hypernim_clean"] = cleanWN(tiffinWN["Hypernim"])
tiffinWN["Hyponim_clean"] = cleanWN(tiffinWN["Hyponim"])

# output 5 data teratas
tiffinWN.head()

,No. Topic,Word,Synonym,Hypernim,Hyponim,Definition,Synonym_clean,Hypernim_clean,Hyponim_clean
0,1,camca,Synset('spoon.n.01'),"[Synset('container.n.01'), Synset('cutlery.n.0...","[Synset('dessert_spoon.n.01'), Synset('runcibl...",a piece of cutlery with a shallow bowl-shaped ...,spoon,"[container, cutlery]","[dessert_spoon, runcible_spoon, soupspoon, sug..."
1,1,bawang,Synset('gazpacho.n.01'),[Synset('soup.n.01')],[],a soup made with chopped tomatoes and onions a...,gazpacho,[soup],None
2,1,biji,Synset('semen.n.01'),[Synset('liquid_body_substance.n.01')],[Synset('milt.n.02')],the thick white fluid containing spermatozoa t...,semen,[liquid_body_substance],[milt]
3,1,sos,Synset('sauce.n.01'),[Synset('condiment.n.01')],"[Synset('aioli.n.01'), Synset('allemande.n.01'...",flavorful relish or dressing or topping served...,sauce,[condiment],"[aioli, allemande, anchovy_sauce, apricot_sauc..."
4,1,hiris,Synset('piece.n.08'),[Synset('helping.n.01')],"[Synset('cutlet.n.01'), Synset('fillet.n.02')]",a serving that has been cut from a larger portion,piece,[helping],"[cutlet, fillet]"


### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/TiffinBiru.owl")
makeOntology(ontoName, tiffinWN)
ontoName.save(file = "TiffinBiru.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1ANVxFEYIcw0tAsHYOJMABap0OFCDu1V2"

In [ ]:
wiki_tiffin = pd.read_excel("Tiffin Biru_final.xlsx").iloc[:, 1:]
wiki_tiffin.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_tiffin["Category"] = translate(wiki_tiffin["Category"])
wiki_tiffin.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_tiffin[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
tiffinTabel_new = tabel_1981.copy()
tiffinTabel_new["correct_category"] = all_cat
tiffinTabel_new["suggestion"] = all_suggestion

# Export Result to File
tiffinTabel_new.to_excel("tiffinTabel_Result.xlsx")
tiffinTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tiffinTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
tiffinTabel_new = concatListInColumn(tiffinTabel_new)
tiffinTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, tiffinTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_tiffin.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
tiffinWiki_new = wiki_df.copy()
tiffinWiki_new["suggest"] = all_cat

# Export Result to file
tiffinWiki_new.to_excel("tiffinWiki_Result.xlsx")
tiffinWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tiffinWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
tiffinOntoWiki = get_ontology("http://test.org/tiffin_onto_wiki.owl")
makeOntology_wiki(tiffinOntoWiki, tiffinWiki_new)
tiffinOntoWiki.save(file = "tiffin_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = tiffinWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
tiffinWN_new = listToDataframe(changesList, tableWordnet)
tiffinWN_new.to_excel("tiffinWN_Result.xlsx")
tiffinWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tiffinWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Ayam - Cookpad

In [ ]:
!gdown --id "1glKh8i7UA0Q8gyGxJ9nrXCwrC6WoOy6A"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1glKh8i7UA0Q8gyGxJ9nrXCwrC6WoOy6A
To: /content/cookpad-ayam.jl
100% 14.6k/14.6k [00:00<00:00, 21.0MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
ayam_cpd=pd.read_json(r'/content/cookpad-ayam.jl', lines = True)
ayam_cpd = concat_bahan_langkah(ayam_cpd)

# fill NaN with empty string
ayam_cpd["page_description"] = ayam_cpd["page_description"].fillna("")

#save dalam acar biar ngebut loading ntar.
ayam_cpd.to_pickle(r'/content/raw_df.pkl')

# covnert page_description to list
ayam_cpd["temp"] = ayam_cpd["page_description"]
ayam_cpd["page_description"] = ayam_cpd["page_bahan_langkah"]

# # clean text
ayam_cpd = pre_cleaning_text(ayam_cpd, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
ayam_cpd.head()

,url_search,list_scraping_time,list_item_url,list_item_title,list_item_ingredients,list_item_author_name,list_item_author_image_url,list_item_image_url,page_scraping_time,page_image_url,...,page_serving_count,page_bahan,page_langkah,page_post_datetime,page_post_love_count,page_post_clap_count,page_post_tasty_count,page_comment_count,page_bahan_langkah,temp
0,https://cookpad.com/id/cari/ayam?event=search....,2021-12-27 08:03:02,https://cookpad.com/id/resep/15827121-minyak-ayam,Minyak Ayam,"[Kulit ayam dan lemak, bawang putih iris tipis...",Ibu Tina,https://img-global.cpcdn.com/users/257098d9c19...,https://img-global.cpcdn.com/recipes/10a0f7cc2...,2021-12-27 08:03:02,https://img-global.cpcdn.com/recipes/10a0f7cc2...,...,None,"[secukupnya Kulit ayam dan lemak, 3 bawang put...",[],27 Desember 2021 06.40,NaN,NaN,NaN,NaN,"[secukupnya Kulit ayam dan lemak, 3 bawang put...","Sangat enak buat masak nasgor, mie maupun tumi..."
1,https://cookpad.com/id/cari/ayam?event=search....,2021-12-27 08:03:02,https://cookpad.com/id/resep/15827176-pindang-...,Pindang ayam khas palembang,"[ayam, air, laos geprek, serai geprek, cuka, g...",Tayooo,https://img-global.cpcdn.com/users/e30967ee548...,https://img-global.cpcdn.com/recipes/1a10091e7...,2021-12-27 08:03:04,https://img-global.cpcdn.com/recipes/1a10091e7...,...,2 orang,"[4 potong ayam, 400 ml air, 2-3 cm laos geprek...",[],27 Desember 2021 07.15,NaN,NaN,NaN,NaN,"[4 potong ayam, 400 ml air, 2-3 cm laos geprek...",Di rumah cuma ada ayam jadi pindang ayam saja
2,https://cookpad.com/id/cari/ayam?event=search....,2021-12-27 08:03:02,https://cookpad.com/id/resep/15826872-garang-a...,Garang Asem Ayam,"[ayam, Ayam tadi (lumuri kunyit bubuk,, lada b...",kdewi,https://img-global.cpcdn.com/users/e207585a1f1...,https://img-global.cpcdn.com/recipes/db9d432e8...,2021-12-27 08:03:17,https://img-global.cpcdn.com/recipes/db9d432e8...,...,None,"[1/2 kg ayam, Ayam tadi (lumuri kunyit bubuk,,...","[Ayam dimarinasi, diamkan 15 menit, Iris² bawa...",27 Desember 2021 04.04,2.0,1.0,3.0,NaN,"[1/2 kg ayam, Ayam tadi (lumuri kunyit bubuk,,...",
3,https://cookpad.com/id/cari/ayam?event=search....,2021-12-27 08:03:02,https://cookpad.com/id/resep/15826969-dimsum-a...,Dimsum Ayam Nori,"[ayam fillet, minyak wijen, saus tiram, gula, ...",Zarmaliawati,https://img-global.cpcdn.com/users/d8eb929105c...,https://img-global.cpcdn.com/recipes/14b9d1ac3...,2021-12-27 08:03:31,https://img-global.cpcdn.com/recipes/14b9d1ac3...,...,10 gulung nori,[10 lembar nori atau kulit lumpia besar dipoto...,[Giling ayam fillet. Saya gilingnya pke choppe...,27 Desember 2021 05.00,1.0,1.0,2.0,NaN,[10 lembar nori atau kulit lumpia besar dipoto...,Ini resep dari kakak yg jualan dimsum. Jd sdh ...
4,https://cookpad.com/id/cari/ayam?event=search....,2021-12-27 08:03:02,https://cookpad.com/id/resep/15827070-ayam-tep...,Ayam tepung crispy,"[ayam fillet aq pakai dada, kaldu ayam secukup...",Hariyati Zahwa,https://img-global.cpcdn.com/users/5539fe6e33a...,https://img-global.cpcdn.com/recipes/4de272159...,2021-12-27 08:03:40,https://img-global.cpcdn.com/recipes/4de272159...,...,2 orang,"[300 gram ayam fillet aq pakai dada, kaldu aya...",[],27 Desember 2021 06.07,2.0,NaN,NaN,NaN,"[300 gram ayam fillet aq pakai dada, kaldu aya...",


## GUI Ayam - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = ayam_cpd.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.100138 -0.067420       1        1  43.699991
1     -0.106764  0.103148       2        1  27.372145
3      0.142317  0.046896       3        1  22.412263
5      0.036642 -0.060135       4        1   6.409687
6      0.009288 -0.007472       5        1   0.037250
2      0.009314 -0.007495       6        1   0.036320
4      0.009342 -0.007522       7        1   0.032344, topic_info=                            Term       Freq      Total Category  logprob  \
108                         paha   5.000000   5.000000  Default  30.0000   
1                           ayam  17.000000  17.000000  Default  29.0000   
16                         merah   7.000000   7.000000  Default  28.0000   
34                         salam   5.000000   5.000000  Default  27.0000   
23                         bubuk   5.000000   5.000000  Default  26.0000   
3                         bawang  14.000000  14.000000  Default  25.0000   
19                          cabe   4.000000   4.000000  Default  24.0000   
4                          putih   9.000000   9.000000  Default  23.0000   
30                          ruas   4.000000   4.000000  Default  22.0000   
110                       kemiri   3.000000   3.000000  Default  21.0000   
117                       tunggu   3.000000   3.000000  Default  20.0000   
118                     mendidih   3.000000   3.000000  Default  19.0000   
109                         buah   3.000000   3.000000  Default  18.0000   
132                       santan   3.000000   3.000000  Default  17.0000   
71                         telur   5.000000   5.000000  Default  16.0000   
24                          lada   3.000000   3.000000  Default  15.0000   
38                         sereh   3.000000   3.000000  Default  14.0000   
8                            air   7.000000   7.000000  Default  13.0000   
14                         bumbu   4.000000   4.000000  Default  12.0000   
69                      haluskan   3.000000   3.000000  Default  11.0000   
33                          daun   7.000000   7.000000  Default  10.0000   
18                        kunyit   3.000000   3.000000  Default   9.0000   
17                          jahe   3.000000   3.000000  Default   8.0000   
10                        geprek   3.000000   3.000000  Default   7.0000   
84                          aduk   3.000000   3.000000  Default   6.0000   
13                         garam   5.000000   5.000000  Default   5.0000   
26                         siung   5.000000   5.000000  Default   4.0000   
50                         tumis   2.000000   2.000000  Default   3.0000   
11                         serai   2.000000   2.000000  Default   2.0000   
58                     tambahkan   2.000000   2.000000  Default   1.0000   
108                         paha   5.374691   5.678689   Topic1  -3.3282   
132                       santan   2.750633   3.053700   Topic1  -3.9981   
109                         buah   2.750478   3.053655   Topic1  -3.9981   
118                     mendidih   2.750441   3.053628   Topic1  -3.9981   
110                       kemiri   2.750230   3.053528   Topic1  -3.9982   
117                       tunggu   2.749886   3.053456   Topic1  -3.9983   
131                       sachet   1.875441   2.178507   Topic1  -4.3811   
133                       instan   1.875438   2.178504   Topic1  -4.3811   
137                        rebus   1.875435   2.178501   Topic1  -4.3811   
144                      masukan   1.875430   2.178496   Topic1  -4.3811   
111                        jamur   1.874859   2.178311   Topic1  -4.3814   
34                         salam   4.500616   5.618144   Topic1  -3.5057   
139                      merebus   1.000238   1.303303   Topic1  -5.0097   
143                      kecuali   1.000235   1.303300   Topic1  -5.0097   
130                       jinten   1.000235   1.303

Topic: 1
ayam, paha, salam, air, bubuk, bawang, sereh, merah, santan, telur, buah, mendidih, bumbu, kemiri, lada, haluskan, ruas, daun, putih, tunggu, siung, matang, sachet, instan, rebus, masukan, butir, tumis, aduk, kaldu

Topic: 2
ayam, bawang, merah, cabe, putih, air, daun, geprek, garam, jahe, kunyit, bubuk, ruas, lembar, siung, tomat, diamkan, dimarinasi, hijau, tambahkan, serai, tutup, lada, salam, bumbu, batang, gula, tumis, cuka, laos

Topic: 3
merah, bawang, ayam, daun, siung, air, ruas, sereh, cabe, putih, tomat, lembar, bubuk, diamkan, salam, matang, hijau, dimarinasi, santan, lada, telur, tumis, bumbu, gula, butir, gulapenyedap, kunyit, masukan, dll, wuluh

Topic: 4
bawang, putih, ayam, nori, kulit, lembar, aduk, daun, wortel, parut, garam, tiram, saus, gula, fillet, telur, wijen, lemak, tipis, iris, goreng, kukus, tebal, air, sesuaikan, teksturnya, ambil, airnya, giling, robek

Topic: 5
ayam, putih, bawang, bumbu, kulit, merah, goreng, siung, garam, tepung, daun, lemak, a

[(0,
  '0.053*"ayam" + 0.036*"paha" + 0.030*"salam" + 0.024*"air" + 0.024*"bubuk" + 0.024*"bawang" + 0.018*"sereh" + 0.018*"merah" + 0.018*"santan" + 0.018*"telur"'),
 (1,
  '0.062*"ayam" + 0.062*"bawang" + 0.045*"merah" + 0.036*"cabe" + 0.027*"putih" + 0.027*"air" + 0.027*"daun" + 0.019*"geprek" + 0.019*"garam" + 0.019*"jahe"'),
 (2,
  '0.007*"merah" + 0.007*"bawang" + 0.007*"ayam" + 0.007*"daun" + 0.007*"siung" + 0.007*"air" + 0.007*"ruas" + 0.007*"sereh" + 0.007*"cabe" + 0.007*"putih"'),
 (3,
  '0.052*"bawang" + 0.052*"putih" + 0.032*"ayam" + 0.032*"nori" + 0.022*"kulit" + 0.022*"lembar" + 0.022*"aduk" + 0.022*"daun" + 0.022*"wortel" + 0.022*"parut"'),
 (4,
  '0.007*"putih" + 0.007*"ayam" + 0.007*"bawang" + 0.007*"merah" + 0.007*"kulit" + 0.007*"bumbu" + 0.007*"geprek" + 0.007*"daun" + 0.007*"garam" + 0.007*"siung"'),
 (5,
  '0.073*"ayam" + 0.049*"tepung" + 0.026*"putih" + 0.026*"bawang" + 0.026*"telur" + 0.026*"fillet" + 0.026*"siung" + 0.026*"butir" + 0.026*"bumbu" + 0.026*"kaldu"

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
ayamCpdProb = make_dictionary_terms_prob(ldamodel, id_map, d)
ayamCpdProb["ayam"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
ayamCpdWN = dfWordnet.copy()

ayamCpdWN["Synonym_clean"] = cleanWN_Synonym(ayamCpdWN["Synonym"])
ayamCpdWN["Hypernim_clean"] = cleanWN(ayamCpdWN["Hypernim"])
ayamCpdWN["Hyponim_clean"] = cleanWN(ayamCpdWN["Hyponim"])
ayamCpdWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/ayamCookpad.owl")
makeOntology(ontoName, ayamCpdWN)
ontoName.save(file = "ayamCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown "1KOTNxtJLNJjPbSEXb30LF1NX3m3cc54_"

In [ ]:
wiki_ayam_cpd = pd.read_excel("AyamCookpadFinal.xlsx").iloc[:, 1:]

# Translate
wiki_ayam_cpd["Category"] = translate(wiki_ayam_cpd["Category"])
wiki_ayam_cpd.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_ayam_cpd[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
ayamCpdTabel_new = tabel_1981.copy()
ayamCpdTabel_new["correct_category"] = all_cat
ayamCpdTabel_new["suggestion"] = all_suggestion

# Export Result to File
ayamCpdTabel_new.to_excel("ayamCpdTabel_Result.xlsx")
ayamCpdTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
ayamCpdTabel_new = concatListInColumn(ayamCpdTabel_new)
ayamCpdTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, ayamCpdTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_ayam_cpd.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
ayamCpdWiki_new = wiki_df.copy()
ayamCpdWiki_new["suggest"] = all_cat

# Export Result to file
ayamCpdWiki_new.to_excel("ayamCpdWiki_Result.xlsx")
ayamCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
ayamCpdOntoWiki = get_ontology("http://test.org/ayamCpd_onto_wiki.owl")
makeOntology_wiki(ayamCpdOntoWiki, ayamCpdWiki_new)
ayamCpdOntoWiki.save(file = "ayamCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = ayamCpdWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
ayamCpdWN_new = listToDataframe(changesList, tableWordnet)
ayamCpdWN_new.to_excel("ayamCpdWN_Result.xlsx")
ayamCpdWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Ayam Selenium - Cookpad

In [ ]:
!gdown --id "1OTIjGPazGUtKTsZ2jpatPRbvxft-Vp1o"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1OTIjGPazGUtKTsZ2jpatPRbvxft-Vp1o
To: /content/cookpad-selenium_ayam.jl
100% 431k/431k [00:00<00:00, 122MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
ayam_cpd_sln=pd.read_json(r'/content/cookpad-selenium_ayam.jl', lines = True)
ayam_cpd_sln = concat_bahan_langkah(ayam_cpd_sln)

# fill NaN with empty string
ayam_cpd_sln["page_description"] = ayam_cpd_sln["page_description"].fillna("")

#save dalam acar biar ngebut loading ntar.
ayam_cpd_sln.to_pickle(r'/content/raw_df.pkl')

# covnert page_description to list
ayam_cpd_sln["temp"] = ayam_cpd_sln["page_description"]
ayam_cpd_sln["page_description"] = ayam_cpd_sln["page_bahan_langkah"]

# # clean text
ayam_cpd_sln = pre_cleaning_text(ayam_cpd_sln, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
ayam_cpd_sln.head()

,url_search,list_scraping_time,list_item_url,list_item_title,list_item_ingredients,list_item_author_name,list_item_author_image_url,list_item_image_url,page_scraping_time,page_image_url,...,page_serving_count,page_bahan,page_langkah,page_post_datetime,page_post_love_count,page_post_clap_count,page_post_tasty_count,page_comment_count,page_bahan_langkah,temp
0,https://cookpad.com/id/cari/ayam?event=search....,2022-06-24 10:30:16,https://cookpad.com/id/resep/15937128-ayam-ung...,\n Ayam Ungkep Kepayang (Keluwek)\n,"[\n ayam potong menurut selera\n , \n ...",Windu Restina,https://img-global.cpcdn.com/users/0c4f59b4f58...,https://img-global.cpcdn.com/recipes/e980f20aa...,2022-06-24 10:30:19,https://img-global.cpcdn.com/recipes/e980f20aa...,...,None,"[500 gr ayam potong menurut selera, 100 ml air...",[],24 Juni 2022 09.28,7.0,5.0,4.0,NaN,"[500 gr ayam potong menurut selera, 100 ml air...",Selamat buat mba
1,https://cookpad.com/id/cari/ayam?event=search....,2022-06-24 10:30:16,https://cookpad.com/id/resep/16326158-nasi-tim...,\n Nasi Tim ayam\n,"[\n Dada ayam fillet\n , \n wortel\...",Elviana,https://img-global.cpcdn.com/users/d7c9e5004f6...,https://img-global.cpcdn.com/recipes/905b923a3...,2022-06-24 10:30:22,https://img-global.cpcdn.com/recipes/905b923a3...,...,4 orang,"[1 Dada ayam fillet, 2 buah wortel, 300 gram b...",[],24 Juni 2022 09.11,1.0,NaN,NaN,NaN,"[1 Dada ayam fillet, 2 buah wortel, 300 gram b...",
2,https://cookpad.com/id/cari/ayam?event=search....,2022-06-24 10:30:16,https://cookpad.com/id/resep/16326162-chicken-...,\n Chicken wings saus barbeque\n,"[\n sayap ayam\n , \n Bawang putih\...",Feriana Feriana,https://img-global.cpcdn.com/users/efde4575a8b...,https://img-global.cpcdn.com/recipes/524f4bca7...,2022-06-24 10:30:25,https://img-global.cpcdn.com/recipes/524f4bca7...,...,4 orang,"[7 potong sayap ayam, 4 siung Bawang putih, Ja...",[],24 Juni 2022 09.11,1.0,NaN,NaN,NaN,"[7 potong sayap ayam, 4 siung Bawang putih, Ja...","Lagi pengen masak ayam ,tapi bosen kalau cuman..."
3,https://cookpad.com/id/cari/ayam?event=search....,2022-06-24 10:30:16,https://cookpad.com/id/resep/16326002-ayam-gor...,\n Ayam Goreng Oat\n,"[\n Marinasi:\n , \n Ayam dada/fill...",LinaS_Cuisine,https://img-global.cpcdn.com/users/95a77d1da4d...,https://img-global.cpcdn.com/recipes/0cd304549...,2022-06-24 10:30:27,https://img-global.cpcdn.com/recipes/0cd304549...,...,None,"[Marinasi:, 400 gr Ayam dada/fillet potong cub...",[],24 Juni 2022 07.48,NaN,NaN,2.0,NaN,"[Marinasi:, 400 gr Ayam dada/fillet potong cub...",Terinspirasi dari udang goreng crispy oat (per...
4,https://cookpad.com/id/cari/ayam?event=search....,2022-06-24 10:30:16,https://cookpad.com/id/resep/16326112-ayam-gor...,\n Ayam Goreng Mentega\n,"[\n Marinasi ayam :\n , \n paha aya...",Putri Larasati,https://img-global.cpcdn.com/users/dc5af1b724d...,https://img-global.cpcdn.com/recipes/a5b99a24c...,2022-06-24 10:30:50,https://img-global.cpcdn.com/recipes/a5b99a24c...,...,None,"[Marinasi ayam :, 4 potong paha ayam, 1 buah j...",[],24 Juni 2022 08.43,NaN,NaN,NaN,NaN,"[Marinasi ayam :, 4 potong paha ayam, 1 buah j...","masak praktis dan bergizi, tentu saja gurih se..."


## GUI Ayam Selenium - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = ayam_cpd_sln.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
6     -0.206636 -0.036072       1        1  39.107383
1     -0.250333 -0.041670       2        1  34.102830
5     -0.135254  0.106983       3        1  14.591090
0      0.120914 -0.245434       4        1   5.097039
4      0.046852  0.118757       5        1   3.736746
2      0.217787  0.050384       6        1   2.145580
3      0.206668  0.047053       7        1   1.219332, topic_info=                Term        Freq       Total Category  logprob  loglift
16              buah  234.000000  234.000000  Default  30.0000  30.0000
73            tepung   66.000000   66.000000  Default  29.0000  29.0000
129          cincang   57.000000   57.000000  Default  28.0000  28.0000
102             ruas  104.000000  104.000000  Default  27.0000  27.0000
5              halus   96.000000   96.000000  Default  26.0000  26.0000
7             bawang  419.000000  419.000000  Default  25.0000  25.0000
28              daun  200.000000  200.000000  Default  24.0000  24.0000
56             kecap  103.000000  103.000000  Default  23.0000  23.0000
8              merah  172.000000  172.000000  Default  22.0000  22.0000
49              gram   63.000000   63.000000  Default  21.0000  21.0000
68              saus   90.000000   90.000000  Default  20.0000  20.0000
27            lembar   83.000000   83.000000  Default  19.0000  19.0000
42              cabe   94.000000   94.000000  Default  18.0000  18.0000
84             telur   49.000000   49.000000  Default  17.0000  17.0000
30             jeruk   97.000000   97.000000  Default  16.0000  16.0000
18            kunyit   70.000000   70.000000  Default  15.0000  15.0000
59             bubuk  174.000000  174.000000  Default  14.0000  14.0000
61             garam  198.000000  198.000000  Default  13.0000  13.0000
1             selera   37.000000   37.000000  Default  12.0000  12.0000
29             salam   60.000000   60.000000  Default  11.0000  11.0000
123           sesuai   35.000000   35.000000  Default  10.0000  10.0000
48            wortel   28.000000   28.000000  Default   9.0000   9.0000
6              butir   78.000000   78.000000  Default   8.0000   8.0000
0               ayam  281.000000  281.000000  Default   7.0000   7.0000
13              jahe   78.000000   78.000000  Default   6.0000   6.0000
66             tomat   53.000000   53.000000  Default   5.0000   5.0000
43             rawit   48.000000   48.000000  Default   4.0000   4.0000
2                air  132.000000  132.000000  Default   3.0000   3.0000
58            merica   40.000000   40.000000  Default   2.0000   2.0000
196             keju   14.000000   14.000000  Default   1.0000   1.0000
56             kecap  102.488798  103.119764   Topic1  -3.4016   0.9327
68              saus   90.219432   90.898907   Topic1  -3.5291   0.9314
55             tiram   51.136342   51.766784   Topic1  -4.0968   0.9266
47            fillet   40.008379   40.639522   Topic1  -4.3422   0.9232
54              saos   37.995999   38.626270   Topic1  -4.3938   0.9224
105           sachet   36.546380   37.177336   Topic1  -4.4327   0.9217
78              dadu   27.614290   28.246329   Topic1  -4.7130   0.9162
127              btr   25.662882   26.299083   Topic1  -4.7863   0.9144
60              asin   23.601619   24.232175   Topic1  -4.8700   0.9125
141            pakai   19.006168   19.645663   Topic1  -5.0866   0.9058
85           maizena   16.331356   16.962094   Topic1  -5.2382   0.9010
248         haluskan   12.057155   12.689868   Topic1  -5.5417   0.8877
82           inggris   11.379563   12.009869   Topic1  -5.5995   0.8849
172          mentega   10.765039   11.395431   Topic1  -5.6550   0.8820
107            pedas    9.976707   10.610127   Topic1  -5.7311   0.8773
106            saori    8.580357    9.211273   Topic1  -5.8818   0.8679
161        serbaguna    8.409101    9.040892   Topic1  -5.9020   0.8664
529           b

Topic: 1
tepung, keju, telur, lihat, cair, resep, instan, ayam, giling, butir, sisa, kuning, chicken, bks, jeruk, powder, air, opsional, terigu, nipis, menggoreng, parut, baking, topping, pelengkap, sambal, sagu, daun, saledri, bubur

Topic: 2
bawang, daun, ruas, merah, siung, ayam, jeruk, putih, garam, bumbu, lembar, kunyit, jahe, salam, kemiri, cabe, bubuk, goreng, air, batang, butir, halus, gula, buah, kaldu, lengkuas, ketumbar, memarkan, serai, sangrai

Topic: 3
kkal, pcs, bolognese, sckupnya, digeprek, bihun, terasi, soto, rosemary, kerupuk, potongpotong, daun, garamkaldu, chillies, ingredients, leaf, khas, champignon, aslinya, herb, pepper, black, pns, seduh, segar, ulek, dipotong, ogut, makenya, pkenya

Topic: 4
pasta, mix, cheddar, spaghetti, cup, sayur, peres, tambahan, cheese, tsp, suwir, centong, lee, kum, kee, abc, uht, sejumput, dirajang, paste, alfredo, italian, penne, heavy, ubi, perasa, karbo, orak, bumtik, prona

Topic: 5
halus, cincang, buah, gram, bawang, wortel, gar

[(0,
  '0.048*"tepung" + 0.034*"keju" + 0.026*"telur" + 0.024*"lihat" + 0.024*"cair" + 0.023*"resep" + 0.022*"instan" + 0.021*"ayam" + 0.019*"giling" + 0.018*"butir"'),
 (1,
  '0.059*"bawang" + 0.057*"daun" + 0.039*"ruas" + 0.037*"merah" + 0.036*"siung" + 0.035*"ayam" + 0.029*"jeruk" + 0.028*"putih" + 0.028*"garam" + 0.028*"bumbu"'),
 (2,
  '0.056*"kkal" + 0.032*"pcs" + 0.023*"bolognese" + 0.017*"sckupnya" + 0.012*"digeprek" + 0.012*"bihun" + 0.012*"terasi" + 0.012*"soto" + 0.010*"rosemary" + 0.009*"kerupuk"'),
 (3,
  '0.055*"pasta" + 0.034*"mix" + 0.028*"cheddar" + 0.024*"spaghetti" + 0.012*"cup" + 0.011*"sayur" + 0.011*"peres" + 0.010*"tambahan" + 0.008*"cheese" + 0.008*"tsp"'),
 (4,
  '0.033*"halus" + 0.031*"cincang" + 0.029*"buah" + 0.026*"gram" + 0.025*"bawang" + 0.025*"wortel" + 0.024*"garam" + 0.024*"susu" + 0.020*"ayam" + 0.019*"nasi"'),
 (5,
  '0.112*"buah" + 0.063*"bawang" + 0.041*"merah" + 0.032*"cabe" + 0.028*"siung" + 0.027*"selera" + 0.027*"daun" + 0.026*"sesuai" + 0.024*

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
ayamSlnCpdProb = make_dictionary_terms_prob(ldamodel, id_map, d)
ayamSlnCpdProb["mee"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
ayamCpdSlnWN = dfWordnet.copy()

ayamCpdSlnWN["Synonym_clean"] = cleanWN_Synonym(ayamCpdSlnWN["Synonym"])
ayamCpdSlnWN["Hypernim_clean"] = cleanWN(ayamCpdSlnWN["Hypernim"])
ayamCpdSlnWN["Hyponim_clean"] = cleanWN(ayamCpdSlnWN["Hyponim"])
ayamCpdSlnWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/ayamSeleniumCookpad.owl")
makeOntology(ontoName, ayamCpdSlnWN)
ontoName.save(file = "ayamSeleniumCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1DgiR9ABRAki_Ut9SBfnCGwzquuhDCjEB"

In [ ]:
wiki_ayam_sln = pd.read_excel("AyamSlnCookpadFinal.xlsx").iloc[:, 1:]

# Translate
wiki_ayam_sln["Category"] = translate(wiki_ayam_sln["Category"])
wiki_ayam_sln.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_ayam_sln[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
ayamSlnTabel_new = tabel_1981.copy()
ayamSlnTabel_new["correct_category"] = all_cat
ayamSlnTabel_new["suggestion"] = all_suggestion

# Export Result to File
ayamSlnTabel_new.to_excel("ayamSlnTabel_Result.xlsx")
ayamSlnTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdSlnTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
ayamSlnTabel_new = concatListInColumn(ayamSlnTabel_new)
ayamSlnTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, ayamSlnTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_ayam_sln.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
ayamCpdSlnWiki_new = wiki_df.copy()
ayamCpdSlnWiki_new["suggest"] = all_cat

# Export Result to file
ayamCpdSlnWiki_new.to_excel("ayamCpdSlnWiki_Result.xlsx")
ayamCpdSlnWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdSlnWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
ayamCpdSlnOntoWiki = get_ontology("http://test.org/ayamCpdSln_onto_wiki.owl")
makeOntology_wiki(ayamCpdSlnOntoWiki, ayamCpdSlnWiki_new)
ayamCpdSlnOntoWiki.save(file = "ayamCpdSln_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = ayamCpdSlnWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
ayamCpdSlnWN_new = listToDataframe(changesList, tableWordnet)
ayamCpdSlnWN_new.to_excel("ayamCpdSlnWN_Result.xlsx")
ayamCpdSlnWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ayamCpdSlnWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Ikan - Cookpad

In [ ]:
!gdown --id "1xMuIl-GL3PIlXyyzhnW_SrJ35RNmlRYT"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1xMuIl-GL3PIlXyyzhnW_SrJ35RNmlRYT
To: /content/cookpad-selenium_ikan.jl
100% 406k/406k [00:00<00:00, 105MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
ikan_cpd=pd.read_json(r'/content/cookpad-selenium_ikan.jl', lines = True)
ikan_cpd = concat_bahan_langkah(ikan_cpd)

# fill NaN with empty string
ikan_cpd["page_description"] = ikan_cpd["page_description"].fillna("")

#save dalam acar biar ngebut loading ntar.
ikan_cpd.to_pickle(r'/content/raw_df.pkl')

# covnert page_description to list
ikan_cpd["temp"] = ikan_cpd["page_description"]
ikan_cpd["page_description"] = ikan_cpd["page_bahan_langkah"]

# # clean text
ikan_cpd = pre_cleaning_text(ikan_cpd, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
ikan_cpd.head()

,url_search,list_scraping_time,list_item_url,list_item_title,list_item_ingredients,list_item_author_name,list_item_author_image_url,list_item_image_url,page_scraping_time,page_image_url,...,page_serving_count,page_bahan,page_langkah,page_post_datetime,page_post_love_count,page_post_clap_count,page_post_tasty_count,page_comment_count,page_bahan_langkah,temp
0,https://cookpad.com/id/cari/ikan?event=search....,2022-06-24 21:59:20,https://cookpad.com/id/resep/16304096-steam-pa...,\n Steam Patin/Dori Saus Hongkong\n,"[fillet ikan patin/dori, jahe, iris korek, jer...",Elia Wahid,https://img-global.cpcdn.com/users/09905b79e3f...,https://img-global.cpcdn.com/recipes/19f92026f...,2022-06-24 21:59:22,https://img-global.cpcdn.com/recipes/19f92026f...,...,1-2 orang,"[Steam Ikan, 330 gram fillet ikan patin/dori, ...",[],24 Juni 2022 21.39,NaN,NaN,NaN,NaN,"[Steam Ikan, 330 gram fillet ikan patin/dori, ...",Ikan dori fillet yang saya dapati di penjual i...
1,https://cookpad.com/id/cari/ikan?event=search....,2022-06-24 21:59:20,https://cookpad.com/id/resep/16318606-salmon-m...,\n Salmon Mentai Rice\n,"[nasi putih, minyak wijen, serpihan nori, butt...",Iftitah Kurniasari,https://img-global.cpcdn.com/users/e6da63fd390...,https://img-global.cpcdn.com/recipes/8beb5f361...,2022-06-24 21:59:25,https://img-global.cpcdn.com/recipes/8beb5f361...,...,5 cup kecil,"[Nasi :, 300 gr nasi putih, 1/2 sdm minyak wij...",[],24 Juni 2022 16.48,2.0,NaN,NaN,NaN,"[Nasi :, 300 gr nasi putih, 1/2 sdm minyak wij...",Ibu ibu biasanya pusing anaknya belum makan na...
2,https://cookpad.com/id/cari/ikan?event=search....,2022-06-24 21:59:20,https://cookpad.com/id/resep/16084408-382-onig...,\n 382. 🍙 Onigiri Tuna mayo\n,"[nasi, mirin (me: Skip), minyak wijen, tuna, m...",Intan Zahra Al Arsyad (Bunda'Reycha),https://img-global.cpcdn.com/users/51213d3b636...,https://img-global.cpcdn.com/recipes/11f187f70...,2022-06-24 21:59:48,https://img-global.cpcdn.com/recipes/11f187f70...,...,None,"[3-4 piring nasi, 3 Sdm mirin (me: Skip), 1 sd...",[],24 Juni 2022 16.55,1.0,1.0,1.0,NaN,"[3-4 piring nasi, 3 Sdm mirin (me: Skip), 1 sd...",Bismillah.. \n
3,https://cookpad.com/id/cari/ikan?event=search....,2022-06-24 21:59:20,https://cookpad.com/id/resep/15881149-orak-ari...,\n Orak arik Tongkol Pedas\n,"[pindang tongkol suwir2, air, bawang merah, ba...",abuk irun,https://img-global.cpcdn.com/users/5e66d2681bd...,https://img-global.cpcdn.com/recipes/c15b57a71...,2022-06-24 21:59:56,https://img-global.cpcdn.com/recipes/c15b57a71...,...,None,"[4 ekor pindang tongkol suwir2, 100 ml air, Bu...",[],24 Juni 2022 16.03,4.0,4.0,1.0,NaN,"[4 ekor pindang tongkol suwir2, 100 ml air, Bu...",Bismillah..
4,https://cookpad.com/id/cari/ikan?event=search....,2022-06-24 21:59:20,https://cookpad.com/id/resep/16327000-pesmol-i...,\n Pesmol Ikan Nila\n,"[Ikan nila, Jeruk nipis, bawang merah, bawang ...",Rika Veronica,https://img-global.cpcdn.com/users/13a5c22dd28...,https://img-global.cpcdn.com/recipes/df50848d6...,2022-06-24 22:00:31,https://img-global.cpcdn.com/recipes/df50848d6...,...,None,"[Ikan nila, Jeruk nipis, Bumbu halus:, 8 siung...",[],24 Juni 2022 16.25,NaN,NaN,NaN,NaN,"[Ikan nila, Jeruk nipis, Bumbu halus:, 8 siung...",


## GUI Ikan - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = ikan_cpd.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5      0.293113  0.072557       1        1  58.380772
4      0.203877 -0.060639       2        1  18.874425
6     -0.100290 -0.172096       3        1  10.324474
1      0.108877 -0.010352       4        1   5.104438
0     -0.156954  0.269474       5        1   3.121851
3     -0.175202 -0.055601       6        1   2.835980
2     -0.173422 -0.043344       7        1   1.358061, topic_info=                    Term        Freq       Total Category  logprob  loglift
8                   buah  267.000000  267.000000  Default  30.0000  30.0000
65                  cabe  138.000000  138.000000  Default  29.0000  29.0000
30                  daun  194.000000  194.000000  Default  28.0000  28.0000
1                   ikan  146.000000  146.000000  Default  27.0000  27.0000
28                 merah  245.000000  245.000000  Default  26.0000  26.0000
66                 rawit   75.000000   75.000000  Default  25.0000  25.0000
6                   iris   58.000000   58.000000  Default  24.0000  24.0000
97                selera   43.000000   43.000000  Default  23.0000  23.0000
138               tepung   51.000000   51.000000  Default  22.0000  22.0000
17                   air   99.000000   99.000000  Default  21.0000  21.0000
168                makan   34.000000   34.000000  Default  20.0000  20.0000
14                bawang  302.000000  302.000000  Default  19.0000  19.0000
507              sckpnya   33.000000   33.000000  Default  18.0000  18.0000
96                sesuai   28.000000   28.000000  Default  17.0000  17.0000
41                 bubuk   93.000000   93.000000  Default  16.0000  16.0000
150                 ikat   24.000000   24.000000  Default  15.0000  15.0000
39                 garam  164.000000  164.000000  Default  14.0000  14.0000
67              keriting   45.000000   45.000000  Default  13.0000  13.0000
118              kemangi   28.000000   28.000000  Default  12.0000  12.0000
125               merica   36.000000   36.000000  Default  11.0000  11.0000
13                 siung  182.000000  182.000000  Default  10.0000  10.0000
69                 resep   15.000000   15.000000  Default   9.0000   9.0000
88                lembar   84.000000   84.000000  Default   8.0000   8.0000
68                 lihat   15.000000   15.000000  Default   7.0000   7.0000
74                  ruas  127.000000  127.000000  Default   6.0000   6.0000
31                 tipis   24.000000   24.000000  Default   5.0000   5.0000
44                 tomat   83.000000   83.000000  Default   4.0000   4.0000
10                 nipis   51.000000   51.000000  Default   3.0000   3.0000
201               ukuran   47.000000   47.000000  Default   2.0000   2.0000
71                 halus   62.000000   62.000000  Default   1.0000   1.0000
74                  ruas  126.481357  127.072551   Topic1  -3.4386   0.5335
89                 salam   74.027128   74.615728   Topic1  -3.9742   0.5303
27                 cabai   61.613955   62.203368   Topic1  -4.1578   0.5287
92              lengkuas   60.326922   60.914607   Topic1  -4.1789   0.5285
73                kemiri   52.199856   52.787494   Topic1  -4.3236   0.5270
79                 serai   39.959188   40.547203   Topic1  -4.5908   0.5236
206               sendok   36.108407   36.705368   Topic1  -4.6921   0.5218
87                geprek   30.272591   30.860046   Topic1  -4.8684   0.5190
76                kunyit   72.008895   73.485190   Topic1  -4.0019   0.5179
313             cakalang   27.888324   28.478537   Topic1  -4.9505   0.5172
62               tongkol   25.464634   26.052503   Topic1  -5.0414   0.5154
30                  daun  189.441246  194.086153   Topic1  -3.0346   0.5140
372                 utuh   22.639564   23.227207   Topic1  -5.1590   0.5126
80                   lbr   22.293438   22.880710   Topic1  -5.1744   0.5122
116                sereh   21.079466   21.666739   

Topic: 1
resep, lihat, ikat, melintang, rajang, kemangi, mas, daun, genggam, gelas, ambil, perasan, rebus, memarkan, dagingnya, petik, sobek, serei, kosong, penuh, bimoli, unt, besarbuang, skipbisa, lidi, bongggol, tali, bijinyairis, segepok, membungkus

Topic: 2
buah, cabe, ikan, sesuai, iris, selera, irisiris, kembung, rawit, utk, dadu, sachet, tomat, keriting, ukuran, siung, halus, marinasi, merah, garam, sate, nipis, sarden, jeruk, kaleng, bawang, cuka, terasi, menggoreng, bamer

Topic: 3
kakap, sambal, wuluh, serbaguna, bombay, belimbing, extra, ditambahdikurangi, mas, telor, kepala, tumis, jenis, dgn, korek, wader, matahari, bunga, himalaya, seraigeprek, sisiknya, kandis, lmbr, grm, dikurangisecukupnya, gereh, talang, susai, kakdu, teriyaki

Topic: 4
serut, kecap, manis, tujuan, kaleng, bawal, margarin, sesukanya, asin, menyesuaikan, lemon, oil, merk, gram, basic, olive, madu, sambal, sarden, timun, irisan, blackpepper, bombay, ditambah, selada, kubis, ceplok, saori, bebas, spagh

[(0,
  '0.070*"resep" + 0.069*"lihat" + 0.034*"ikat" + 0.027*"melintang" + 0.027*"rajang" + 0.020*"kemangi" + 0.020*"mas" + 0.019*"daun" + 0.019*"genggam" + 0.019*"gelas"'),
 (1,
  '0.074*"buah" + 0.061*"cabe" + 0.043*"ikan" + 0.042*"sesuai" + 0.042*"iris" + 0.038*"selera" + 0.030*"irisiris" + 0.027*"kembung" + 0.024*"rawit" + 0.023*"utk"'),
 (2,
  '0.065*"kakap" + 0.063*"sambal" + 0.052*"wuluh" + 0.036*"serbaguna" + 0.025*"bombay" + 0.020*"belimbing" + 0.011*"extra" + 0.008*"ditambahdikurangi" + 0.007*"mas" + 0.007*"telor"'),
 (3,
  '0.033*"serut" + 0.032*"kecap" + 0.024*"manis" + 0.018*"tujuan" + 0.015*"kaleng" + 0.014*"bawal" + 0.014*"margarin" + 0.012*"sesukanya" + 0.012*"asin" + 0.012*"menyesuaikan"'),
 (4,
  '0.053*"bawang" + 0.044*"air" + 0.041*"bubuk" + 0.040*"garam" + 0.040*"tepung" + 0.034*"putih" + 0.030*"ikan" + 0.028*"merica" + 0.028*"buah" + 0.020*"nipis"'),
 (5,
  '0.060*"merah" + 0.059*"bawang" + 0.052*"buah" + 0.048*"daun" + 0.039*"siung" + 0.032*"ruas" + 0.030*"putih"

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
ikanCpdProb = make_dictionary_terms_prob(ldamodel, id_map, d)
ikanCpdProb["mee"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
ikanCpdWN = dfWordnet.copy()

ikanCpdWN["Synonym_clean"] = cleanWN_Synonym(ikanCpdWN["Synonym"])
ikanCpdWN["Hypernim_clean"] = cleanWN(ikanCpdWN["Hypernim"])
ikanCpdWN["Hyponim_clean"] = cleanWN(ikanCpdWN["Hyponim"])
ikanCpdWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/ikanCookpad.owl")
makeOntology(ontoName, ikanCpdWN)
ontoName.save(file = "ikanCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1Ar_A-vPp8TWorA1BsEQjWR8cf1RE6GI4"

In [ ]:
wiki_ikan_cpd = pd.read_excel("IkanCookpadFinal.xlsx").iloc[:, 1:]

# Translate
wiki_ikan_cpd["Category"] = translate(wiki_ikan_cpd["Category"])
wiki_ikan_cpd.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_ikan_cpd[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
ikanCpdTabel_new = tabel_1981.copy()
ikanCpdTabel_new["correct_category"] = all_cat
ikanCpdTabel_new["suggestion"] = all_suggestion

# Export Result to File
ikanCpdTabel_new.to_excel("ikanCpdTabel_Result.xlsx")
ikanCpdTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanCpdTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
ikanCpdTabel_new = concatListInColumn(ikanCpdTabel_new)
ikanCpdTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, ikanCpdTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_ikan_cpd.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
ikanCpdWiki_new = wiki_df.copy()
ikanCpdWiki_new["suggest"] = all_cat

# Export Result to file
ikanCpdWiki_new.to_excel("ikanCpdWiki_Result.xlsx")
ikanCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
ikanCpdOntoWiki = get_ontology("http://test.org/ikanCpd_onto_wiki.owl")
makeOntology_wiki(ikanCpdOntoWiki, ikanCpdWiki_new)
ikanCpdOntoWiki.save(file = "ikanCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = ikanCpdWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
ikanCpdWN_new = listToDataframe(changesList, tableWordnet)
ikanCpdWN_new.to_excel("ikanCpdWN_Result.xlsx")
ikanCpdWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "ikanCpdWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Kambing - Cookpad

In [ ]:
!gdown --id "1ccVQ68WN2fmGRvPO8_3JUhRNZ6Uy3cie"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1ccVQ68WN2fmGRvPO8_3JUhRNZ6Uy3cie
To: /content/cookpad-selenium_kambing.jl
100% 434k/434k [00:00<00:00, 50.4MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
kambing_cpd=pd.read_json(r'/content/cookpad-selenium_kambing.jl', lines = True)
kambing_cpd = concat_bahan_langkah(kambing_cpd)

# fill NaN with empty string
kambing_cpd["page_description"] = kambing_cpd["page_description"].fillna("")

#save dalam acar biar ngebut loading ntar.
kambing_cpd.to_pickle(r'/content/raw_df.pkl')

# covnert page_description to list
kambing_cpd["temp"] = kambing_cpd["page_description"]
kambing_cpd["page_description"] = kambing_cpd["page_bahan_langkah"]

# # clean text
kambing_cpd = pre_cleaning_text(kambing_cpd, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
kambing_cpd.head()

,url_search,list_scraping_time,list_item_url,list_item_title,list_item_ingredients,list_item_author_name,list_item_author_image_url,list_item_image_url,page_scraping_time,page_image_url,...,page_serving_count,page_bahan,page_langkah,page_post_datetime,page_post_love_count,page_post_clap_count,page_post_tasty_count,page_comment_count,page_bahan_langkah,temp
0,https://cookpad.com/id/cari/kambing?event=sear...,2022-06-24 22:34:41,https://cookpad.com/id/resep/16326243-nasi-bri...,\n Nasi Briyani Daging Kambing\n,"[daging kambing dg tulang, Bumbu marinasi dagi...",Um Syara,https://img-global.cpcdn.com/users/ae4b45294af...,https://img-global.cpcdn.com/recipes/f8cf1e21b...,2022-06-24 22:35:00,https://img-global.cpcdn.com/recipes/f8cf1e21b...,...,None,"[20 potong daging kambing dg tulang, Bumbu mar...",[],24 Juni 2022 12.20,3.0,3.0,2.0,NaN,"[20 potong daging kambing dg tulang, Bumbu mar...",
1,https://cookpad.com/id/cari/kambing?event=sear...,2022-06-24 22:34:41,https://cookpad.com/id/resep/16325562-sup-kaki...,\n Sup kaki kambing seger mantap\n,"[Kaki Kambing yang sudah dibakar dan bersih, W...",Ana,https://img-global.cpcdn.com/users/6d0ca1474cc...,https://img-global.cpcdn.com/recipes/2008d4583...,2022-06-24 22:35:17,https://img-global.cpcdn.com/recipes/2008d4583...,...,8 porsi,[1 kg Kaki Kambing yang sudah dibakar dan bers...,[],24 Juni 2022 02.49,6.0,5.0,5.0,NaN,[1 kg Kaki Kambing yang sudah dibakar dan bers...,Berawal dari si bocil yg sudah bosan dg ikan d...
2,https://cookpad.com/id/cari/kambing?event=sear...,2022-06-24 22:34:41,https://cookpad.com/id/resep/15311012-krengsen...,\n Krengsengan Daging Kambing🧆\n,[daging kambing (rebus dengan daun salam dan s...,Anggela Ayik,https://img-global.cpcdn.com/users/90e1df0754a...,https://img-global.cpcdn.com/recipes/5247f8800...,2022-06-24 22:35:32,https://img-global.cpcdn.com/recipes/5247f8800...,...,None,[500 gr daging kambing (rebus dengan daun sala...,[],22 Juni 2022 07.04,8.0,3.0,1.0,NaN,[500 gr daging kambing (rebus dengan daun sala...,"Ini di masak sampai bener2 air menyusut yak, k..."
3,https://cookpad.com/id/cari/kambing?event=sear...,2022-06-24 22:34:41,https://cookpad.com/id/resep/13311878-nasi-keb...,\n Nasi kebuli kambing homemade\n,"[daging kambing, Santan 700ml (saya pake fiber...",Wahida _BundaMufida&Royyan,https://img-global.cpcdn.com/users/37b18e865a6...,https://img-global.cpcdn.com/recipes/ea4a16b1d...,2022-06-24 22:35:54,https://img-global.cpcdn.com/recipes/ea4a16b1d...,...,10 porsi,"[1/2 kg daging kambing, Santan 700ml (saya pak...",[],22 Juni 2022 12.57,4.0,NaN,1.0,NaN,"[1/2 kg daging kambing, Santan 700ml (saya pak...","Anak2 paling suka yang namanya nasi kebuli, na..."
4,https://cookpad.com/id/cari/kambing?event=sear...,2022-06-24 22:34:41,https://cookpad.com/id/resep/16311977-sup-kamb...,\n Sup Kambing + Sambal Kecap\n,"[kambing (daging, tulang, hati, lemak), kentan...",Fadilla Hanum,https://img-global.cpcdn.com/users/f0323be2404...,https://img-global.cpcdn.com/recipes/e9b8a602d...,2022-06-24 22:35:58,https://img-global.cpcdn.com/recipes/e9b8a602d...,...,3-4 orang,"[1/4 kg kambing (daging, tulang, hati, lemak),...",[],18 Juni 2022 07.03,4.0,2.0,1.0,NaN,"[1/4 kg kambing (daging, tulang, hati, lemak),...",Seorang mahasiswi yg sedang belajar memasak su...


## GUI Kambing - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = kambing_cpd.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.283459 -0.024202       1        1  80.907188
4     -0.110744  0.067709       2        1   6.027193
1      0.062374  0.259563       3        1   5.369491
2      0.169723 -0.099032       4        1   4.361796
5     -0.061617 -0.181928       5        1   1.888576
6     -0.183758 -0.009935       6        1   0.767004
0     -0.159437 -0.012175       7        1   0.678753, topic_info=                     Term        Freq       Total Category  logprob  loglift
30                   daun  325.000000  325.000000  Default  30.0000  30.0000
81                   ruas  164.000000  164.000000  Default  29.0000  29.0000
34                 bawang  484.000000  484.000000  Default  28.0000  28.0000
29                 lembar  176.000000  176.000000  Default  27.0000  27.0000
27                 batang  143.000000  143.000000  Default  26.0000  26.0000
31                  salam  130.000000  130.000000  Default  25.0000  25.0000
60                   jari   69.000000   69.000000  Default  24.0000  24.0000
207                geprek   81.000000   81.000000  Default  23.0000  23.0000
1                 kambing  218.000000  218.000000  Default  22.0000  22.0000
122                 sereh   58.000000   58.000000  Default  21.0000  21.0000
25                  butir  133.000000  133.000000  Default  20.0000  20.0000
23                   biji   60.000000   60.000000  Default  19.0000  19.0000
38                  putih  219.000000  219.000000  Default  18.0000  18.0000
7                   garam  193.000000  193.000000  Default  17.0000  17.0000
78                  halus  122.000000  122.000000  Default  16.0000  16.0000
39                   jahe  125.000000  125.000000  Default  15.0000  15.0000
151              lengkuas   66.000000   66.000000  Default  14.0000  14.0000
3                   bumbu  224.000000  224.000000  Default  13.0000  13.0000
59                  manis  140.000000  140.000000  Default  12.0000  12.0000
58                   kayu   65.000000   65.000000  Default  11.0000  11.0000
24               kapulaga   55.000000   55.000000  Default  10.0000  10.0000
26                cengkeh   56.000000   56.000000  Default   9.0000   9.0000
18                 kunyit  108.000000  108.000000  Default   8.0000   8.0000
130                sendok   31.000000   31.000000  Default   7.0000   7.0000
82                 santan   67.000000   67.000000  Default   6.0000   6.0000
16                   lada   57.000000   57.000000  Default   5.0000   5.0000
32               penyedap   46.000000   46.000000  Default   4.0000   4.0000
55                   pala   60.000000   60.000000  Default   3.0000   3.0000
57                  jeruk  122.000000  122.000000  Default   2.0000   2.0000
15               ketumbar  107.000000  107.000000  Default   1.0000   1.0000
79                  siung  271.404294  271.886168   Topic1  -3.3124   0.2101
35                  merah  332.792828  333.579272   Topic1  -3.1084   0.2095
0                  daging  190.429241  190.914256   Topic1  -3.6667   0.2093
75                   gula  136.311371  136.791228   Topic1  -4.0010   0.2084
69                  kecap  103.494577  103.974637   Topic1  -4.2764   0.2072
54                 merica   81.512244   81.992672   Topic1  -4.5152   0.2060
63                  kaldu   74.060657   74.555535   Topic1  -4.6111   0.2052
80                 kemiri   69.697402   70.177471   Topic1  -4.6718   0.2050
51                   iris   65.838338   66.386539   Topic1  -4.7288   0.2036
33                   buah  302.770590  305.377739   Topic1  -3.2030   0.2033
112                 cabai   49.002955   49.482473   Topic1  -5.0241   0.2021
105                   kol   44.104160   44.583965   Topic1  -5.1294   0.2010
407              irisiris   38.900205   39.379407   Topic1  -5.2550   0.1996
124                 pasir   34.827300   35.308343   Topic1  -5.3656   0.1981
321   

Topic: 1
pack, pre, tdk, ujung, kencur, digeprek, steak, slice, grill, master, memanggang, kuping, kapri, serut, ikan, jahesecukupnya, basmatibumbu, memerk, almadina, minyakmargarin, kobis, disesuaikan, laoslengkuas, kepedasannya, halusulek, presto, kalo, gajih, jantung, pisang

Topic: 2
biji, beras, kapulaga, kayu, cengkeh, bawang, kambing, cincang, hijau, garam, bumbu, lada, india, bombay, manis, ambil, bay, putihnya, mentimun, koja, nasi, basmati, potongpotong, nanas, acar, lemon, margarin, kari, goreng, blender

Topic: 3
biji, daun, batang, lembar, bawang, ruas, butiran, jari, lengkuas, salam, putih, halus, kambing, jawa, kriting, jinten, buncis, sayuran, tambahan, pala, sangrai, seledri, kunyit, mrica, seruas, setan, butir, ketumbar, jeruk, hijau

Topic: 4
bawang, merah, buah, daun, siung, putih, bumbu, bubuk, kambing, daging, garam, lembar, ruas, cabe, gula, manis, batang, jahe, jeruk, halus, salam, butir, air, kecap, kunyit, ketumbar, rawit, tomat, merica, geprek

Topic: 5
sendo

[(0,
  '0.052*"pack" + 0.052*"pre" + 0.010*"ujung" + 0.010*"tdk" + 0.010*"kencur" + 0.007*"digeprek" + 0.005*"memanggang" + 0.005*"grill" + 0.005*"slice" + 0.005*"steak"'),
 (1,
  '0.047*"biji" + 0.036*"beras" + 0.032*"kapulaga" + 0.028*"kayu" + 0.028*"cengkeh" + 0.028*"bawang" + 0.027*"kambing" + 0.024*"cincang" + 0.023*"hijau" + 0.022*"garam"'),
 (2,
  '0.069*"biji" + 0.059*"daun" + 0.043*"batang" + 0.041*"lembar" + 0.033*"bawang" + 0.030*"ruas" + 0.026*"butiran" + 0.024*"jari" + 0.024*"lengkuas" + 0.023*"salam"'),
 (3,
  '0.061*"bawang" + 0.045*"merah" + 0.041*"buah" + 0.040*"daun" + 0.036*"siung" + 0.028*"putih" + 0.028*"bumbu" + 0.027*"bubuk" + 0.027*"kambing" + 0.026*"daging"'),
 (4,
  '0.057*"sendok" + 0.042*"lawang" + 0.041*"bunga" + 0.028*"teh" + 0.028*"butir" + 0.023*"kapulaga" + 0.022*"royco" + 0.019*"nya" + 0.017*"makan" + 0.016*"wortel"'),
 (5,
  '0.086*"jari" + 0.056*"ruas" + 0.038*"tumis" + 0.037*"hitam" + 0.036*"plain" + 0.032*"salam" + 0.030*"rebusan" + 0.029*"geprek" 

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
kambingCpdProb = make_dictionary_terms_prob(ldamodel, id_map, d)
kambingCpdProb["mee"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
kambingCpdWN = dfWordnet.copy()

kambingCpdWN["Synonym_clean"] = cleanWN_Synonym(kambingCpdWN["Synonym"])
kambingCpdWN["Hypernim_clean"] = cleanWN(kambingCpdWN["Hypernim"])
kambingCpdWN["Hyponim_clean"] = cleanWN(kambingCpdWN["Hyponim"])
kambingCpdWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/kambingCookpad.owl")
makeOntology(ontoName, kambingCpdWN)
ontoName.save(file = "kambingCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1B6Cw9zU7dPMqhzyNd3CIRjLiP_ZPrQeQ"

In [ ]:
wiki_kambing_cpd = pd.read_excel("KambingCookpadFinal.xlsx").iloc[:, 1:]

# Translate
wiki_kambing_cpd["Category"] = translate(wiki_kambing_cpd["Category"])
wiki_kambing_cpd.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_kambing_cpd[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
kambingCpdTabel_new = tabel_1981.copy()
kambingCpdTabel_new["correct_category"] = all_cat
kambingCpdTabel_new["suggestion"] = all_suggestion

# Export Result to File
kambingCpdTabel_new.to_excel("kambingCpdTabel_Result.xlsx")
kambingCpdTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingCpdTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
kambingCpdTabel_new = concatListInColumn(kambingCpdTabel_new)
kambingCpdTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, kambingCpdTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_kambing_cpd.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
kambingCpdWiki_new = wiki_df.copy()
kambingCpdWiki_new["suggest"] = all_cat

# Export Result to file
kambingCpdWiki_new.to_excel("kambingCpdWiki_Result.xlsx")
kambingCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
kambingCpdOntoWiki = get_ontology("http://test.org/kambingCpd_onto_wiki.owl")
makeOntology_wiki(kambingCpdOntoWiki, kambingCpdWiki_new)
kambingCpdOntoWiki.save(file = "kambingCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = kambingCpdWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
kambingCpdWN_new = listToDataframe(changesList, tableWordnet)
kambingCpdWN_new.to_excel("kambingCpdWN_Result.xlsx")
kambingCpdWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "kambingCpdWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Sapi - Cookpad

In [ ]:
!gdown --id "1_HiEr1_J-c55HsGvRatqmWpjXQdLEt7_"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1_HiEr1_J-c55HsGvRatqmWpjXQdLEt7_
To: /content/cookpad-selenium_sapi.jl
100% 415k/415k [00:00<00:00, 97.0MB/s]


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
sapi_cpd=pd.read_json(r'/content/cookpad-selenium_sapi.jl', lines = True)
sapi_cpd = concat_bahan_langkah(sapi_cpd)

# fill NaN with empty string
sapi_cpd["page_description"] = sapi_cpd["page_description"].fillna("")

#save dalam acar biar ngebut loading ntar.
sapi_cpd.to_pickle(r'/content/raw_df.pkl')

# covnert page_description to list
sapi_cpd["temp"] = sapi_cpd["page_description"]
sapi_cpd["page_description"] = sapi_cpd["page_bahan_langkah"]

# # clean text
sapi_cpd = pre_cleaning_text(sapi_cpd, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
sapi_cpd.head()

,url_search,list_scraping_time,list_item_url,list_item_title,list_item_ingredients,list_item_author_name,list_item_author_image_url,list_item_image_url,page_scraping_time,page_image_url,...,page_serving_count,page_bahan,page_langkah,page_post_datetime,page_post_love_count,page_post_clap_count,page_post_tasty_count,page_comment_count,page_bahan_langkah,temp
0,https://cookpad.com/id/cari/sapi?event=search....,2022-06-24 23:14:28,https://cookpad.com/id/resep/16327449-us-beef-...,\n US Beef Slice Pedas Manis\n,"[US Beef Slice, Kecap Manis Daging Asap ABC, S...",Doyan Makan!,https://img-global.cpcdn.com/users/8cf96b2c882...,https://img-global.cpcdn.com/recipes/c5d03baf9...,2022-06-24 23:14:43,https://img-global.cpcdn.com/recipes/c5d03baf9...,...,"1,5 orang","[100 gr US Beef Slice, 1,5 sdt Kecap Manis Dag...",[],24 Juni 2022 22.42,NaN,NaN,NaN,NaN,"[100 gr US Beef Slice, 1,5 sdt Kecap Manis Dag...",US Beef Slice ini menu terinspirasi dari Raa C...
1,https://cookpad.com/id/cari/sapi?event=search....,2022-06-24 23:14:28,https://cookpad.com/id/resep/16321842-280-riso...,\n 280. RISOLES MAYO SMOKE BEEF\n,"[resep kulit dadar, resep adonan pencelup riso...",Ibu Endah,https://img-global.cpcdn.com/users/610bbdb12d2...,https://img-global.cpcdn.com/recipes/d6fa4e8ab...,2022-06-24 23:14:50,https://img-global.cpcdn.com/recipes/d6fa4e8ab...,...,24 biji,"[1 resep kulit dadar\n (lihat resep),...",[],24 Juni 2022 09.53,7.0,5.0,5.0,NaN,"[1 resep kulit dadar\n (lihat resep),...",Assalamualaikum wr wb\n
2,https://cookpad.com/id/cari/sapi?event=search....,2022-06-24 23:14:28,https://cookpad.com/id/resep/16325861-sop-tete...,\n Sop Tetelan Sedap Wangi\n,"[tetelan, kentang, wortel, daun bawang dan sel...",Hayati MS,https://img-global.cpcdn.com/users/fed80c02c96...,https://img-global.cpcdn.com/recipes/971c2ab7a...,2022-06-24 23:14:56,https://img-global.cpcdn.com/recipes/971c2ab7a...,...,1 panci,"[250 gram tetelan, 2 buah kentang, 2 buah wort...",[],24 Juni 2022 12.09,20.0,4.0,6.0,2.0,"[250 gram tetelan, 2 buah kentang, 2 buah wort...",Masak simpel tapi ala resto dan dilahap habis ...
3,https://cookpad.com/id/cari/sapi?event=search....,2022-06-24 23:14:28,https://cookpad.com/id/resep/16320884-gulai-da...,\n Gulai Daging Sapi Pakai Bumbu ...,"[daging sapi, potong dadu, Jeruk nipis, Bumbu ...",Tanti Mak Ibra,https://img-global.cpcdn.com/users/c58dfedf57f...,https://img-global.cpcdn.com/recipes/89474b551...,2022-06-24 23:15:02,https://img-global.cpcdn.com/recipes/89474b551...,...,2 orang,"[1/2 kg daging sapi, potong dadu, Jeruk nipis,...",[],22 Juni 2022 05.32,10.0,3.0,7.0,NaN,"[1/2 kg daging sapi, potong dadu, Jeruk nipis,...",Lusa lalu beli Ind*mie goreng di minimarket. A...
4,https://cookpad.com/id/cari/sapi?event=search....,2022-06-24 23:14:28,https://cookpad.com/id/resep/16321487-tumis-ka...,\n Tumis Kangkung Beef Slice\n,"[kangkung, beef slice, saos tiram, saos teriya...",Martha Christinawatie,https://img-global.cpcdn.com/users/9cb5007c0e4...,https://img-global.cpcdn.com/recipes/f77275b45...,2022-06-24 23:15:08,https://img-global.cpcdn.com/recipes/f77275b45...,...,2 orang,"[1 ikat kangkung, 150 gram beef slice, 1 sdt s...",[],22 Juni 2022 08.54,6.0,4.0,3.0,NaN,"[1 ikat kangkung, 150 gram beef slice, 1 sdt s...",


## GUI Sapi - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = sapi_cpd.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4     -0.240078 -0.002265       1        1  41.167074
6     -0.103117 -0.152694       2        1  34.180843
2      0.017067  0.012328       3        1   7.268400
1     -0.065870 -0.046550       4        1   6.053477
3     -0.043193  0.248163       5        1   5.179283
5      0.201905 -0.156557       6        1   4.393116
0      0.233286  0.097575       7        1   1.757806, topic_info=              Term        Freq       Total Category  logprob  loglift
37            buah  225.000000  225.000000  Default  30.0000  30.0000
176           saus   95.000000   95.000000  Default  29.0000  29.0000
47           bumbu  146.000000  146.000000  Default  28.0000  28.0000
41            daun  158.000000  158.000000  Default  27.0000  27.0000
60            iris  136.000000  136.000000  Default  26.0000  26.0000
85           garam  160.000000  160.000000  Default  25.0000  25.0000
3            manis  120.000000  120.000000  Default  24.0000  24.0000
55           butir   98.000000   98.000000  Default  23.0000  23.0000
14           bubuk  148.000000  148.000000  Default  22.0000  22.0000
35            gram   40.000000   40.000000  Default  21.0000  21.0000
51           salam   61.000000   61.000000  Default  20.0000  20.0000
64            sapi  153.000000  153.000000  Default  19.0000  19.0000
0             beef   51.000000   51.000000  Default  18.0000  18.0000
286          sayur   25.000000   25.000000  Default  17.0000  17.0000
52           halus  108.000000  108.000000  Default  16.0000  16.0000
56          kemiri   47.000000   47.000000  Default  15.0000  15.0000
54           putih  217.000000  217.000000  Default  14.0000  14.0000
45             air  151.000000  151.000000  Default  13.0000  13.0000
61            jahe   80.000000   80.000000  Default  12.0000  12.0000
94            cabe   49.000000   49.000000  Default  11.0000  11.0000
77            ruas   67.000000   67.000000  Default  10.0000  10.0000
4           daging  149.000000  149.000000  Default   9.0000   9.0000
26             lbr   22.000000   22.000000  Default   8.0000   8.0000
70           merah  188.000000  188.000000  Default   7.0000   7.0000
92           kaldu   84.000000   84.000000  Default   6.0000   6.0000
134         santan   42.000000   42.000000  Default   5.0000   5.0000
29           telur   25.000000   25.000000  Default   4.0000   4.0000
44          goreng   52.000000   52.000000  Default   3.0000   3.0000
78        lengkuas   32.000000   32.000000  Default   2.0000   2.0000
42          bawang  443.000000  443.000000  Default   1.0000   1.0000
49          lembar   69.841049   70.479421   Topic1  -3.7893   0.8784
75          selera   66.593694   67.232284   Topic1  -3.8369   0.8780
74          sesuai   63.948534   64.587320   Topic1  -3.8774   0.8776
172         geprek   55.053437   55.692014   Topic1  -4.0272   0.8760
50           jeruk   52.317163   53.075099   Topic1  -4.0782   0.8731
103           jawa   26.031883   26.670222   Topic1  -4.7762   0.8633
315          serai   23.340182   23.977827   Topic1  -4.8854   0.8606
306           asam   18.776327   19.428511   Topic1  -5.1029   0.8534
72        keriting   11.629819   12.268482   Topic1  -5.5820   0.8341
113            amp   10.619655   11.258264   Topic1  -5.6728   0.8291
223            iga    9.947266   10.584933   Topic1  -5.7382   0.8254
58          jinten   16.750557   17.846977   Topic1  -5.2171   0.8241
236          bunga    9.514269   10.152126   Topic1  -5.7827   0.8226
607         keping    8.216781    8.853995   Topic1  -5.9293   0.8128
454          bulat    7.972523    8.612802   Topic1  -5.9595   0.8103
331         kental   22.025700   23.978106   Topic1  -4.9433   0.8026
95           rawit   20.172853   21.970499   Topic1  -5.0312   0.8022
182          tumis    6.990176    7.627478   Topic1  -6.0910   0.8003
610         salmon    6.557531    7.19471

Topic: 1
baput, bamer, iris, seruas, menyesuaikan, terasi, gelas, sirloin, haluskan, skip, tipis, sour, smua, mebeli, dipake, indomie, hijaupotong, seujung, limau, matah, kemangi, cuci, bersih, panas, cabe, steak, grain, milk, lemon, juice

Topic: 2
buah, iris, manis, bawang, bubuk, kelapa, bumbu, cemplung, menumis, jahe, black, paper, sapi, ruas, gram, goreng, dihaluskan, daun, tetelan, garam, lawang, seledri, aromatik, lada, pelengkap, kayu, parut, cengkeh, pala, ukuran

Topic: 3
bubuk, telur, air, bawang, rebus, resep, halus, lihat, kaldu, butir, iris, tepung, merica, daun, sauce, taburan, btg, potongpotong, sapi, soy, butiran, kentang, diskip, gula, garam, daging, liter, kurangi, madunya, takaran

Topic: 4
bumbu, lbr, daun, btg, sapi, salam, daging, sereh, kara, kemiri, cabe, halus, butir, ketumbar, kunyit, kol, lengkuas, gula, lada, santan, ijo, kelapa, kayumanis, bawang, laja, airnya, tomat, presto, rawitklau, sasakecap

Topic: 5
bawang, merah, siung, daun, bumbu, putih, buah, gu

[(0,
  '0.053*"baput" + 0.038*"bamer" + 0.014*"iris" + 0.012*"seruas" + 0.010*"menyesuaikan" + 0.010*"terasi" + 0.010*"gelas" + 0.009*"sirloin" + 0.008*"haluskan" + 0.008*"skip"'),
 (1,
  '0.113*"buah" + 0.043*"iris" + 0.035*"manis" + 0.034*"bawang" + 0.033*"bubuk" + 0.029*"kelapa" + 0.025*"bumbu" + 0.024*"cemplung" + 0.023*"menumis" + 0.022*"jahe"'),
 (2,
  '0.033*"bubuk" + 0.032*"telur" + 0.031*"air" + 0.030*"bawang" + 0.029*"rebus" + 0.025*"resep" + 0.025*"halus" + 0.024*"lihat" + 0.023*"kaldu" + 0.022*"butir"'),
 (3,
  '0.057*"bumbu" + 0.050*"lbr" + 0.047*"daun" + 0.036*"btg" + 0.034*"sapi" + 0.024*"salam" + 0.024*"daging" + 0.024*"sereh" + 0.023*"kara" + 0.022*"kemiri"'),
 (4,
  '0.065*"bawang" + 0.049*"merah" + 0.044*"siung" + 0.036*"daun" + 0.029*"bumbu" + 0.028*"putih" + 0.028*"buah" + 0.026*"gula" + 0.025*"garam" + 0.024*"sapi"'),
 (5,
  '0.075*"sayur" + 0.063*"saus" + 0.058*"gram" + 0.040*"garam" + 0.028*"oil" + 0.028*"putih" + 0.025*"sdk" + 0.024*"beef" + 0.023*"telur" + 0.0

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sapiCpdProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sapiCpdProb["sapi"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
sapiCpdWN = dfWordnet.copy()

sapiCpdWN["Synonym_clean"] = cleanWN_Synonym(sapiCpdWN["Synonym"])
sapiCpdWN["Hypernim_clean"] = cleanWN(sapiCpdWN["Hypernim"])
sapiCpdWN["Hyponim_clean"] = cleanWN(sapiCpdWN["Hyponim"])
sapiCpdWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/sapiCookpad.owl")
makeOntology(ontoName, sapiCpdWN)
ontoName.save(file = "sapiCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown --id "1VB_hbAtT1k3woShItXm2NVOZLWB4LLhk"

In [ ]:
wiki_sapi_cpd = pd.read_excel("SapiCookpadFinal.xlsx").iloc[:, 1:]

# Translate
wiki_sapi_cpd["Category"] = translate(wiki_sapi_cpd["Category"])
wiki_sapi_cpd.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_sapi_cpd[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
sapiCpdTabel_new = tabel_1981.copy()
sapiCpdTabel_new["correct_category"] = all_cat
sapiCpdTabel_new["suggestion"] = all_suggestion

# Export Result to File
sapiCpdTabel_new.to_excel("sapiCpdTabel_Result.xlsx")
sapiCpdTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiCpdTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
sapiCpdTabel_new = concatListInColumn(sapiCpdTabel_new)
sapiCpdTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, sapiCpdTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_sapi_cpd.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

In [ ]:
sapiCpdWiki_new = wiki_df.copy()
sapiCpdWiki_new["suggest"] = all_cat

# Export Result to file
sapiCpdWiki_new.to_excel("sapiCpdWiki_Result.xlsx")
sapiCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
sapiCpdOntoWiki = get_ontology("http://test.org/sapiCpd_onto_wiki.owl")
makeOntology_wiki(sapiCpdOntoWiki, sapiCpdWiki_new)
sapiCpdOntoWiki.save(file = "sapiCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = sapiCpdWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
sapiCpdWN_new = listToDataframe(changesList, tableWordnet)
sapiCpdWN_new.to_excel("sapiCpdWN_Result.xlsx")
sapiCpdWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "sapiCpdWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Tahu - Cookpad

In [ ]:
!gdown 1H4AQrK-VnNAhhx0hevdTAPr9UFRcquoI

Downloading...
From: https://drive.google.com/uc?id=1H4AQrK-VnNAhhx0hevdTAPr9UFRcquoI
To: /content/cookpad-selenium_tahu.jl
100% 396k/396k [00:00<00:00, 78.8MB/s]


In [ ]:
cps_tahu = pd.read_json(r'/content/cookpad-selenium_tahu.jl', lines = True)
print(f'NaN page description: {len(cps_tahu.loc[cps_tahu["page_description"].isna()])}')

# fill NaN with empty string
cps_tahu["page_description"] = cps_tahu["page_description"].fillna("")

# covnert page_description to list
cps_tahu["temp"] = cps_tahu["page_description"]
cps_tahu = concat_bahan_langkah(cps_tahu)
cps_tahu["page_description"] = cps_tahu["page_bahan_langkah"]

# clean text
cps_tahu = pre_cleaning_text(cps_tahu, threshold = 20)

NaN page description: 34


## GUI Tahu - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = cps_tahu.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3     -0.161535 -0.073724       1        1  50.938766
0     -0.123733  0.053217       2        1  22.076286
6     -0.145807 -0.018596       3        1  11.102594
2     -0.047562  0.219558       4        1   4.892865
4      0.005917 -0.193662       5        1   4.852979
1      0.226136 -0.014493       6        1   4.700520
5      0.246584  0.027699       7        1   1.435990, topic_info=                Term        Freq       Total Category  logprob  loglift
1               buah  296.000000  296.000000  Default  30.0000  30.0000
9              bubuk  156.000000  156.000000  Default  29.0000  29.0000
18            bawang  378.000000  378.000000  Default  28.0000  28.0000
8              kaldu  120.000000  120.000000  Default  27.0000  27.0000
5             tepung   75.000000   75.000000  Default  26.0000  26.0000
7              garam  191.000000  191.000000  Default  25.0000  25.0000
34             rawit   66.000000   66.000000  Default  24.0000  24.0000
2              putih  237.000000  237.000000  Default  23.0000  23.0000
41             merah  153.000000  153.000000  Default  22.0000  22.0000
56             telur   77.000000   77.000000  Default  21.0000  21.0000
123             lada   59.000000   59.000000  Default  20.0000  20.0000
33              cabe  123.000000  123.000000  Default  19.0000  19.0000
178             ikat   29.000000   29.000000  Default  18.0000  18.0000
36            selera   45.000000   45.000000  Default  17.0000  17.0000
16            batang   87.000000   87.000000  Default  16.0000  16.0000
24            wortel   58.000000   58.000000  Default  15.0000  15.0000
52            goreng   64.000000   64.000000  Default  14.0000  14.0000
146             iris   70.000000   70.000000  Default  13.0000  13.0000
22             halus   69.000000   69.000000  Default  12.0000  12.0000
20             siung  192.000000  192.000000  Default  11.0000  11.0000
15              ayam   48.000000   48.000000  Default  10.0000  10.0000
12               air   87.000000   87.000000  Default   9.0000   9.0000
30             bumbu   80.000000   80.000000  Default   8.0000   8.0000
285          genggam   17.000000   17.000000  Default   7.0000   7.0000
44            terigu   26.000000   26.000000  Default   6.0000   6.0000
187            resep   15.000000   15.000000  Default   5.0000   5.0000
105              nya   22.000000   22.000000  Default   4.0000   4.0000
325          secukup   22.000000   22.000000  Default   3.0000   3.0000
145            rebus   20.000000   20.000000  Default   2.0000   2.0000
172          cincang   30.000000   30.000000  Default   1.0000   1.0000
3              butir   77.815843   78.393323   Topic1  -3.6984   0.6672
76             salam   35.441148   36.017891   Topic1  -4.4849   0.6584
23              biji   28.444418   29.021668   Topic1  -4.7048   0.6545
166            kulit   27.648184   28.226159   Topic1  -4.7332   0.6539
70            kemiri   25.883988   26.460638   Topic1  -4.7991   0.6525
68            kunyit   25.378851   25.955477   Topic1  -4.8188   0.6521
72            geprek   25.998790   26.599899   Topic1  -4.7947   0.6517
77             jeruk   20.690019   21.267412   Topic1  -5.0231   0.6470
67              ruas   20.510538   21.087304   Topic1  -5.0318   0.6468
120           kuning   20.005222   20.582089   Topic1  -5.0568   0.6461
73          lengkuas   18.021756   18.598593   Topic1  -5.1612   0.6430
663           rolade   17.841565   18.418072   Topic1  -5.1712   0.6427
221            serai   17.409504   17.985915   Topic1  -5.1957   0.6420
81            santan   14.234954   14.811834   Topic1  -5.3970   0.6348
96             tempe   14.112180   14.701586   Topic1  -5.4057   0.6336
260          kriting   12.143337   12.720061   Topic1  -5.5560   0.6281
469              ons   11.494827   12.071283   Topic1  -5.6108   0.6256
259           p

Topic: 1
bawang, putih, iris, siung, kaldu, garam, bubuk, gula, cincang, asin, air, selera, sesuai, kecap, merica, jamur, kotak, daun, lada, cabe, saus, buah, tipis, tiram, tepung, potongpotong, bumbu, wijen, sawi, lembar

Topic: 2
genggam, resep, ikat, goreng, segenggam, cabe, lihat, rebus, kacang, rajang, rawit, irisan, makaroni, serut, aldente, asli, petis, sayur, seledri, lada, putih, lontong, kerupuk, air, kecap, ijo, manis, garam, taoge, pelengkap

Topic: 3
bubuk, tepung, garam, kaldu, lada, putih, selera, bawang, prei, terigu, telur, air, batang, kocok, halus, sesuaikan, pencelup, blok, saji, mencairkan, cabut, daunnya, maizena, sosis, singkong, roll, egg, sepucuk, nori, dgn

Topic: 4
bawang, buah, daun, merah, putih, siung, garam, cabe, butir, batang, gula, telur, bubuk, kaldu, halus, rawit, bumbu, lembar, wortel, salam, ukuran, air, ayam, biji, kulit, jamur, tiram, merica, geprek, kemiri

Topic: 5
buah, bawang, sejumput, putih, bayam, merah, rawit, merk, tomat, bumbu, pelapis,

[(0,
  '0.058*"bawang" + 0.040*"putih" + 0.037*"iris" + 0.035*"siung" + 0.031*"kaldu" + 0.030*"garam" + 0.030*"bubuk" + 0.026*"gula" + 0.022*"cincang" + 0.022*"asin"'),
 (1,
  '0.057*"genggam" + 0.052*"resep" + 0.048*"ikat" + 0.031*"goreng" + 0.031*"segenggam" + 0.030*"cabe" + 0.027*"lihat" + 0.024*"rebus" + 0.024*"kacang" + 0.024*"rajang"'),
 (2,
  '0.138*"bubuk" + 0.069*"tepung" + 0.060*"garam" + 0.057*"kaldu" + 0.051*"lada" + 0.044*"putih" + 0.034*"selera" + 0.029*"bawang" + 0.029*"prei" + 0.029*"terigu"'),
 (3,
  '0.071*"bawang" + 0.065*"buah" + 0.042*"daun" + 0.041*"merah" + 0.041*"putih" + 0.038*"siung" + 0.034*"garam" + 0.029*"cabe" + 0.025*"butir" + 0.024*"batang"'),
 (4,
  '0.136*"buah" + 0.078*"bawang" + 0.036*"sejumput" + 0.034*"putih" + 0.033*"bayam" + 0.032*"merah" + 0.030*"rawit" + 0.026*"merk" + 0.025*"tomat" + 0.024*"bumbu"'),
 (5,
  '0.047*"homemade" + 0.047*"mangkok" + 0.046*"hancurkan" + 0.013*"sdh" + 0.013*"bentuk" + 0.009*"nugget" + 0.009*"lmbr" + 0.009*"kuranglebi

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sTahuProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sTahuProb["bawang"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
cpsTahuWN = dfWordnet.copy()

cpsTahuWN["Synonym_clean"] = cleanWN_Synonym(cpsTahuWN["Synonym"])
cpsTahuWN["Hypernim_clean"] = cleanWN(cpsTahuWN["Hypernim"])
cpsTahuWN["Hyponim_clean"] = cleanWN(cpsTahuWN["Hyponim"])
cpsTahuWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/tahuCookpad.owl")
makeOntology(ontoName, cpsTahuWN)
ontoName.save(file = "tahuCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown 1RVnjWTvdYu4b-kDFrhfRcaKcK7RfYjoe

In [ ]:
wiki_tahu_sln = pd.read_excel("TahuCookpadFinal.xlsx").iloc[:, 1:]
wiki_tahu_sln.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_tahu_sln["Category"] = translate(wiki_tahu_sln["Category"])
wiki_tahu_sln.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_tahu_sln[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
tahuSlnTabel_new = tabel_1981.copy()
tahuSlnTabel_new["correct_category"] = all_cat
tahuSlnTabel_new["suggestion"] = all_suggestion

# Export Result to File
tahuSlnTabel_new.to_excel("tahuSlnTabel_Result.xlsx")
tahuSlnTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tahuSlnTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
tahuSlnTabel_new = concatListInColumn(tahuSlnTabel_new)
tahuSlnTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, tahuSlnTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_tahu_sln.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
tahuCpdWiki_new = wiki_df.copy()
tahuCpdWiki_new["suggest"] = all_cat

# Export Result to file
tahuCpdWiki_new.to_excel("tahuCpdWiki_Result.xlsx")
tahuCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tahuCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
tahuCpdOntoWiki = get_ontology("http://test.org/tahuCpd_onto_wiki.owl")
makeOntology_wiki(tahuCpdOntoWiki, tahuCpdWiki_new)
tahuCpdOntoWiki.save(file = "tahuCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = cpsTahuWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
cpsTahuWN_new = listToDataframe(changesList, tableWordnet)
cpsTahuWN_new.to_excel("cpsTahuWN_Result.xlsx")
cpsTahuWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cpsTahuWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Telur - Cookpad

In [ ]:
!gdown 1RizCmOSgDG9krsvBFCieRQP5hpe9aIdW

Downloading...
From: https://drive.google.com/uc?id=1RizCmOSgDG9krsvBFCieRQP5hpe9aIdW
To: /content/cookpad-selenium_telur.jl
100% 384k/384k [00:00<00:00, 112MB/s]


In [ ]:
cps_telur = pd.read_json(r'/content/cookpad-selenium_telur.jl', lines = True)
print(f'NaN page description: {len(cps_telur.loc[cps_telur["page_description"].isna()])}')

# fill NaN with empty string
cps_telur["page_description"] = cps_telur["page_description"].fillna("")

# covnert page_description to list
cps_telur["temp"] = cps_telur["page_description"]
cps_telur = concat_bahan_langkah(cps_telur)
cps_telur["page_description"] = cps_telur["page_bahan_langkah"]

# clean text
cps_telur = pre_cleaning_text(cps_telur, threshold = 20)

NaN page description: 41


## GUI Telur - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = cps_telur.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sTelurProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sTelurProb["bawang"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
cpsTelurWN = dfWordnet.copy()

cpsTelurWN["Synonym_clean"] = cleanWN_Synonym(cpsTelurWN["Synonym"])
cpsTelurWN["Hypernim_clean"] = cleanWN(cpsTelurWN["Hypernim"])
cpsTelurWN["Hyponim_clean"] = cleanWN(cpsTelurWN["Hyponim"])
cpsTelurWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/telurCookpad.owl")
makeOntology(ontoName, cpsTelurWN)
ontoName.save(file = "telurCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown 1zDnVhtmf3HGT2YagYDYFxxmGM1hDm4QV

In [ ]:
wiki_telur_sln = pd.read_excel("TelurCookpadFinal.xlsx").iloc[:, 1:]
wiki_telur_sln.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_telur_sln["Category"] = translate(wiki_telur_sln["Category"])
wiki_telur_sln.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_telur_sln[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
telurSlnTabel_new = tabel_1981.copy()
telurSlnTabel_new["correct_category"] = all_cat
telurSlnTabel_new["suggestion"] = all_suggestion

# Export Result to File
telurSlnTabel_new.to_excel("telurSlnTabel_Result.xlsx")
telurSlnTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "telurSlnTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
telurSlnTabel_new = concatListInColumn(telurSlnTabel_new)
telurSlnTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, telurSlnTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_telur_sln.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
telurCpdWiki_new = wiki_df.copy()
telurCpdWiki_new["suggest"] = all_cat

# Export Result to file
telurCpdWiki_new.to_excel("telurCpdWiki_Result.xlsx")
telurCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "telurCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
telurCpdOntoWiki = get_ontology("http://test.org/telurCpd_onto_wiki.owl")
makeOntology_wiki(telurCpdOntoWiki, telurCpdWiki_new)
telurCpdOntoWiki.save(file = "telurCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = cpsTelurWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
cpsTelurWN_new = listToDataframe(changesList, tableWordnet)
cpsTelurWN_new.to_excel("cpsTelurWN_Result.xlsx")
cpsTelurWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cpsTelurWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Tempe Cookpad

In [ ]:
!gdown 1dpqsr-WMdFtALig6js2Lc7Xk_Nc7u4q_

Downloading...
From: https://drive.google.com/uc?id=1dpqsr-WMdFtALig6js2Lc7Xk_Nc7u4q_
To: /content/cookpad-selenium_tempe.jl
100% 401k/401k [00:00<00:00, 104MB/s]


In [ ]:
cps_tempe = pd.read_json(r'/content/cookpad-selenium_tempe.jl', lines = True)
print(f'NaN page description: {len(cps_tempe.loc[cps_tempe["page_description"].isna()])}')

# fill NaN with empty string
cps_tempe["page_description"] = cps_tempe["page_description"].fillna("")

# covnert page_description to list
cps_tempe["temp"] = cps_tempe["page_description"]
cps_tempe = concat_bahan_langkah(cps_tempe)
cps_tempe["page_description"] = cps_tempe["page_bahan_langkah"]

# clean text
cps_tempe = pre_cleaning_text(cps_tempe, threshold = 20)

NaN page description: 25


## GUI Tempe - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = cps_tempe.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.241870  0.095309       1        1  39.662271
6      0.232971  0.022779       2        1  32.463099
3     -0.018340 -0.317111       3        1  13.611750
0      0.015337 -0.059538       4        1   8.269289
5     -0.088787  0.155886       5        1   2.165622
1     -0.200545  0.037830       6        1   1.991393
4     -0.182507  0.064846       7        1   1.836577, topic_info=              Term        Freq       Total Category  logprob  loglift
96          tepung   67.000000   67.000000  Default  30.0000  30.0000
21            buah  170.000000  170.000000  Default  29.0000  29.0000
35           merah  177.000000  177.000000  Default  28.0000  28.0000
45           bubuk  117.000000  117.000000  Default  27.0000  27.0000
27             air  154.000000  154.000000  Default  26.0000  26.0000
28            gula   90.000000   90.000000  Default  25.0000  25.0000
95            gram   63.000000   63.000000  Default  24.0000  24.0000
104           ragi   51.000000   51.000000  Default  23.0000  23.0000
37          bawang  251.000000  251.000000  Default  22.0000  22.0000
22            cabe  105.000000  105.000000  Default  21.0000  21.0000
232         terigu   33.000000   33.000000  Default  20.0000  20.0000
99         kedelai   42.000000   42.000000  Default  19.0000  19.0000
24            iris   80.000000   80.000000  Default  18.0000  18.0000
16           tempe  232.000000  232.000000  Default  17.0000  17.0000
93          bersih   35.000000   35.000000  Default  16.0000  16.0000
12          batang   59.000000   59.000000  Default  15.0000  15.0000
44        ketumbar   57.000000   57.000000  Default  14.0000  14.0000
205        plastik   34.000000   34.000000  Default  13.0000  13.0000
60           butir   39.000000   39.000000  Default  12.0000  12.0000
87          goreng   56.000000   56.000000  Default  11.0000  11.0000
23           rawit   67.000000   67.000000  Default  10.0000  10.0000
191          liter   28.000000   28.000000  Default   9.0000   9.0000
54           telur   16.000000   16.000000  Default   8.0000   8.0000
14           garam  151.000000  151.000000  Default   7.0000   7.0000
135          pasir   19.000000   19.000000  Default   6.0000   6.0000
103        merebus   26.000000   26.000000  Default   5.0000   5.0000
36           siung  169.000000  169.000000  Default   4.0000   4.0000
140          resep   47.000000   47.000000  Default   3.0000   3.0000
102       merendam   23.000000   23.000000  Default   2.0000   2.0000
38           putih  159.000000  159.000000  Default   1.0000   1.0000
35           merah  176.772033  177.408272   Topic1  -2.5385   0.9212
22            cabe  105.317041  105.907057   Topic1  -3.0564   0.9192
23           rawit   66.690121   67.277052   Topic1  -3.5133   0.9160
20           jeruk   32.231296   32.818175   Topic1  -4.2405   0.9067
172       lengkuas   30.690653   31.277500   Topic1  -4.2894   0.9058
162          cabai   26.776548   27.368411   Topic1  -4.4259   0.9029
115       keriting   25.767975   26.354826   Topic1  -4.4643   0.9023
134         kencur   19.792950   20.379975   Topic1  -4.7281   0.8955
706            bwg   18.590956   19.177402   Topic1  -4.7907   0.8937
118          kecap   45.804383   47.287137   Topic1  -3.8890   0.8929
68             bks   15.355929   15.945027   Topic1  -4.9819   0.8871
125         sambal   21.658592   22.571519   Topic1  -4.6380   0.8835
150         geprek   13.329109   13.948105   Topic1  -5.1235   0.8794
648          gurih   11.977293   12.564099   Topic1  -5.2304   0.8769
445          racik   11.780016   12.374317   Topic1  -5.2470   0.8756
362        instant   10.920386   11.510681   Topic1  -5.3228   0.8721
19         kemangi   10.774538   11.361576   Topic1  -5.3362   0.8717
175           jawa   11.399261   12.029074   Topic1  -5.2799   0.8710
707       sejempol    9.384612    9.97105

Topic: 1
tepung, gram, terigu, bubuk, tempe, air, goreng, ketumbar, batang, garam, beras, kunyit, brokoli, coklat, adonan, kriuk, sachet, protein, susu, gula, tapioka, powder, kobe, halus, bon, baking, sapi, sate, vanili, mendoan

Topic: 2
nampan, wadah, tutup, datar, ltr, ragi, ayak, bening, munjung, soto, kripik, charcoal, krim, kelor, kedele, peres, liter, celupan, angkak, utuh, diserut, rupiah, pepaya, campuran, kating, indomie, special, dilubangi, tunggak, sdh

Topic: 3
merah, buah, bawang, putih, cabe, siung, garam, gula, tempe, rawit, daun, kecap, ketumbar, bumbu, salam, jeruk, lengkuas, cabai, keriting, manis, selera, kemiri, tomat, sesuai, sambal, ruas, kencur, halus, bwg, biji

Topic: 4
air, ragi, kedelai, tempe, bersih, plastik, liter, merebus, gram, merendam, panci, kacang, kupas, cetakan, daun, lap, tusuk, gigi, jarum, tampah, kulit, alat, saringan, baskom, wrap, bungkus, utk, pisang, pembungkus, cuka

Topic: 5
pelapis, keju, cheddar, onion, cacah, kismis, strawberry, flou

[(0,
  '0.143*"tepung" + 0.078*"gram" + 0.057*"terigu" + 0.056*"bubuk" + 0.050*"tempe" + 0.037*"air" + 0.033*"goreng" + 0.030*"ketumbar" + 0.030*"batang" + 0.028*"garam"'),
 (1,
  '0.079*"nampan" + 0.073*"wadah" + 0.069*"tutup" + 0.033*"datar" + 0.016*"ltr" + 0.012*"ragi" + 0.010*"ayak" + 0.010*"bening" + 0.007*"munjung" + 0.007*"soto"'),
 (2,
  '0.079*"merah" + 0.062*"buah" + 0.053*"bawang" + 0.047*"putih" + 0.047*"cabe" + 0.042*"siung" + 0.041*"garam" + 0.036*"gula" + 0.034*"tempe" + 0.030*"rawit"'),
 (3,
  '0.081*"air" + 0.065*"ragi" + 0.055*"kedelai" + 0.053*"tempe" + 0.045*"bersih" + 0.044*"plastik" + 0.035*"liter" + 0.033*"merebus" + 0.032*"gram" + 0.030*"merendam"'),
 (4,
  '0.078*"pelapis" + 0.055*"keju" + 0.041*"cheddar" + 0.011*"onion" + 0.011*"cacah" + 0.008*"kismis" + 0.008*"strawberry" + 0.008*"flour" + 0.008*"gorengfried" + 0.008*"roti"'),
 (5,
  '0.072*"bahan" + 0.054*"terigu" + 0.042*"telur" + 0.042*"buah" + 0.033*"kering" + 0.029*"gula" + 0.029*"pasir" + 0.028*"tangkai

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sTempeProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sTempeProb["bawang"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
cpsTempeWN = dfWordnet.copy()

cpsTempeWN["Synonym_clean"] = cleanWN_Synonym(cpsTempeWN["Synonym"])
cpsTempeWN["Hypernim_clean"] = cleanWN(cpsTempeWN["Hypernim"])
cpsTempeWN["Hyponim_clean"] = cleanWN(cpsTempeWN["Hyponim"])
cpsTempeWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/tempeCookpad.owl")
makeOntology(ontoName, cpsTempeWN)
ontoName.save(file = "tempeCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown 1Ca3lXQKYevxhdo-BQkWRw3zVGoFIdPGF

In [ ]:
wiki_tempe_sln = pd.read_excel("TempeCookpadFinal.xlsx").iloc[:, 1:]
wiki_tempe_sln.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_tempe_sln["Category"] = translate(wiki_tempe_sln["Category"])
wiki_tempe_sln.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_tempe_sln[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
tempeSlnTabel_new = tabel_1981.copy()
tempeSlnTabel_new["correct_category"] = all_cat
tempeSlnTabel_new["suggestion"] = all_suggestion

# Export Result to File
tempeSlnTabel_new.to_excel("tempeSlnTabel_Result.xlsx")
tempeSlnTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tempeSlnTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
tempeSlnTabel_new = concatListInColumn(tempeSlnTabel_new)
tempeSlnTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, tempeSlnTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_tempe_sln.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
tempeCpdWiki_new = wiki_df.copy()
tempeCpdWiki_new["suggest"] = all_cat

# Export Result to file
tempeCpdWiki_new.to_excel("tempeCpdWiki_Result.xlsx")
tempeCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "tempeCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
tempeCpdOntoWiki = get_ontology("http://test.org/tempeCpd_onto_wiki.owl")
makeOntology_wiki(tempeCpdOntoWiki, tempeCpdWiki_new)
tempeCpdOntoWiki.save(file = "tempeCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = cpsTempeWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
cpsTempeWN_new = listToDataframe(changesList, tableWordnet)
cpsTempeWN_new.to_excel("cpsTempeWN_Result.xlsx")
cpsTempeWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cpsTempeWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Udang - Cookpad

In [ ]:
!gdown 1zV22F5HEqQiRMukMWx7rkWAs7JNtMEmF

Downloading...
From: https://drive.google.com/uc?id=1zV22F5HEqQiRMukMWx7rkWAs7JNtMEmF
To: /content/cookpad-selenium_udang.jl
100% 403k/403k [00:00<00:00, 97.3MB/s]


In [ ]:
cps_udang = pd.read_json(r'/content/cookpad-selenium_udang.jl', lines = True)
print(f'NaN page description: {len(cps_udang.loc[cps_udang["page_description"].isna()])}')

# fill NaN with empty string
cps_udang["page_description"] = cps_udang["page_description"].fillna("")

# covnert page_description to list
cps_udang["temp"] = cps_udang["page_description"]
cps_udang = concat_bahan_langkah(cps_udang)
cps_udang["page_description"] = cps_udang["page_bahan_langkah"]

# clean text
cps_udang = pre_cleaning_text(cps_udang, threshold = 20)

NaN page description: 41


## GUI Udang - Cookpad

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = cps_udang.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(run_button_clicked)
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.217062 -0.078868       1        1  52.268626
0      0.201345  0.148554       2        1  19.986747
1      0.131884 -0.131859       3        1  13.522870
3     -0.044559  0.189196       4        1   8.266531
5     -0.187083  0.019997       5        1   2.707614
6     -0.118521 -0.155045       6        1   2.162112
4     -0.200129  0.008024       7        1   1.085501, topic_info=                  Term        Freq       Total Category  logprob  loglift
37                buah  196.000000  196.000000  Default  30.0000  30.0000
114               iris   92.000000   92.000000  Default  29.0000  29.0000
27               merah  188.000000  188.000000  Default  28.0000  28.0000
131              cabai   30.000000   30.000000  Default  27.0000  27.0000
29               rawit   60.000000   60.000000  Default  26.0000  26.0000
28                cabe  110.000000  110.000000  Default  25.0000  25.0000
47               jeruk   54.000000   54.000000  Default  24.0000  24.0000
207              manis   57.000000   57.000000  Default  23.0000  23.0000
11               telur   63.000000   63.000000  Default  22.0000  22.0000
43                daun  130.000000  130.000000  Default  21.0000  21.0000
197               saos   69.000000   69.000000  Default  20.0000  20.0000
12              tepung   76.000000   76.000000  Default  19.0000  19.0000
4                udang  218.000000  218.000000  Default  18.0000  18.0000
32                 air  126.000000  126.000000  Default  17.0000  17.0000
39               bumbu   94.000000   94.000000  Default  16.0000  16.0000
30              sambal   24.000000   24.000000  Default  15.0000  15.0000
147               ruas   33.000000   33.000000  Default  14.0000  14.0000
34              goreng   55.000000   55.000000  Default  13.0000  13.0000
49               kupas   41.000000   41.000000  Default  12.0000  12.0000
78                jahe   41.000000   41.000000  Default  11.0000  11.0000
83              lembar   36.000000   36.000000  Default  10.0000  10.0000
51              bersih   23.000000   23.000000  Default   9.0000   9.0000
8               bawang  417.000000  417.000000  Default   8.0000   8.0000
0                 ayam   48.000000   48.000000  Default   7.0000   7.0000
7                siung  185.000000  185.000000  Default   6.0000   6.0000
172              lemon   13.000000   13.000000  Default   5.0000   5.0000
177               ikat   14.000000   14.000000  Default   4.0000   4.0000
38               bubuk   92.000000   92.000000  Default   3.0000   3.0000
232               cuci   18.000000   18.000000  Default   2.0000   2.0000
10               butir   58.000000   58.000000  Default   1.0000   1.0000
58              selera   68.395333   69.004140   Topic1  -3.9269   0.6399
175             bombay   63.175639   63.776651   Topic1  -4.0063   0.6393
3              cincang   38.512518   39.114100   Topic1  -4.5013   0.6333
110             ukuran   31.181981   31.782920   Topic1  -4.7124   0.6297
57              sesuai   43.364054   44.271833   Topic1  -4.3826   0.6281
98              geprek   24.492795   25.094432   Topic1  -4.9539   0.6245
123            maizena   19.637873   20.238839   Topic1  -5.1748   0.6186
105             sendok   19.188138   19.789086   Topic1  -5.1980   0.6179
205             jagung   14.943191   15.544241   Topic1  -5.4480   0.6093
86             menumis   14.814793   15.415798   Topic1  -5.4566   0.6090
85                skip   14.350474   14.959672   Topic1  -5.4885   0.6072
197               saos   66.484838   69.526353   Topic1  -3.9553   0.6040
182              makan   12.435407   13.036065   Topic1  -5.6317   0.6016
319               utuh   12.123274   12.738135   Topic1  -5.6571   0.5993
437           larutkan   11.566790   12.167619   Topic1  -5.7041   0.5981
183               dadu   11.431076   12.038488   Topic1  -5.7159   

Topic: 1
telur, tepung, putih, garam, bawang, bubuk, air, ayam, gula, udang, gram, goreng, kaldu, butir, pasir, kulit, terigu, wijen, daun, lada, jamur, tapioka, sagu, ikan, menggoreng, kupas, pelengkap, wortel, merica, jeruk

Topic: 2
daun, merah, cabe, bumbu, lembar, bawang, ruas, siung, jeruk, halus, kunyit, salam, buah, butir, putih, biji, jari, udang, kemiri, kaldu, ijo, gula, batang, lengkuas, jahe, sct, keriting, garam, ketumbar, santan

Topic: 3
bawang, putih, siung, udang, buah, merah, garam, saus, tiram, kecap, air, selera, cabe, tomat, saos, gula, bombay, daun, iris, manis, bumbu, halus, rawit, sesuai, bubuk, cincang, penyedap, ukuran, batang, lada

Topic: 4
iris, bersih, kupas, cuci, buah, goreng, udang, air, wortel, tipis, lihat, resep, kepalanya, kol, tepung, memanjang, brokoli, tergantung, spesial, kekentalan, jamur, daun, bakwan, serong, genggam, garam, lbr, jeruk, belah, gram

Topic: 5
jeruk, garlic, mince, honey, water, black, vinegar, sriracha, rice, pepper, powder, 

[(0,
  '0.045*"telur" + 0.039*"tepung" + 0.037*"putih" + 0.037*"garam" + 0.035*"bawang" + 0.033*"bubuk" + 0.032*"air" + 0.029*"ayam" + 0.028*"gula" + 0.027*"udang"'),
 (1,
  '0.056*"daun" + 0.055*"merah" + 0.047*"cabe" + 0.037*"bumbu" + 0.036*"lembar" + 0.035*"bawang" + 0.034*"ruas" + 0.029*"siung" + 0.028*"jeruk" + 0.025*"halus"'),
 (2,
  '0.098*"bawang" + 0.044*"putih" + 0.043*"siung" + 0.043*"udang" + 0.042*"buah" + 0.039*"merah" + 0.031*"garam" + 0.029*"saus" + 0.025*"tiram" + 0.023*"kecap"'),
 (3,
  '0.069*"iris" + 0.042*"bersih" + 0.036*"kupas" + 0.032*"cuci" + 0.030*"buah" + 0.026*"goreng" + 0.024*"udang" + 0.023*"air" + 0.022*"wortel" + 0.022*"tipis"'),
 (4,
  '0.038*"jeruk" + 0.035*"garlic" + 0.033*"sriracha" + 0.033*"rice" + 0.033*"pepper" + 0.033*"honey" + 0.033*"water" + 0.033*"mince" + 0.033*"vinegar" + 0.033*"black"'),
 (5,
  '0.056*"serbaguna" + 0.054*"ikat" + 0.054*"perasan" + 0.030*"terasi" + 0.027*"sauce" + 0.027*"soy" + 0.025*"kunci" + 0.025*"sambal" + 0.024*"kangkun

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
sUdangProb = make_dictionary_terms_prob(ldamodel, id_map, d)
sUdangProb["bawang"] #probabilitas kata mee pada setiap topik

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
cpsUdangWN = dfWordnet.copy()

cpsUdangWN["Synonym_clean"] = cleanWN_Synonym(cpsUdangWN["Synonym"])
cpsUdangWN["Hypernim_clean"] = cleanWN(cpsUdangWN["Hypernim"])
cpsUdangWN["Hyponim_clean"] = cleanWN(cpsUdangWN["Hyponim"])
cpsUdangWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/udangCookpad.owl")
makeOntology(ontoName, cpsUdangWN)
ontoName.save(file = "udangCookpad.owl")

## Wikipedia & Category

In [ ]:
!gdown 1rQqb216082yJiR5TJqOHELNHpu9STyOj

In [ ]:
wiki_udang_sln = pd.read_excel("UdangCookpadFinal.xlsx").iloc[:, 1:]
wiki_udang_sln.rename(columns={"category": "Category"}, inplace = True)

# Translate
wiki_udang_sln["Category"] = translate(wiki_udang_sln["Category"])
wiki_udang_sln.head()

## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_udang_sln[["Title", "Category"]].copy()

# idx_nan, percent = check_nan_cat(tabel_1981)
# tabel_1981.iloc[idx_nan]

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

In [ ]:
udangSlnTabel_new = tabel_1981.copy()
udangSlnTabel_new["correct_category"] = all_cat
udangSlnTabel_new["suggestion"] = all_suggestion

# Export Result to File
udangSlnTabel_new.to_excel("udangSlnTabel_Result.xlsx")
udangSlnTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "udangSlnTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
udangSlnTabel_new = concatListInColumn(udangSlnTabel_new)
udangSlnTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, udangSlnTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_udang_sln.iloc[:, :-1].copy()

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check'
)

asyncio.ensure_future(f_wiki())

In [ ]:
udangCpdWiki_new = wiki_df.copy()
udangCpdWiki_new["suggest"] = all_cat

# Export Result to file
udangCpdWiki_new.to_excel("udangCpdWiki_Result.xlsx")
udangCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "udangCpdWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
udangCpdOntoWiki = get_ontology("http://test.org/udangCpd_onto_wiki.owl")
makeOntology_wiki(udangCpdOntoWiki, udangCpdWiki_new)
udangCpdOntoWiki.save(file = "udangCpd_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = cpsUdangWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(fWordnet())

In [ ]:
# Export Result to File
cpsUdangWN_new = listToDataframe(changesList, tableWordnet)
cpsUdangWN_new.to_excel("cpsUdangWN_Result.xlsx")
cpsUdangWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cpsUdangWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

# Malaysian Cooking PDF


In [ ]:
!gdown --id "1XZojUfe1couf7XUkNP54I2E_D4v4uPuZ"

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1XZojUfe1couf7XUkNP54I2E_D4v4uPuZ
To: /content/Malaysian_cooking.jl
100% 78.2k/78.2k [00:00<00:00, 48.7MB/s]


In [ ]:
cookPdf=pd.read_json(r'/content/Malaysian_cooking.jl', lines = True)

In [ ]:
cookPdf

,Title,page_bahan,page_langkah,page_description
0,Coffee Beef Ribs,"[1 kg Beef ribs, 3 tbsp Cooking oil, Marinade,...",[Put the beef ribs into a large bowl and add t...,"[1 kg Beef ribs, 3 tbsp Cooking oil, Marinade,..."
1,Beef Curry,"[1 kg Beef cubed, 1/2 tsp Fenugreek, 2 cm Cinn...",[Put the ingredients of Paste 1 into a blender...,"[1 kg Beef cubed, 1/2 tsp Fenugreek, 2 cm Cinn..."
2,Pepper Beef,"[350g Beef sliced, 3 cloves Garlic chopped fin...",[Place the beef in a bowl with the marinade in...,"[350g Beef sliced, 3 cloves Garlic chopped fin..."
3,Tangy Beef,"[350g Beef sliced, 2/3 tsp Sugar, 1/2 cube Bee...",[Mix the coconut powder and water and set asid...,"[350g Beef sliced, 2/3 tsp Sugar, 1/2 cube Bee..."
4,Beef Rendang,"[900g Beef cubed, 1/2 tin Coconut cream, 1/2 t...",[We have been tryin g to perfect this recipe f...,"[900g Beef cubed, 1/2 tin Coconut cream, 1/2 t..."
5,Beef with Red Curry Paste,"[Beef with Red Curry Paste, Ingredients, 400g ...",[Thinly slice the beef and then rub in the bic...,"[Beef with Red Curry Paste, Ingredients, 400g ..."
6,Szechuan Beef,"[1/2 tsp Light soya sauce, 250 g Beef sliced i...",[Put the beef in a bowl and add some salt (a s...,"[1/2 tsp Light soya sauce, 250 g Beef sliced i..."
7,Fennel Beef,"[400g Beef cubed, Hand full Raw peanuts, 1 tbs...",[Add the marinade ingredients to the beef in a...,"[400g Beef cubed, Hand full Raw peanuts, 1 tbs..."
8,Serunding Lembu,"[400g Beef sliced, 1/2 tsp Aniseed powder, 1 t...","[Blend the lemon grass, 1 onion, and ginger in...","[400g Beef sliced, 1/2 tsp Aniseed powder, 1 t..."
9,Beef Tomchi,"[300g Steak diced, 1/4 tsp Bi-Carb of Soda, 1 ...","[Put the diced beef into a bowl, Add the bi-ca...","[300g Steak diced, 1/4 tsp Bi-Carb of Soda, 1 ..."


In [ ]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
cookPdf=pd.read_json(r'/content/Malaysian_cooking.jl', lines = True)

#save dalam acar biar ngebut loading ntar.
cookPdf.to_pickle(r'/content/raw_df.pkl')

# # clean text
cookPdf = pre_cleaning_text(cookPdf, threshold = 20)

#data["cleaned_description"] = cleaning_text(data["page_description"])
#data["cleaned_description"].head()

# ingredients_all = data["cleaned_description"].apply(lambda x: treat_ingredients(x))
# ingredients_all

In [ ]:
cookPdf.head()

,Title,page_bahan,page_langkah,page_description
0,Coffee Beef Ribs,"[1 kg Beef ribs, 3 tbsp Cooking oil, Marinade,...",[Put the beef ribs into a large bowl and add t...,"[1 kg Beef ribs, 3 tbsp Cooking oil, Marinade,..."
1,Beef Curry,"[1 kg Beef cubed, 1/2 tsp Fenugreek, 2 cm Cinn...",[Put the ingredients of Paste 1 into a blender...,"[1 kg Beef cubed, 1/2 tsp Fenugreek, 2 cm Cinn..."
2,Pepper Beef,"[350g Beef sliced, 3 cloves Garlic chopped fin...",[Place the beef in a bowl with the marinade in...,"[350g Beef sliced, 3 cloves Garlic chopped fin..."
3,Tangy Beef,"[350g Beef sliced, 2/3 tsp Sugar, 1/2 cube Bee...",[Mix the coconut powder and water and set asid...,"[350g Beef sliced, 2/3 tsp Sugar, 1/2 cube Bee..."
4,Beef Rendang,"[900g Beef cubed, 1/2 tin Coconut cream, 1/2 t...",[We have been tryin g to perfect this recipe f...,"[900g Beef cubed, 1/2 tin Coconut cream, 1/2 t..."


## GUI Malaysian Cooking PDF

In [ ]:
# Run lagi jika sehabis running dataset lain
listSW = listStopword
print("Harap tekan tombol \"Add words and clean\" setiap runtime baru")

data = cookPdf.copy()

button = widgets.Button(description="Add words and clean")
run_button = widgets.Button(description = "Run Model")
output_run = widgets.Output()
output = widgets.Output()

teks = widgets.Text(
    value='',
    placeholder='Type something',
    description='String:',
    disabled=False
)

n_topics = widgets.IntSlider(
    value=7,
    min=5,
    max=10,
    step=1,
    description='n_topics:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
iteration = widgets.IntText(
    value=20,
    description='Number of Iteration:',
    disabled=False,
    layout = widgets.Layout(width='250px'),
    style = {'description_width': 'initial'}
)

display(teks)
display(n_topics)
display(iteration)
run_button.on_click(functools.partial(run_button_clicked, lang="eng"))
button.on_click(on_button_clicked)
display(button, output)
display(run_button, output_run)

Text(value='', description='String:', placeholder='Type something')

IntSlider(value=7, continuous_update=False, description='n_topics:', max=10, min=5)

IntText(value=20, description='Number of Iteration:', layout=Layout(width='250px'), style=DescriptionStyle(des…

Button(description='Add words and clean', style=ButtonStyle())

Output()

...

Output()

Running..


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.124472 -0.093419       1        1  53.785848
4     -0.112408 -0.095194       2        1  19.983216
3     -0.136839  0.155385       3        1  13.304130
6      0.041944  0.069057       4        1  12.909775
1      0.110524 -0.011896       5        1   0.005931
2      0.110891 -0.011829       6        1   0.005700
5      0.110360 -0.012105       7        1   0.005399, topic_info=               Term        Freq       Total Category  logprob  loglift
96          coconut   31.000000   31.000000  Default  30.0000  30.0000
97           powder   46.000000   46.000000  Default  29.0000  29.0000
100           paste   60.000000   60.000000  Default  28.0000  28.0000
15            sauce   62.000000   62.000000  Default  27.0000  27.0000
32              add  280.000000  280.000000  Default  26.0000  26.0000
109             set   39.000000   39.000000  Default  25.0000  25.0000
140             fry   55.000000   55.000000  Default  24.0000  24.0000
1              ribs   23.000000   23.000000  Default  23.0000  23.0000
62              pan   45.000000   45.000000  Default  22.0000  22.0000
13             salt   33.000000   33.000000  Default  21.0000  21.0000
93           leaves   15.000000   15.000000  Default  20.0000  20.0000
0              beef   79.000000   79.000000  Default  19.0000  19.0000
142            stir   39.000000   39.000000  Default  18.0000  18.0000
112        fragrant   17.000000   17.000000  Default  17.0000  17.0000
45            aside   42.000000   42.000000  Default  16.0000  16.0000
12              tsp   82.000000   82.000000  Default  15.0000  15.0000
47          minutes   46.000000   46.000000  Default  14.0000  14.0000
141            high   17.000000   17.000000  Default  13.0000  13.0000
42              mix   86.000000   86.000000  Default  12.0000  12.0000
44             well   53.000000   53.000000  Default  11.0000  11.0000
188          bicarb   16.000000   16.000000  Default  10.0000  10.0000
87         cinnamon   11.000000   11.000000  Default   9.0000   9.0000
33      ingredients   36.000000   36.000000  Default   8.0000   8.0000
23              msg   22.000000   22.000000  Default   7.0000   7.0000
24         optional   22.000000   22.000000  Default   6.0000   6.0000
29              put   56.000000   56.000000  Default   5.0000   5.0000
73             heat   59.000000   59.000000  Default   4.0000   4.0000
19             soya   24.000000   24.000000  Default   3.0000   3.0000
28            water   58.000000   58.000000  Default   2.0000   2.0000
4               oil   59.000000   59.000000  Default   1.0000   1.0000
96          coconut   31.134114   31.555136   Topic1  -4.1241   0.6067
93           leaves   14.625991   15.069214   Topic1  -4.8796   0.5903
87         cinnamon   10.802925   11.196493   Topic1  -5.1826   0.5844
306        potatoes    9.833580   10.227244   Topic1  -5.2766   0.5809
89             star    7.894572    8.287675   Topic1  -5.4962   0.5716
90            anise    7.894336    8.287570   Topic1  -5.4963   0.5715
115          reduce    7.894281    8.287637   Topic1  -5.4963   0.5715
149           basil    7.894085    8.287787   Topic1  -5.4963   0.5715
139           hours    7.893617    8.287629   Topic1  -5.4963   0.5715
143            cube    7.892293    8.287579   Topic1  -5.4965   0.5713
151          kaffir    8.741997    9.244911   Topic1  -5.3943   0.5642
92            curry    5.955519    6.348530   Topic1  -5.7781   0.5563
101            part    5.955145    6.348340   Topic1  -5.7781   0.5562
94         cardamom    5.955114    6.348337   Topic1  -5.7781   0.5562
86        fenugreek    5.955053    6.348313   Topic1  -5.7782   0.5562
117           rinse    5.955031    6.348388   Topic1  -5.7782   0.5562
106        turmeric    7.735492    8.271616   Topic1  -5.5166   0.5531
98             ghee    4.985835    5.378777   Topic1  -5.9558  

Topic: 1
add, mix, paste, beef, tsp, powder, heat, fry, oil, water, minutes, coconut, stir, set, cook, aside, salt, well, put, pan, cooking, sugar, serves, rice, serve, medium, cup, onion, garlic, ingredients

Topic: 2
add, paste, rendang, tsp, mix, ingredients, water, cook, put, powder, oil, low, cover, heat, cup, sugar, coconut, minutes, boiled, sauce, tbsp, get, serves, pan, dry, dish, well, cooking, set, malaysian

Topic: 3
add, mix, sauce, water, tsp, pan, put, oil, beef, fry, sugar, marinade, well, ribs, heat, paste, tbsp, dish, ingredients, aside, peanuts, minutes, serves, set, boiled, choice, bowl, vegetable, cooking, cook

Topic: 4
sauce, ribs, add, pan, bowl, mix, ingredients, put, well, water, tbsp, hot, plate, dish, frying, mixture, bbq, sure, prepare, place, use, cook, minutes, sugar, oil, beef, marinade, rice, cooking, flour

Topic: 5
add, beef, tsp, sauce, soya, mix, oil, sugar, flour, corn, fry, aside, set, tbsp, heat, put, cooking, ginger, stir, medium, water, pepper, 

[(0,
  '0.102*"add" + 0.029*"mix" + 0.028*"paste" + 0.026*"beef" + 0.026*"tsp" + 0.021*"powder" + 0.021*"heat" + 0.021*"fry" + 0.020*"oil" + 0.019*"water"'),
 (1,
  '0.003*"add" + 0.003*"paste" + 0.003*"rendang" + 0.003*"tsp" + 0.003*"mix" + 0.003*"ingredients" + 0.003*"water" + 0.003*"cook" + 0.003*"put" + 0.003*"powder"'),
 (2,
  '0.003*"add" + 0.003*"mix" + 0.003*"sauce" + 0.003*"water" + 0.003*"tsp" + 0.003*"pan" + 0.003*"put" + 0.003*"oil" + 0.003*"beef" + 0.003*"fry"'),
 (3,
  '0.055*"sauce" + 0.048*"ribs" + 0.046*"add" + 0.031*"pan" + 0.023*"bowl" + 0.023*"mix" + 0.023*"ingredients" + 0.023*"put" + 0.023*"well" + 0.017*"water"'),
 (4,
  '0.079*"add" + 0.032*"beef" + 0.028*"tsp" + 0.025*"sauce" + 0.025*"soya" + 0.020*"mix" + 0.019*"oil" + 0.018*"sugar" + 0.017*"flour" + 0.017*"corn"'),
 (5,
  '0.003*"add" + 0.003*"beef" + 0.003*"fry" + 0.003*"tsp" + 0.003*"oil" + 0.003*"sauce" + 0.003*"mix" + 0.003*"water" + 0.003*"heat" + 0.003*"powder"'),
 (6,
  '0.026*"sauce" + 0.026*"meat" + 

### Inspect Probability

In [ ]:
# Jika mau cek probabilitas per kata di topik dataset
cookPdfProb = make_dictionary_terms_prob(ldamodel, id_map, d)
cookPdfProb["beef"] #probabilitas kata beef pada setiap topik

[(0, 0.08954082),
 (1, 0.012025514),
 (2, 0.011557357),
 (3, 0.016079266),
 (4, 0.83193976),
 (5, 0.010947535),
 (6, 0.027909735)]

## Wordnet

In [ ]:
# Harus running modelnya dulu jangan lupa
cookPdfWN = dfWordnet.copy()

cookPdfWN["Synonym_clean"] = cleanWN_Synonym(cookPdfWN["Synonym"])
cookPdfWN["Hypernim_clean"] = cleanWN(cookPdfWN["Hypernim"])
cookPdfWN["Hyponim_clean"] = cleanWN(cookPdfWN["Hyponim"])
cookPdfWN

### Make Ontology

In [ ]:
# Make Ontology
ontoName = get_ontology("http://test.org/malaysianCook.owl")
makeOntology(ontoName, cookPdfWN)
ontoName.save(file = "malaysianCook.owl")

## Wikipedia & Category

In [ ]:
!gdown "1Be4Xi9t6ulzOfIXxZ2SYHo7v_ZTi9zRX"

Downloading...
From: https://drive.google.com/uc?id=1Be4Xi9t6ulzOfIXxZ2SYHo7v_ZTi9zRX
To: /content/MalayCookPdfFinal.xlsx
100% 6.42k/6.42k [00:00<00:00, 9.61MB/s]


In [ ]:
wiki_cook_pdf = pd.read_excel("MalayCookPdfFinal.xlsx").iloc[:, 1:]

# Translate
wiki_cook_pdf["Category"] = translate(wiki_cook_pdf["Category"])
wiki_cook_pdf.head()

,Title,Cleaned,Def_IND,Def_MS,Def_ENG,Category
0,Coffee Beef Ribs,coffee beef ribs,Not Found,Not Found,dark them in a variety of strengths,['daging']
1,Beef Curry,beef curry,Not Found,Not Found,Not Found,['daging']
2,Pepper Beef,pepper beef,perusahaan waralaba (franchise) lokal yang ber...,Not Found,French-inspired Vietnamese dish that consists ...,['daging']
3,Tangy Beef,tangy beef,Not Found,Not Found,Not Found,['daging']
4,Beef Rendang,beef rendang,Not Found,hidangan asli bangsa Minangkabau yang menduduk...,Minang dish originating from the Minangkabau r...,['daging']
5,Beef with Red Curry Paste,beef red curry paste,seorang sutradara film film hasil adaptasi The...,makanan tradisonal resepi Perancis,French beef stew braised in red wine with whol...,['daging']
6,Szechuan Beef,szechuan beef,Not Found,Not Found,Not Found,['daging']
7,Fennel Beef,fennel beef,Not Found,Not Found,Not Found,['daging']
8,Serunding Lembu,serunding lembu,Not Found,sejenis masakan yang dibuat dengan rempah dan ...,Not Found,[]
9,Beef Tomchi,beef tomchi,Not Found,Not Found,Not Found,['daging']


## Counter Category Tabel 1981

In [ ]:
tabel_1981 = wiki_cook_pdf[["Title", "Category"]].copy()

benar = 0
salah = 0
all_cat = []
all_suggestion = []
lanjut = False
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

asyncio.ensure_future(f())

ToggleButton(value=False, button_style='success', description='Next row', icon='check', tooltip='Description')


Row: 1/14
 Coffee Beef Ribs


SelectMultiple(description='category', options=('daging',), value=())

In [ ]:
cookPdfTabel_new = tabel_1981.copy()
cookPdfTabel_new["correct_category"] = all_cat
cookPdfTabel_new["suggestion"] = all_suggestion

# Export Result to File
cookPdfTabel_new.to_excel("cookPdfTabel_Result.xlsx")
cookPdfTabel_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cookPdfTabel_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Update Ontology NFO

In [ ]:
# NFO Ontology

# If runtime expired, put updated file in colab or gdrive and run commented code below
# !gdown "<insert-id-here>"
# table1981_Onto = get_ontology(r'/content/NFO_Updated.owl').load()

In [ ]:
# Concat List in Column "suggestion" & "correct_category"
cookPdfTabel_new = concatListInColumn(cookPdfTabel_new)
cookPdfTabel_new

In [ ]:
# Update Ontology
makeOntologyTable1981(table1981_Onto, cookPdfTabel_new)
table1981_Onto.save(file = "NFO_Updated.owl")

Downloading...
From: https://drive.google.com/uc?id=1T8NVqHoq4BMzNUFuTdWT9z1oaMrFZ7Ru
To: /content/NFOversion2April22.owl
100% 14.8k/14.8k [00:00<00:00, 12.7MB/s]


## Counter Wikipedia

In [ ]:
wiki_df = wiki_cook_pdf.iloc[:, :-1].copy().iloc[:]

all_cat = []
idx = []
button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success',
  tooltip='Description',
  icon='check' 
)

# kategori --> usulan class dari juri, textbox
asyncio.ensure_future(f_wiki())

ToggleButton(value=False, button_style='success', description='Next row', icon='check', tooltip='Description')


Row: 1/14
 Coffee Beef Ribs

======Definition========

Indonesia: Not Found
Malaysia: Not Found
English: dark them in a variety of strengths




Text(value='', description='Usulan Class', layout=Layout(height='40px', width='600px'), placeholder='input cla…

In [ ]:
cookCpdWiki_new = wiki_df.copy()
cookCpdWiki_new["suggest"] = all_cat

# Export Result to file
cookCpdWiki_new.to_excel("cookPdfWiki_Result.xlsx")
cookCpdWiki_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cookPdfWiki_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.

### Make Ontology Wikipedia

In [ ]:
# Make Ontology
cookCpdOntoWiki = get_ontology("http://test.org/malaysianCook_onto_wiki.owl")
makeOntology_wiki(cookCpdOntoWiki, cookCpdWiki_new)
cookCpdOntoWiki.save(file = "malaysianCook_onto_wiki.owl")

## Counter Wordnet Hypo, Syno, Hyper

In [ ]:
tableWordnet = cookPdfWN[["Word", "Synonym_clean", "Hypernim_clean", "Hyponim_clean"]].copy()

benarWordnet = 0
salahWordnet = 0
lanjut = False

button = widgets.ToggleButton(
  value=False,
  description='Next row',
  disabled=False,
  button_style='success', # 'success', 'info', 'warning', 'danger' or ''
  tooltip='Description',
  icon='check' # (FontAwesome names without the `fa-` prefix)
)

# status --> usulan class, kasih dua pilihan apakah setuju jika tidak judge kasih input sendiri (hipernim saja)
# jika hipernim kosong, maka judge input manual dan wordnet auto salah
asyncio.ensure_future(fWordnet())

ToggleButton(value=False, button_style='success', description='Next row', icon='check', tooltip='Description')


Row: 1/210

Word: add
Synonym: attention_deficit_disorder
Hypernim: ['syndrome']
Hyponim: None



RadioButtons(description='Hypernym (True/False?)', options=('True', 'False'), value='True')

Button(description='Submit', style=ButtonStyle())

In [ ]:
# Export Result to File
cookPdfWN_new = listToDataframe(changesList, tableWordnet)
cookPdfWN_new.to_excel("cookPdfWN_Result.xlsx")
cookPdfWN_new

In [ ]:
# Upload file to drive
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

upload_file = "cookPdfWN_Result.xlsx"

# setelah code dijalankan maka file akan ada di https://drive.google.com/drive/folders/1EorSiyj_0figmlI3aVIdobFtTIQZRuZF?usp=sharing
gfile = drive.CreateFile({'parents': [{'id': '1EorSiyj_0figmlI3aVIdobFtTIQZRuZF'}]})
# Read file and set it as the content of this instance.
gfile.SetContentFile(upload_file)
gfile.Upload() # Upload the file.